# Operational Intelligence - v3.0.0 Overview & Reporting Hub (Refined)

Based on the **Overview & Reporting Hub** application. This refinement removes the Overview Consolidated View hero card, moves Workforce immediately after Overview in the sidebar, makes the Overview near-miss chart obey the active light/dark theme, and replaces typographic dashes in desktop presentation strings with ordinary hyphens. Existing application modules, reports, and business calculations are preserved.


## 1. Configuration, SQLite and analytics

Run this first. Input CSV date parsing is intentionally British-only; ISO dates are also accepted.

In [ ]:
"""
OPERATIONAL INTELLIGENCE v3.0.0 - DESKTOP + JUPYTER EXECUTIVE APPLICATION

Run from VS Code with Python 3.10+:
  python -m pip install -r requirements.txt
  python -m playwright install chromium   # only for optional WhatsApp monitoring
  python app.py  # launch the modular desktop application

The dashboard works completely locally using SQLite in ./data/.
WhatsApp Web collection is optional and requires network access and user login.
"""
from __future__ import annotations

"""Offline operational intelligence: authoritative SQLite store and analytics."""
import csv, hashlib, json, logging, math, random, re, sqlite3, statistics, threading
from collections import Counter, defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

# A notebook has no __file__: resolve local assets relative to its working folder.
# When running as a .py script, keep assets alongside that script instead.
APP_VERSION = '3.0.0'
ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)
DB = DATA / 'operations.db'
CONFIG = DATA / 'settings.json'
FIELDS = ['Date','Time','Group','Sender','Full Message','T002/T003','PWT','SPL','PL','Near Miss','Late']
METRICS = {'T002/T003':'t002_t003','PWT':'pwt','SPL':'spl','PL':'pl','Near Miss':'near_miss','Late':'late'}
DEFAULT_CONFIG = {'groups':['Gang 1','Team 5'], 'sweep_seconds':5, 'alert_late_threshold':5,
                  'alert_missing_hours':24, 'alert_near_miss_threshold':1, 'theme':'Dark',
                  'db_path': None}


def configuration():
    try: return {**DEFAULT_CONFIG, **json.loads(CONFIG.read_text('utf-8'))}
    except (OSError,ValueError): return DEFAULT_CONFIG.copy()

def save_configuration(obj):
    """Write user preferences separately from the data being monitored."""
    CONFIG.write_text(json.dumps(obj,indent=2),encoding='utf-8')


def group_registry(config=None):
    """Read a validated, ordered group register without breaking legacy settings.

    The original collector expects config['groups'] to be a list of enabled
    group names. This richer register is UI-only metadata. Earlier-version settings
    are automatically shown as active entries without modifying the file.
    """
    cfg = config if config is not None else configuration()
    records = cfg.get('group_registry')
    if not isinstance(records, list):
        records = [{'name': name, 'enabled': True} for name in cfg.get('groups', [])]
    cleaned, seen = [], set()
    for entry in records:
        if isinstance(entry, str):
            name, enabled = entry.strip(), True
        elif isinstance(entry, dict):
            name = str(entry.get('name') or '').strip()
            enabled = entry.get('enabled', True) is not False
        else:
            continue
        if not name or len(name) > 160 or '\n' in name or '\r' in name:
            continue
        key = name.casefold()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append({'name': name, 'enabled': enabled})
    return cleaned


def active_group_names(registry):
    """The exact titles passed to the original, unchanged WhatsApp engine."""
    return [entry['name'] for entry in registry if entry['enabled']]


def validated_group_title(value):
    """Reject invalid names without excluding emoji or ordinary punctuation."""
    raw = str(value)
    if any(c in raw for c in ('\r', '\n', '\t')) or any(ord(c) < 32 for c in raw):
        raise ValueError('Group names cannot contain line breaks or control characters.')
    name = raw.strip()
    if not name:
        raise ValueError('Enter a WhatsApp group name.')
    if len(name) > 160:
        raise ValueError('Group names cannot exceed 160 characters.')
    return name


# The settings file stays in ./data even if the main database is relocated.
# This lets the application remember the selected SQLite directory on restart.
configured_db = configuration().get('db_path')
if configured_db:
    DB = Path(configured_db).expanduser().resolve()
    if not DB.is_file():
        raise FileNotFoundError(
            f'Configured SQLite file is missing: {DB}. Restore it or correct {CONFIG}. '
            'The application will not silently create an empty replacement.'
        )


def relocate_database(folder):
    """Snapshot the DB to a NEW directory, switch, and retain the old DB.

    Caller must stop live monitoring before moving. SQLite's backup API includes
    committed WAL data; a plain file copy would not necessarily do so. Never
    overwrite a database already present in the target directory.
    """
    import os
    import shutil
    import tempfile
    from contextlib import closing

    global DB
    destination_dir = Path(folder).expanduser().resolve()
    if not destination_dir.is_dir():
        raise NotADirectoryError(f'Choose an existing folder: {destination_dir}')
    target = destination_dir / 'operations.db'
    original = DB.resolve()
    if target == original:
        return target
    if target.exists():
        raise FileExistsError(f'{target} already exists. Choose another empty folder.')

    temp_name = None
    try:
        with tempfile.NamedTemporaryFile(dir=destination_dir, prefix='.operations-',
                                         suffix='.db', delete=False) as tmp:
            temp_name = tmp.name
        # Both context managers close before the atomic rename.
        with closing(connect()) as source, closing(sqlite3.connect(temp_name)) as backup:
            source.backup(backup)
        with closing(sqlite3.connect(temp_name)) as verified:
            check = verified.execute('PRAGMA integrity_check').fetchone()[0]
            if check != 'ok':
                raise RuntimeError(f'Copied database failed validation: {check}')
            count_before = verified.execute('SELECT COUNT(*) FROM reports').fetchone()[0]
        with closing(connect()) as source:
            count_after = source.execute('SELECT COUNT(*) FROM reports').fetchone()[0]
        if count_before != count_after:
            raise RuntimeError('Database changed during relocation. Stop imports and retry.')
        # Avoid accidentally overwriting a database introduced during the backup.
        if target.exists():
            raise FileExistsError(f'{target} already exists; no files were overwritten.')
        # Exclusive destination creation prevents overwriting existing DBs.
        try:
            os.link(temp_name, target)
        except OSError:
            # Hardlinks are not available on all Windows/network filesystems.
            # 'xb' is an exclusive-create fallback: never overwrite a file.
            try:
                with open(temp_name, 'rb') as reader, open(target, 'xb') as writer:
                    shutil.copyfileobj(reader, writer)
                    writer.flush()
                    os.fsync(writer.fileno())
            except Exception:
                # Keep the original intact; remove a partial destination only
                # when this process created it (target did not exist earlier).
                raise
        os.unlink(temp_name)
        temp_name = None
        cfg = configuration()
        cfg['db_path'] = str(target)
        # Save the choice before changing the active connection path, so a
        # failed settings write leaves the original connection untouched.
        save_configuration(cfg)
        DB = target
        audit('DATABASE_RELOCATED', f'Main database changed; previous copy retained at {original}')
        return target
    finally:
        if temp_name and Path(temp_name).exists():
            Path(temp_name).unlink()


class _ClosingSQLiteConnection(sqlite3.Connection):
    """Commit/rollback AND close after with-connect blocks (safe for 40+ groups)."""
    def __exit__(self,exc_type,exc_value,traceback):
        try:
            return super().__exit__(exc_type,exc_value,traceback)
        finally:
            self.close()


def connect():
    con=sqlite3.connect(DB,timeout=15,factory=_ClosingSQLiteConnection)
    con.row_factory=sqlite3.Row
    con.execute('PRAGMA busy_timeout=15000')
    con.execute('PRAGMA journal_mode=WAL')
    return con

def init_db():
    with connect() as c:
        c.executescript('''
        CREATE TABLE IF NOT EXISTS reports (
          id INTEGER PRIMARY KEY, fingerprint TEXT UNIQUE NOT NULL,
          event_at TEXT NOT NULL, captured_at TEXT NOT NULL,
          group_name TEXT NOT NULL, sender TEXT NOT NULL, message TEXT NOT NULL,
          t002_t003 INTEGER, pwt INTEGER, spl INTEGER, pl INTEGER,
          near_miss INTEGER, late INTEGER, no_shows INTEGER, origin TEXT NOT NULL DEFAULT 'import',
          quality_note TEXT DEFAULT ''
        );
        CREATE INDEX IF NOT EXISTS idx_reports_event ON reports(event_at);
        CREATE INDEX IF NOT EXISTS idx_reports_group ON reports(group_name);
        CREATE TABLE IF NOT EXISTS alerts (
          id INTEGER PRIMARY KEY, alert_key TEXT UNIQUE NOT NULL, created_at TEXT NOT NULL,
          level TEXT NOT NULL, title TEXT NOT NULL, detail TEXT NOT NULL,
          status TEXT NOT NULL DEFAULT 'Open', owner TEXT DEFAULT '', resolved_at TEXT
        );
        CREATE TABLE IF NOT EXISTS audit (
          id INTEGER PRIMARY KEY, at TEXT NOT NULL, action TEXT NOT NULL, detail TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS monitor_state (
          id INTEGER PRIMARY KEY CHECK(id=1), status TEXT NOT NULL,
          last_checked TEXT, last_success TEXT, last_error TEXT
        );
        INSERT OR IGNORE INTO monitor_state(id,status) VALUES (1,'Stopped');
        ''')
        columns = {r[1] for r in c.execute('PRAGMA table_info(reports)')}
        if 'no_shows' not in columns:
            c.execute('ALTER TABLE reports ADD COLUMN no_shows INTEGER')
        # Existing CSVs/WhatsApp messages keep their original schema. Backfill
        # only explicitly reported numeric no-show counts, never infer from gaps.
        for message_id, raw in c.execute('SELECT id,message FROM reports WHERE no_shows IS NULL'):
            n = parse_explicit_no_shows(raw)
            if n is not None:
                c.execute('UPDATE reports SET no_shows=? WHERE id=?', (n, message_id))

def stamp(): return datetime.now().isoformat(timespec='seconds')

def audit(action,detail):
    with connect() as c: c.execute('INSERT INTO audit(at,action,detail) VALUES(?,?,?)',(stamp(),action,str(detail)[:1500]))

def parse_metrics(message):
    text=' '.join(str(message).split())
    result={}
    patterns={'T002/T003':r'\bT002\s*[/\\_-]\s*T003\s*[:\s-]*([0-9]+)\b',
              'Near Miss':r'\bNear\s+Miss(?:es)?\s*[:\s-]*([0-9]+)\b'}
    for name in METRICS:
        pattern=patterns.get(name, rf'\b{re.escape(name)}\s*[:\s-]*([0-9]+)\b')
        m=re.search(pattern,text,re.I)
        result[name]=int(m.group(1)) if m else None
    return result

def parse_explicit_no_shows(message):
    """Read a numeric no-show count only when WhatsApp explicitly provides one.

    Examples: 'No Shows: 2', 'No-Show 1', 'No show(s) - 0'.
    Missing mention is UNKNOWN, never zero and never equal to shortage.
    """
    text = ' '.join(str(message or '').split())
    match = re.search(r'\bno[\s-]*shows?(?:\(s\))?\s*[:=\-]?\s*(\d{1,5})\b', text, re.I)
    return int(match.group(1)) if match else None


def parse_date(value):
    if isinstance(value,datetime): return value.date()
    if isinstance(value,date): return value
    s=str(value or '').strip()
    for fmt in ('%d/%m/%Y','%Y-%m-%d','%d/%m/%y','%Y-%m-%d %H:%M:%S','%d-%m-%Y'):
        try: return datetime.strptime(s,fmt).date()
        except ValueError: pass
    raise ValueError(f'Unrecognised date: {s!r}')

def normalise_datetime(d,t):
    day=parse_date(d)
    txt=str(t or '00:00:00').strip().upper()
    for fmt in ('%H:%M:%S','%H:%M','%I:%M %p','%I:%M:%S %p'):
        try:
            tt=datetime.strptime(txt,fmt).time()
            return datetime.combine(day,tt).isoformat(timespec='seconds')
        except ValueError: pass
    raise ValueError(f'Unrecognised time: {t!r}')

def fingerprint(group,sender,at,message):
    payload='\0'.join((str(group).strip().casefold(),str(sender).strip().casefold(),at,str(message).strip()))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def insert_report(group,sender,message,dt,metrics=None,origin='manual'):
    """Returns True if inserted, False if exact duplicate. Never equates unreported values with zero."""
    group=str(group).strip(); sender=str(sender).strip(); message=str(message).strip()
    if not group or not message: raise ValueError('Group and message cannot be empty')
    at=dt.isoformat(timespec='seconds') if isinstance(dt,datetime) else str(dt)
    datetime.fromisoformat(at) # reject invalid timestamps
    vals=metrics if metrics is not None else parse_metrics(message)
    clean={}
    for k in METRICS:
        v=vals.get(k,None)
        if v in (None,''): clean[k]=None
        else:
            n=int(v)
            if n < 0 or n > 100000: raise ValueError(f'Invalid {k} value: {v}')
            clean[k]=n
    token=fingerprint(group,sender,at,message)
    reported_no_shows = parse_explicit_no_shows(message)
    # Optional structured imports may include a separate No Shows column.
    if metrics is not None and metrics.get('No Shows') not in (None, ''):
        reported_no_shows = _ex_validate_nonneg(metrics['No Shows'], 'No Shows') if '_ex_validate_nonneg' in globals() else int(metrics['No Shows'])
    qnote='Missing safety fields' if clean['Near Miss'] is None or clean['Late'] is None else ''
    with connect() as c:
        cursor=c.execute('''INSERT OR IGNORE INTO reports
          (fingerprint,event_at,captured_at,group_name,sender,message,t002_t003,pwt,spl,pl,near_miss,late,no_shows,origin,quality_note)
          VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
          (token,at,stamp(),group,sender,message,*(clean[k] for k in METRICS),reported_no_shows,origin,qnote))
        return cursor.rowcount==1

# Only for automatic replay of the original historical staging CSV.
# These specific erroneous legacy dates were identified and explicitly
# rejected by the user in the 10 November 2026 monitoring records.
# Manual historical CSV imports are unaffected.
KNOWN_MISDATED_LEGACY_DATES = frozenset({
    '2026-06-11', '2026-09-11', '2026-10-11'
})


def import_csv(path, *, quarantine_legacy_dates=False):
    added=0; dupes=0; invalid=[]
    with open(path,newline='',encoding='utf-8-sig') as f:
        reader=csv.DictReader(f)
        absent={'Date','Time','Group'}-set(reader.fieldnames or [])
        if absent: raise ValueError(f'CSV missing required columns: {", ".join(sorted(absent))}')
        for number,row in enumerate(reader,2):
            # A blank preformatted spreadsheet row is not an invalid report.
            if not any(str(value or "").strip() for value in row.values()):
                continue
            try:
                dt=normalise_datetime(row['Date'],row['Time'])
                # Old staging files may contain dates written incorrectly by
                # the pre-fix WhatsApp parser. Do not re-import those known
                # bad legacy values during automatic startup reconciliation.
                # This is NOT a global restriction on legitimate historical
                # CSV imports performed manually by the user.
                if quarantine_legacy_dates and dt[:10] in KNOWN_MISDATED_LEGACY_DATES:
                    invalid.append(f'Row {number}: quarantined known legacy date {dt[:10]}')
                    continue
                metric_values={**{k:row.get(k,'') for k in METRICS}, 'No Shows': row.get('No Shows', '')}
                if all(v in ('',None) for v in metric_values.values()) and row.get('Full Message'): metric_values=parse_metrics(row['Full Message'])
                inserted=insert_report(row['Group'],row.get('Sender') or 'Structured export',row.get('Full Message') or ('Structured import: ' + ' | '.join(f'{k}={metric_values.get(k,chr(45))}' for k in METRICS)),dt,metric_values,origin='csv')
                added+=int(inserted); dupes+=int(not inserted)
            except Exception as e:
                invalid.append(f'Row {number}: {e}')
    audit('CSV_IMPORT',f'{Path(path).name}: added={added} duplicates={dupes} rejected={len(invalid)}')
    # If a historical staging CSV contains inconsistent US-formatted dates,
    # do NOT silently reinterpret them; users can review rejected rows.
    return added,dupes,invalid

def import_excel(path):
    from openpyxl import load_workbook
    from tempfile import NamedTemporaryFile
    import os
    wb=load_workbook(path,read_only=True,data_only=True)
    sheet=wb.active
    rows=sheet.iter_rows(values_only=True)
    headers=[str(v) if v is not None else '' for v in next(rows)]
    with NamedTemporaryFile('w',suffix='.csv',delete=False,newline='',encoding='utf-8') as f:
        p=f.name; writer=csv.DictWriter(f,fieldnames=FIELDS);writer.writeheader()
        for values in rows:
            item=dict(zip(headers,values));writer.writerow({k:item.get(k,'') for k in FIELDS})
    wb.close()
    try:return import_csv(p)
    finally:os.unlink(p)

def filtered(start=None,end=None,group=None):
    sql='SELECT * FROM reports WHERE 1=1';params=[]
    if start: sql+=' AND event_at>=?';params.append(str(start)+'T00:00:00')
    if end: sql+=' AND event_at<=?';params.append(str(end)+'T23:59:59')
    if group and group!='All Groups':sql+=' AND group_name=?';params.append(group)
    sql+=' ORDER BY event_at DESC,id DESC'
    with connect() as c:return [dict(r) for r in c.execute(sql,params).fetchall()]

def groups():
    with connect() as c:return [x[0] for x in c.execute('SELECT DISTINCT group_name FROM reports ORDER BY group_name')]

def summary(rows,expected_groups=None):
    count=len(rows);present={r['group_name'] for r in rows}
    expected=set(expected_groups or [])
    total=lambda field:sum(r[field] for r in rows if r[field] is not None)
    daily=Counter(r['event_at'][:10] for r in rows)
    bygroup=Counter(r['group_name'] for r in rows)
    now=datetime.now();last=max((r['event_at'] for r in rows),default=None)
    return dict(reports=count, teams=len(present), late=total('late'), near_miss=total('near_miss'),
                missing_safety=sum(r['near_miss'] is None or r['late'] is None for r in rows),
                missing_groups=sorted(expected-present), coverage=(len(expected & present)/len(expected)*100 if expected else None),
                daily=dict(sorted(daily.items())), bygroup=bygroup,
                staffing={k:total(k) for k in ('t002_t003','pwt','spl','pl')},last_event=last,
                freshness_hours=((now-datetime.fromisoformat(last)).total_seconds()/3600 if last else None))

def compare(rows,start,end):
    if not start or not end: return None
    a=parse_date(start);b=parse_date(end);n=(b-a).days+1
    if n<1:return None
    prior=filtered(a-timedelta(days=n),a-timedelta(days=1))
    curr=summary(rows);prev=summary(prior)
    return {k: (curr[k]-prev[k])/prev[k]*100 if prev[k] else None for k in ('reports','late','near_miss')}

def trend_forecast(rows,days=7):
    """Rolling weekday-aware baseline with uncertainty from historical variability; not causal prediction."""
    if not rows:return [],'No historical data'
    dates=[parse_date(r['event_at'][:10]) for r in rows]; earliest=min(dates); latest=max(dates)
    observed=(latest-earliest).days+1
    if observed<14:return [],'At least 14 calendar days of history needed'
    daily=Counter(dates)
    series=[daily[earliest+timedelta(days=i)] for i in range(observed)]
    recent=series[-min(56,len(series)):]
    predictions=[]
    for i in range(1,days+1):
        target=latest+timedelta(days=i)
        weekday_values=[daily[earliest+timedelta(days=j)] for j in range(observed)
                        if (earliest+timedelta(days=j)).weekday()==target.weekday()]
        vals=weekday_values[-8:] if len(weekday_values)>=3 else recent
        mean=statistics.mean(vals); sigma=statistics.stdev(vals) if len(vals)>1 else 0
        predictions.append({'date':target.isoformat(),'predicted':round(mean,1),
                            'low':max(0,round(mean-1.96*sigma,1)), 'high':round(mean+1.96*sigma,1)})
    return predictions,'Historical weekday baseline; indicative variability band, not a calibrated 95% prediction interval'

def scenarios(demand,available,absence_pct,contractors):
    vals=[float(demand),float(available),float(absence_pct),float(contractors)]
    if min(vals)<0 or absence_pct>100: raise ValueError('Scenario inputs out of range')
    effective=max(0,available*(1-absence_pct/100))+contractors
    return {'effective':effective,'gap':max(0,demand-effective),'surplus':max(0,effective-demand),
            'coverage':(100*effective/demand if demand else 100)}

def upsert_alert(key,level,title,detail):
    with connect() as c:
        c.execute('''INSERT OR IGNORE INTO alerts(alert_key,created_at,level,title,detail) VALUES(?,?,?,?,?)''',
                  (key,stamp(),level,title,detail))

def detect_reporting_anomalies(days=35):
    """Flags unusually low/high daily report counts using a robust median/MAD baseline.
    Informational only; minimum 14 observed calendar days required.
    """
    today=date.today()
    rows=filtered(today-timedelta(days=days-1),today)
    if not rows:return []
    earliest=min(parse_date(x['event_at'][:10]) for x in rows)
    span=(today-earliest).days+1
    if span<14:return []
    observed=Counter(parse_date(x['event_at'][:10]) for x in rows)
    history=[observed[earliest+timedelta(days=i)] for i in range(span-1)]
    if len(history)<13:return []
    baseline=statistics.median(history)
    mad=statistics.median(abs(v-baseline) for v in history)
    robust_sigma=max(1.0,1.4826*mad)
    current=observed[today]
    if abs(current-baseline)<max(3,3*robust_sigma):return []
    direction='above' if current>baseline else 'below'
    return [f'Today currently has {current} report(s), {direction} historical daily median {baseline:g}; incomplete day / collection gaps may explain differences.']

def evaluate_alerts():
    cfg=configuration();today=date.today();rows=filtered(today-timedelta(days=1),today)
    stats=summary(rows, cfg['groups'])
    if stats['near_miss'] >= cfg['alert_near_miss_threshold'] and stats['near_miss']>0:
        upsert_alert('safety-'+today.isoformat(),'High','Near-miss reports require review',
                     f"{stats['near_miss']} reported near-miss values over last two calendar days; verify events and duplicates.")
    if stats['late'] >= cfg['alert_late_threshold']:
        upsert_alert('late-'+today.isoformat(),'Medium','Late reporting above threshold',
                     f"{stats['late']} reported late values across last two calendar days.")
    for group in cfg['groups']:
        with connect() as c:
            last=c.execute('SELECT MAX(event_at) FROM reports WHERE group_name=?',(group,)).fetchone()[0]
        if last and (datetime.now()-datetime.fromisoformat(last)).total_seconds()>cfg['alert_missing_hours']*3600:
            upsert_alert('missing-'+group+'-'+today.isoformat(),'Medium','Possible missing team reporting',
                         f'{group}: latest captured message {last}. Check expected reporting schedule.')
    for index,description in enumerate(detect_reporting_anomalies()):
        upsert_alert('anomaly-'+today.isoformat()+'-'+str(index),'Info','Unusual reporting volume',description)
    if stats['missing_safety']:
        upsert_alert('quality-'+today.isoformat(),'Info','Incomplete safety reporting fields',
                     f"{stats['missing_safety']} reports have an unreported Near Miss or Late field (not assumed zero).")

def alerts(status=None):
    with connect() as c:
        query='SELECT * FROM alerts'+(' WHERE status=?' if status else '')+' ORDER BY CASE level WHEN "High" THEN 0 WHEN "Medium" THEN 1 ELSE 2 END, created_at DESC'
        return [dict(x) for x in c.execute(query,(status,) if status else ())]

def update_alert(id,status,owner=''):
    if status not in ('Open','Acknowledged','Closed'):raise ValueError('Invalid alert state')
    with connect() as c:
        c.execute('UPDATE alerts SET status=?,owner=?,resolved_at=? WHERE id=?',
                  (status,owner,stamp() if status=='Closed' else None,id))
    audit('ALERT_UPDATE',f'{id}: {status} owner={owner}')

def set_monitor_state(status,error=None,success=False):
    with connect() as c:
        c.execute('''UPDATE monitor_state SET status=?,last_checked=?,last_success=CASE WHEN ? THEN ? ELSE last_success END,
                     last_error=? WHERE id=1''',(status,stamp(),int(success),stamp(),error))

def monitor_state():
    with connect() as c:return dict(c.execute('SELECT * FROM monitor_state WHERE id=1').fetchone())

# Exports include the sender when known, but NEVER the original raw message.
EXPORT_FIELDS = ['Date','Time','Group','Sender','T002/T003','PWT','SPL','PL','Near Miss','Late']

def export_rows(rows):
    for r in rows:
        dt=datetime.fromisoformat(r['event_at'])
        yield [dt.strftime('%d/%m/%Y'), dt.strftime('%H:%M:%S'), r['group_name'],
               (r.get('sender') or '') if (r.get('sender') or '') not in ('Unknown Sender', 'Structured export') else ''] + [
            r[METRICS[k]] if r[METRICS[k]] is not None else '' for k in METRICS
        ]

def export_csv(path,rows):
    with open(path,'w',newline='',encoding='utf-8-sig') as f:
        writer=csv.writer(f);writer.writerow(EXPORT_FIELDS);writer.writerows(export_rows(rows))
    audit('CSV_EXPORT',f'{len(rows)} structured records (no raw messages)')

def export_excel(path,rows):
    from openpyxl import Workbook
    from openpyxl.styles import Font,PatternFill,Alignment
    wb=Workbook();ws=wb.active;ws.title='Operational Reports'
    ws.append(EXPORT_FIELDS)
    for record in export_rows(rows):ws.append(record)
    ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
    for cell in ws[1]:
        cell.fill=PatternFill('solid',fgColor='17243A')
        cell.font=Font(bold=True,color='FFFFFF')
        cell.alignment=Alignment(horizontal='center')
    for col,width in {'A':15,'B':14,'C':28,'D':26,'E':17,'F':12,'G':12,'H':12,'I':16,'J':14}.items():
        ws.column_dimensions[col].width=width
    wb.save(path);audit('EXCEL_EXPORT',f'{len(rows)} structured records (no raw messages)')

def safety_daily(rows):
    days=defaultdict(lambda:{'near_miss':0,'late':0})
    for r in rows:
        d=r['event_at'][:10]
        days[d]['near_miss']+=r['near_miss'] or 0
        days[d]['late']+=r['late'] or 0
    return dict(sorted(days.items()))


def near_miss_reported_series(rows):
    """Return sorted dates with received reports only (no fabricated days).

    This aggregation is calculated afresh from the currently filtered rows.
    A date with a report but a reported near-miss count of zero remains
    visible; a date with no reports is not included at all.
    """
    return list(safety_daily(rows).items())


def near_miss_calendar_series(rows, max_days=None):
    """Backward-compatible wrapper for notebooks using the earlier name.

    No continuous calendar or fixed number-of-days cap is applied.
    The boolean marks actual reported dates, always True here.
    """
    return [(day, values, True) for day, values in near_miss_reported_series(rows)]


def plot_near_miss_calendar(ax, rows, *, dark=True):
    """Dynamic near-miss chart: one bar and DD-MM label per *reported* day.

    Do not insert blank calendar dates or skip labels using a fixed stride.
    A user changing a date/group filter causes the chart to recompute its
    dates directly from the filtered records passed by the dashboard.
    """
    ax.clear()
    series = near_miss_reported_series(rows)
    ink = '#AFC0D3' if dark else '#4D5C6F'
    grid = '#38475C' if dark else '#D3DCE8'
    if not series:
        ax.text(.5, .5, 'No reports for selected period', color=ink,
                ha='center', va='center', transform=ax.transAxes)
    else:
        xs = list(range(len(series)))
        values = [counts['near_miss'] for _, counts in series]
        ax.bar(xs, values, width=.62, color='#D35400', zorder=2)
        n = len(series)
        # Label every *reported* day. Adapt label angle as data grows.
        angle = 0 if n <= 12 else (45 if n <= 24 else 75)
        font_size = 8 if n <= 20 else 7
        ax.set_xticks(xs)
        ax.set_xticklabels([uk_chart_date(day) for day, _ in series],
                           rotation=angle,
                           ha='right' if angle else 'center', fontsize=font_size)
        ax.set_xlim(-.65, n - .35)
    ax.tick_params(colors=ink, labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(grid)
    ax.grid(axis='y', color=grid, alpha=.4, zorder=0)

def gang_attendance(rows):
    """Aggregate workforce category values AND lateness by gang.

    Late is an attendance indicator, not a workforce category. It is drawn as
    an independent bar series, never included in workforce pie denominators.
    """
    result = defaultdict(lambda: {k: 0 for k in
                           ('t002_t003', 'pwt', 'spl', 'pl', 'late')})
    for row in rows:
        name = row.get('group_name') or 'Unknown group'
        for key in result[name]:
            result[name][key] += row.get(key) or 0
    return dict(sorted(result.items()))


def safety_completeness(rows):
    """Return numerator, denominator and percentage of complete safety fields."""
    total = len(rows)
    complete = sum(r.get('near_miss') is not None and r.get('late') is not None
                   for r in rows)
    return complete, total, (100 * complete / total if total else None)


def near_miss_by_group(rows):
    """Reported near-miss values by group; not a confirmed incident count."""
    result = Counter()
    for row in rows:
        result[row.get('group_name') or 'Unknown group'] += row.get('near_miss') or 0
    return dict(sorted(result.items(), key=lambda item: (-item[1], item[0])))


def workforce_totals(rows):
    keys=['t002_t003','pwt','spl','pl']
    return [sum((r[k] or 0) for r in rows) for k in keys]

def original_style_pie(ax,labels,values,colors,dark=True):
    """Original notebook styling, with corrected workforce-only denominator."""
    ax.clear()
    indices=[i for i,v in enumerate(values) if v>0]
    if not indices:
        ax.text(.5,.5,'No data in selected period',ha='center',va='center',color='#a0a0a0' if dark else '#666666',transform=ax.transAxes)
        return
    ax.pie([values[i] for i in indices],labels=[labels[i] for i in indices],
           colors=[colors[i] for i in indices],autopct='%1.1f%%',startangle=140,
           textprops={'color':'#ffffff' if dark else '#202C3D','fontsize':8})
    ax.axis('equal')

def uk_chart_date(iso_day):
    """Display a compact UK DD-MM label for Executive near-miss charts only.

    Event timestamps remain stored in ISO format. CSV and Excel exports keep
    their required full DD/MM/YYYY date format. Do not alter database dates.
    """
    return datetime.strptime(iso_day[:10], '%Y-%m-%d').strftime('%d-%m')


# ---------------------------------------------------------------------------
# C-SUITE EXPORTS — SAME EXCEPTION MODEL AS EXECUTIVE DASHBOARD
# ---------------------------------------------------------------------------
# The previous PDF and HTML exporters only accepted WhatsApp message rows.
# Exports now accept the complete exception_dashboard rows for the SAME scope.

def _build_report_export_payload(rows, exception_rows=None):
    """Build a lossless, offline reporting snapshot.

    If exception_rows is omitted by a legacy caller, derive the applicable
    group/date range from the supplied WhatsApp reports. The GUI always passes
    the selected exception_dashboard explicitly, including before-report entries.
    """
    source_rows = list(rows or [])
    if exception_rows is None:
        if source_rows:
            dates = sorted(r['event_at'][:10] for r in source_rows)
            names = {r['group_name'].casefold() for r in source_rows}
            selected = source_rows[0]['group_name'] if len(names) == 1 else 'All Groups'
            exception_rows = exception_dashboard(dates[0], dates[-1], selected)['rows']
        else:
            exception_rows = []
    role_names = list(EX_METRIC_FIELDS)
    exceptions = []
    for record in exception_rows:
        roles = [{k:int(part.get(k,0)) for k in ('required','present','shortage','surplus')}
                 | {'role':str(part.get('role',''))}
                 for part in record.get('role_breakdown',[])]
        exceptions.append({
            'date':str(record['service_date']),
            'group':str(record['group']),
            'before':record.get('before_count'),
            'after':record.get('after_count'),
            'no_shows':record.get('no_shows'),
            'required':record.get('plan_total'),
            'available':record.get('available'),
            'shortage':record.get('shortage'),
            'status':str(record.get('status','Pending')),
            'reason':str(record.get('reason') or ''),
            'report_id':record.get('report_id'),
            'roles':roles
        })
    exceptions.sort(key=lambda x:(x['date'],x['group'].casefold()),reverse=True)
    # Night-shift attendance can be captured AFTER MIDNIGHT. If a source
    # report is linked to an exception service date but falls outside the
    # message-date UI filter, include that evidence explicitly and avoid a
    # duplicate of any report already selected by message date.
    linked_days={int(x['report_id']):x['date'] for x in exceptions
                 if x['report_id'] is not None}
    selected_ids={int(x['id']) for x in source_rows if x.get('id') is not None}
    need_ids=sorted(set(linked_days)-selected_ids)
    if need_ids:
        with connect() as conn:
            for start in range(0,len(need_ids),300):
                batch=need_ids[start:start+300]
                qmarks=','.join('?' for _ in batch)
                source_rows.extend(dict(x) for x in conn.execute(
                    'SELECT * FROM reports WHERE id IN ('+qmarks+')',batch))
    reports = []
    for r in source_rows:
        reports.append({
            'day':linked_days.get(r.get('id'),str(r['event_at'])[:10]),
            'when':str(r['event_at'])[:19].replace('T',' '),
            'group':str(r['group_name']),
            'sender':str(r.get('sender') or ''),
            **{k:r.get(k) for k in ('t002_t003','pwt','spl','pl','near_miss','late','no_shows')}
        })
    reports.sort(key=lambda x:(x['when'],x['group'].casefold()),reverse=True)
    known = lambda k: [x for x in exceptions if x[k] is not None]
    totals = {
        'reports':len(reports),
        'groups':len({x['group'] for x in exceptions} | {x['group'] for x in reports}),
        'changes_before':sum(int(x['before']) for x in known('before')),
        'changes_after':sum(int(x['after']) for x in known('after')),
        'before_known':len(known('before')),
        'after_known':len(known('after')),
        'no_shows':sum(int(x['no_shows']) for x in known('no_shows')),
        'no_shows_known':len(known('no_shows')),
        'shortage':sum(int(x['shortage']) for x in known('shortage')),
        'shortage_known':len(known('shortage')),
        'pending':sum(x['shortage'] is None for x in exceptions),
        'changes_complete':sum(x['before'] is not None and x['after'] is not None for x in exceptions),
        'near_miss':sum(int(x['near_miss'] or 0) for x in reports),
        'late':sum(int(x['late'] or 0) for x in reports)
    }
    return {'reports':reports,'exceptions':exceptions,'totals':totals,
            'roles':role_names,'generated_at':stamp()}


def export_interactive_html(path, rows, scope='All Groups', exception_rows=None):
    """Full-fidelity offline executive HTML with filtering and 4-category charts."""
    import html as _htmlmod
    payload=_build_report_export_payload(rows,exception_rows)
    # Encode script-breaking characters even when strings come from WhatsApp.
    embedded=json.dumps(payload,ensure_ascii=True,separators=(',',':'))
    for ch,replacement in [('<',r'\u003c'),('>',r'\u003e'),('&',r'\u0026'),
                           ('\u2028',r'\u2028'),('\u2029',r'\u2029')]:
        embedded=embedded.replace(ch,replacement)
    output=_HTML_REPORT_TEMPLATE.replace('__PAYLOAD__',embedded)
    output=output.replace('__SCOPE__',_htmlmod.escape(str(scope),quote=True))
    output=output.replace('__GENERATED__',_htmlmod.escape(payload['generated_at'],quote=True))
    Path(path).write_text(output,encoding='utf-8')
    audit('HTML_EXPORT',f'{scope} / {len(payload["exceptions"])} exception records')
    return path



# -------------------------------------------------------------------------
# v3.0.0: Additional board charts matching existing Workforce / Safety views.
# All chart data is derived from the same exported WhatsApp report snapshot.
# -------------------------------------------------------------------------
def _report_chart_dataset(report_rows):
    gang = {}
    near = {}
    roles = ('t002_t003','pwt','spl','pl')
    allocation = {key: 0 for key in roles}
    for row in report_rows:
        name = str(row.get('group') or 'Unknown group')
        values = gang.setdefault(name,{key:0 for key in (*roles,'late')})
        for key in values:
            values[key] += int(row.get(key) or 0)
        near[name] = near.get(name,0) + int(row.get('near_miss') or 0)
        for key in roles:
            allocation[key] += int(row.get(key) or 0)
    return {'gang':gang, 'near':near, 'allocation':allocation}

def _append_pdf_workforce_and_safety_charts(story,reports,heading,p,figimage):
    from matplotlib.figure import Figure
    metrics=_report_chart_dataset(reports)
    columns=[('t002_t003','T002/T003','#2980b9'),('pwt','PWT','#2ecc71'),
             ('spl','SPL','#e67e22'),('pl','PL','#e74c3c'),('late','Late','#f1c40f')]
    story.append(heading('Gang-wise attendance — stacked horizontal comparison'))
    names=sorted(metrics['gang'],key=str.casefold)
    if names:
        for start in range(0,len(names),12):
            segment=names[start:start+12]
            fig=Figure(figsize=(8.2,max(2.5,.38*len(segment)+1.0)))
            ax=fig.add_subplot(111)
            y=list(range(len(segment)));left=[0]*len(segment)
            for key,title,color in columns:
                v=[metrics['gang'][name][key] for name in segment]
                ax.barh(y,v,left=left,label=title,color=color,height=.68)
                left=[a+b for a,b in zip(left,v)]
            ax.set_yticks(y,[(n[:26]+'…') if len(n)>27 else n for n in segment],fontsize=8)
            ax.invert_yaxis();ax.grid(axis='x',alpha=.18);ax.set_axisbelow(True)
            ax.legend(loc='lower center',bbox_to_anchor=(.5,1.02),ncol=5,fontsize=7,frameon=False)
            fig.subplots_adjust(left=.29,right=.97,bottom=.09,top=.86)
            if len(names)>12:
                story.append(p(f'Groups {start+1}–{start+len(segment)} of {len(names)}','ReportSmall'))
            story.append(figimage(fig,min(350,max(155,70+21*len(segment)))))
    else:
        story.append(p('No gang-wise WhatsApp attendance in the selected period.'))
    story.append(p('Stacked bars use T002/T003, PWT, SPL and PL reported values by group. '
                   'Late is a separate attendance exception indicator, not a staffing role. '
                   'Multiple WhatsApp messages may include the same people.','ReportSmall'))
    from reportlab.platypus import KeepTogether
    positive=sorted(((g,n) for g,n in metrics['near'].items() if n>0),
                    key=lambda x:(-x[1],x[0].casefold()))
    if len(positive)>12:
        display_near=positive[:11]+[('Other groups',sum(v for _,v in positive[11:]))]
    else:
        display_near=positive
    role_labels=[('t002_t003','T002/T003'),('pwt','PWT'),('spl','SPL'),('pl','PL')]
    allocation=[(title,metrics['allocation'][key]) for key,title in role_labels
                if metrics['allocation'][key]>0]
    fig=Figure(figsize=(8.2,4.2))
    ax1=fig.add_subplot(121);ax2=fig.add_subplot(122)
    near_colors=['#d35400','#2980b9','#27ae60','#8e44ad','#f39c12',
                 '#16a085','#c0392b','#597cbb','#ae7ac0','#b9a045','#7a9e49','#e18a71']
    def add_pie(ax,items,colors,title):
        ax.set_title(title,fontsize=10,pad=8,color='#15263E')
        if not items:
            ax.text(.5,.5,'No positive reported values',ha='center',va='center',
                    fontsize=9,transform=ax.transAxes)
            ax.axis('off');return
        total=sum(v for _,v in items)
        ax.pie([v for _,v in items],startangle=135,
               colors=[colors[i%len(colors)] for i in range(len(items))],
               autopct=lambda pct:f'{pct:.0f}%' if pct>=5 else '',pctdistance=.75,
               textprops={'fontsize':7,'color':'white'},
               wedgeprops={'linewidth':.7,'edgecolor':'white'})
        ax.legend([f'{name[:18]}  {100*v/total:.1f}%' for name,v in items],
                  loc='upper center',bbox_to_anchor=(.5,-.02),fontsize=6.7,
                  frameon=False,ncol=2 if len(items)>6 else 1)
        ax.set_aspect('equal')
    add_pie(ax1,display_near,near_colors,'Near misses by group')
    add_pie(ax2,allocation,['#4d9ff8','#18c4ac','#f3c66b','#a78bfa'],
            'Workforce allocation by role')
    fig.subplots_adjust(left=.03,right=.97,top=.90,bottom=.19,wspace=.21)
    story.append(figimage(fig,260))
    note=('For readability, smaller group shares are combined under Other groups in the PDF; '
          'the HTML displays each group separately. ' if len(positive)>12 else '')
    story.append(KeepTogether([
        heading('Near-miss share by group and workforce allocation'),
        figimage(fig,260),
        p(note+'Near-miss percentages show each group’s share of reported near-miss values, '
          'not confirmed incidents or per-employee risk. Workforce allocation percentages '
          'use reported T002/T003, PWT, SPL and PL values only; Late is excluded.','ReportSmall')
    ]))

def export_pdf(path, rows, scope='All Groups', exception_rows=None):
    """Board-ready PDF containing both source WhatsApp and staffing exceptions.

    Multi-page with charts, quality status, detailed exception register, and
    a bounded message appendix. The HTML export preserves *every* source row.
    """
    from io import BytesIO
    from xml.sax.saxutils import escape as X
    from matplotlib.figure import Figure
    from reportlab.lib import colors as C
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                     TableStyle, Image, PageBreak, KeepTogether)
    dataset=_build_report_export_payload(rows,exception_rows)
    e=dataset['exceptions'];r=dataset['reports'];t=dataset['totals']
    navy='#15263E';blue='#4d9ff8';teal='#18c4ac';purple='#a78bfa';red='#fb7185';gold='#f3c66b'
    series=[('before','Changes before 15:00',blue),('after','Changes after 15:00',teal),
            ('no_shows','No Shows',purple),('shortage','Shortages',red)]
    def display(value):
        return '—' if value is None else str(value)
    def shortened(name,n=23):
        return name[:n-1]+'…' if len(name)>n else name
    styles=getSampleStyleSheet()
    styles.add(ParagraphStyle(name='ReportTitle',parent=styles['Title'],fontName='Helvetica-Bold',fontSize=20,
                              leading=24,textColor=C.HexColor(navy),spaceAfter=9))
    styles.add(ParagraphStyle(name='ReportSection',parent=styles['Heading2'],fontName='Helvetica-Bold',fontSize=13,
                              leading=17,textColor=C.HexColor(navy),spaceBefore=15,spaceAfter=8))
    styles.add(ParagraphStyle(name='ReportBody',parent=styles['Normal'],fontSize=8.6,leading=12,
                              textColor=C.HexColor('#35475A'),spaceAfter=7))
    styles.add(ParagraphStyle(name='ReportSmall',parent=styles['Normal'],fontSize=7.2,leading=10,
                              textColor=C.HexColor('#53667A'),spaceAfter=7))
    styles.add(ParagraphStyle(name='ReportLabel',parent=styles['Normal'],fontName='Helvetica-Bold',fontSize=8,
                              textColor=C.HexColor('#62728A')))
    doc=SimpleDocTemplate(str(path),pagesize=A4,
         leftMargin=43,rightMargin=43,topMargin=48,bottomMargin=43,
         title='Operational Intelligence - Executive Reporting',author='Operational Intelligence')
    width=A4[0]-86
    def footer(canvas, _doc):
        canvas.saveState()
        canvas.setStrokeColor(C.HexColor('#D7E0E9'));canvas.line(43,37,A4[0]-43,37)
        canvas.setFont('Helvetica',7);canvas.setFillColor(C.HexColor('#586C80'))
        canvas.drawString(43,26,'INTERNAL • Operational Intelligence • Point-in-time snapshot')
        canvas.drawRightString(A4[0]-43,26,f'Page {_doc.page}')
        canvas.restoreState()
    def p(text,style='ReportBody'):
        return Paragraph(X(str(text)),styles[style])
    def heading(text):return Paragraph(X(text),styles['ReportSection'])
    def figimage(fig,h=190):
        stream=BytesIO();fig.savefig(stream,format='png',dpi=125,bbox_inches='tight',facecolor='white');stream.seek(0)
        image=Image(stream,width=width,height=h)
        image.hAlign='CENTER'
        return image
    def table(items,colWidths,small=7,firstrow=True):
        raw=[[str(v) for v in row] for row in items]
        tbl=Table(raw,colWidths=colWidths,repeatRows=1 if firstrow else 0,hAlign='LEFT')
        tbl.setStyle(TableStyle([
            ('BACKGROUND',(0,0),(-1,0),C.HexColor(navy)),
            ('TEXTCOLOR',(0,0),(-1,0),C.white),
            ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
            ('FONTSIZE',(0,0),(-1,-1),small),
            ('TOPPADDING',(0,0),(-1,-1),6),('BOTTOMPADDING',(0,0),(-1,-1),6),
            ('LINEBELOW',(0,0),(-1,-1),.3,C.HexColor('#E2E9F0')),
            ('ROWBACKGROUNDS',(0,1),(-1,-1),[C.white,C.HexColor('#F1F5F9')]),
            ('VALIGN',(0,0),(-1,-1),'TOP'),
        ]))
        return tbl
    story=[Paragraph('OPERATIONAL INTELLIGENCE',styles['ReportTitle']),
           p('EXECUTIVE WORKFORCE, OPERATIONS AND SAFETY REPORT','ReportLabel'),
           p('Scope: '+str(scope)+'  |  Generated: '+dataset['generated_at']+'  |  Source: local database','ReportSmall'),
           Spacer(1,9)]
    v=lambda k,kn:display(t[k]) if t[kn] else '—'
    cards=[('WhatsApp reports',str(t['reports'])),('Groups represented',str(t['groups'])),
           ('Before 15:00',v('changes_before','before_known')),('After 15:00',v('changes_after','after_known')),
           ('Explicit no-shows',v('no_shows','no_shows_known')),('Calculated shortages',v('shortage','shortage_known')),
           ('Pending nights',str(t['pending'])),('Near-miss reports',str(t['near_miss']))]
    def kpicell(label,value):
        return [Paragraph(X(label),styles['ReportLabel']),Paragraph(X(value),styles['Heading2'])]
    story.append(table([[f'{lab}\n{val}' for lab,val in cards[:4]],
                        [f'{lab}\n{val}' for lab,val in cards[4:]]],[width/4]*4,small=9,firstrow=False))
    story.extend([Spacer(1,13),heading('Management interpretation'),
        p('The WhatsApp message represents final actual night-shift attendance after staffing changes. '
          'Shortages are calculated separately for each required role against the active staffing plan; '
          'operator-entered changes and WhatsApp-reported no-shows are not deducted a second time.'),
        p(f"Confirmed shortage calculations: {t['shortage_known']} night(s). Pending nights: {t['pending']}. "
          f"Both change counts submitted: {t['changes_complete']} night(s). "
          f"Explicit no-show counts: {t['no_shows_known']} night(s). Unreported values remain unknown.",'ReportSmall'),
        heading('Workforce changes, absences and shortages')])
    grouped={}
    for item in e:
        g=grouped.setdefault(item['group'],{k:0 for k,_,_ in series})
        for k,_,_ in series:g[k]+=int(item[k] or 0)
    ranking=sorted(grouped.items(),key=lambda kv:(-sum(kv[1].values()),kv[0].casefold()))[:12]
    fig=Figure(figsize=(7.6,3.1 if ranking else 1.6));ax=fig.add_subplot(111)
    if ranking:
        y=list(range(len(ranking)));left=[0]*len(ranking)
        for k,label,color in series:
            values=[pair[1][k] for pair in ranking]
            ax.barh(y,values,left=left,label=label,color=color,height=.68)
            left=[a+b for a,b in zip(left,values)]
        ax.set_yticks(y,[shortened(pair[0],22) for pair in ranking],fontsize=8)
        ax.invert_yaxis();ax.legend(loc='upper center',bbox_to_anchor=(.5,1.20),ncol=4,fontsize=7,frameon=False)
        ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
    else:
        ax.text(.5,.5,'No exceptions in selected scope',ha='center',va='center',transform=ax.transAxes)
        ax.axis('off')
    fig.tight_layout();story.append(figimage(fig,200 if ranking else 115))
    story.append(p('Bars show four independent event categories, not unique people. Values of zero are displayed only when '
                   'the source is known; unreported no-show counts are not treated as confirmed zero.','ReportSmall'))
    story.append(heading('Daily workforce exception trend'))
    dates=sorted(set(x['date'] for x in e))[-14:]
    fig=Figure(figsize=(7.7,2.5));ax=fig.add_subplot(111)
    if dates:
        positions=list(range(len(dates)));bar_width=.18
        for index,(k,label,color) in enumerate(series):
            vals=[sum(int(x[k] or 0) for x in e if x['date']==d) for d in dates]
            ax.bar([q+(index-1.5)*bar_width for q in positions],vals,width=bar_width,label=label,color=color)
        ax.set_xticks(positions,[d[5:] for d in dates],fontsize=7,rotation=35)
        ax.legend(ncol=4,fontsize=7,frameon=False)
        ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    else:
        ax.text(.5,.5,'No dated exception entries',ha='center',va='center',transform=ax.transAxes);ax.axis('off')
    fig.tight_layout();story.append(figimage(fig,170))
    story.extend([heading('Role-specific staffing requirements and gaps')])
    totals={k:{'required':0,'present':0,'shortage':0} for k in dataset['roles']}
    for item in e:
        if item['shortage'] is None:continue
        for role in item['roles']:
            if role['role'] in totals:
                for key in ('required','present','shortage'):totals[role['role']][key]+=int(role[key])
    fig=Figure(figsize=(7.5,2.35));ax=fig.add_subplot(111)
    if t['shortage_known']:
        names=list(totals);y=list(range(len(names)))
        ax.barh([n+.18 for n in y],[totals[k]['present'] for k in names],height=.35,color=teal,label='Actual reported')
        ax.barh([n-.18 for n in y],[totals[k]['required'] for k in names],height=.35,color=blue,label='Required')
        ax.set_yticks(y,names);ax.invert_yaxis();ax.legend(frameon=False,fontsize=8)
        ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
    else:
        ax.text(.5,.5,'No role-level shortage calculation available',ha='center',va='center',transform=ax.transAxes);ax.axis('off')
    fig.tight_layout();story.append(figimage(fig,145))
    story.append(p('Role shortages: '+', '.join(f'{name}: {v["shortage"]}' for name,v in totals.items())+'. '
                   'A surplus in another role does not cancel a role-specific deficit.','ReportSmall'))
    story.append(heading('Exception register — latest service-date entries'))
    head=['Service date','Group','Before','After','No shows','Plan / Actual','Short','Status']
    lines=[]
    for item in e[:120]:
        lines.append([item['date'],shortened(item['group'],24),display(item['before']),display(item['after']),
                      display(item['no_shows']),f'{display(item["required"])} / {display(item["available"])}',
                      display(item['shortage']),item['status']])
    if lines:story.append(table([head]+lines,[65,94,40,40,49,75,32,57],small=6.5))
    else:story.append(p('No exception entries in this reporting scope.'))
    if len(e)>len(lines):story.append(p(f'Showing {len(lines)} of {len(e)} exception records; the interactive HTML includes all.','ReportSmall'))
    story.append(heading('WhatsApp attendance and safety reporting'))
    attendance=['T002/T003','PWT','SPL','PL']
    source_keys=['t002_t003','pwt','spl','pl']
    values=[sum(int(x[key] or 0) for x in r) for key in source_keys]
    fig=Figure(figsize=(7.6,2.3));ax=fig.add_subplot(111)
    ax.bar(attendance,values,color=[blue,teal,gold,purple])
    ax.set_ylabel('Sum of reported message values',fontsize=8)
    ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    fig.tight_layout();story.append(figimage(fig,155))
    story.append(p('Attendance is aggregated from WhatsApp report values; multiple reports may refer to the same people, '
                   'so this is not a deduplicated employee count.','ReportSmall'))
    _append_pdf_workforce_and_safety_charts(story,r,heading,p,figimage)
    story.append(heading('Near-miss and Late metrics'))
    fig=Figure(figsize=(7.5,2.4));ax=fig.add_subplot(111)
    near_dates=sorted({row['day'] for row in r})[-18:]
    if near_dates:
        near=[sum(int(q['near_miss'] or 0) for q in r if q['day']==day) for day in near_dates]
        late=[sum(int(q['late'] or 0) for q in r if q['day']==day) for day in near_dates]
        ax.plot(near_dates,near,label='Near Miss',color=red,marker='o',linewidth=1.2)
        ax.plot(near_dates,late,label='Late',color=blue,marker='s',linewidth=1.2)
        ax.tick_params(axis='x',labelrotation=45,labelsize=7);ax.legend(frameon=False)
        ax.grid(axis='y',alpha=.2)
    else:
        ax.text(.5,.5,'No WhatsApp safety reports in selected scope',ha='center',va='center',transform=ax.transAxes);ax.axis('off')
    fig.tight_layout();story.append(figimage(fig,175))
    story.append(p(f'Reported near-miss values: {t["near_miss"]}; Late values: {t["late"]}. '
                   'Both are source-reported indicators, not verified incident counts.','ReportSmall'))
    story.append(heading('Recent WhatsApp report evidence'))
    head=['Message date','Group','T002/T003','PWT','SPL','PL','Near Miss','Late']
    items=[]
    for row in r[:70]:
        items.append([row['when'][:16],shortened(row['group'],23)]+[display(row[key])
                 for key in source_keys+['near_miss','late']])
    if items:story.append(table([head]+items,[91,106,61,36,36,36,63,43],small=6.4))
    else:story.append(p('No WhatsApp reports in this period.'))
    if len(r)>len(items):story.append(p(f'Showing {len(items)} of {len(r)} received messages. The HTML export includes all.','ReportSmall'))
    story.append(heading('Definitions and data quality'))
    story.append(p('Before / after 15:00: counts manually saved by a signed-in coordinator. '
                   'No-show counts are included only when explicitly stated in a WhatsApp message. '
                   '“—” denotes unknown or not submitted; it is not zero. '
                   'Shortage: sum across staffing roles of max(0, required minus final actual attendance). '
                   'A night can have a known staffing shortage while its change-entry status remains incomplete. '
                   'After-midnight WhatsApp reports can be linked to their previous service date.'))
    doc.build(story,onFirstPage=footer,onLaterPages=footer)
    audit('PDF_EXPORT',f'{scope} / {len(e)} exception records')
    return path


_HTML_REPORT_TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Operational Intelligence | Executive report</title>
<style>
:root{--bg:#0b1422;--surface:#152338;--surface2:#1b2c43;--border:#324860;--text:#f1f6fc;--muted:#b2c5da;--blue:#4d9ff8;--teal:#18c4ac;--violet:#a78bfa;--red:#fb7185;--gold:#f3c66b;--good:#40d5a0}
*{box-sizing:border-box}body{margin:0;background:linear-gradient(120deg,#0a1220,#0e1b2b 55%,#0c1729);color:var(--text);font:14px/1.5 Inter,Segoe UI,system-ui,sans-serif}main{max-width:1450px;margin:0 auto;padding:26px 30px 65px}header{display:flex;justify-content:space-between;align-items:flex-start;gap:20px;padding:12px 0 25px;border-bottom:1px solid var(--border)}h1{font-size:30px;letter-spacing:-.03em;margin:2px 0 6px}h2{font-size:19px;letter-spacing:-.02em;margin:0 0 12px}h3{font-size:13px;color:#b3c7dd;letter-spacing:.05em;text-transform:uppercase;margin:0 0 12px}.eyebrow{font-size:11px;text-transform:uppercase;letter-spacing:.18em;color:var(--teal);font-weight:700}.muted,.hint{color:var(--muted)}.hint{font-size:12px}.pill{font-size:12px;border:1px solid #397565;padding:8px 12px;background:#163b35;border-radius:20px;white-space:nowrap}.controlbar{display:flex;flex-wrap:wrap;align-items:end;gap:11px;margin:21px 0 13px}.field{display:flex;flex-direction:column;gap:5px;min-width:115px}.field label{font-size:11px;color:var(--muted)}select,input,button{background:#1c3048;color:#fff;border:1px solid #47627f;padding:10px 12px;border-radius:9px;font:inherit;min-height:40px}select:focus,input:focus,button:focus{outline:2px solid var(--blue);outline-offset:2px}button{cursor:pointer;font-weight:600}button:hover{background:#29445f}.nowrap{white-space:nowrap}.scope{padding:11px 14px;border-radius:9px;background:#172c40;border-left:3px solid var(--blue);color:#c8d7e6;margin-bottom:18px}.cards{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:16px 0 23px}.card,.panel{background:var(--surface);border:1px solid var(--border);border-radius:14px;box-shadow:0 9px 25px #0001}.card{padding:16px 18px}.card .name{font-size:11px;text-transform:uppercase;letter-spacing:.075em;color:var(--muted)}.card .value{font-size:28px;letter-spacing:-.04em;font-weight:750;margin:3px 0;color:#f3f7fc}.card .sub{font-size:11px;color:var(--muted)}.sectionhead{margin-top:32px;margin-bottom:14px;display:flex;justify-content:space-between;align-items:end;gap:15px}.sectionhead p{margin:0;color:var(--muted)}.columns{display:grid;grid-template-columns:1.3fr 1fr;gap:13px}.panel{padding:18px;min-width:0}.legend{display:flex;flex-wrap:wrap;gap:14px;margin:7px 0 15px}.legend b{display:inline-block;width:10px;height:10px;border-radius:3px;margin-right:6px}.hrow{display:grid;grid-template-columns:minmax(110px,180px) 1fr 52px;gap:9px;align-items:center;padding:7px 0;border-bottom:1px solid #2b4056}.hlabel{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.track{height:17px;display:flex;overflow:hidden;border-radius:5px;background:#293a50}.segment{height:100%;min-width:0}.hvalue{text-align:right;color:var(--muted);font-variant-numeric:tabular-nums}.empty{padding:45px 12px;text-align:center;color:var(--muted)}svg{display:block;max-width:100%}.svgwrap{overflow-x:auto}.smallnote{color:var(--muted);font-size:11px;margin-top:12px}.tablewrap{overflow-x:auto;max-height:530px;overflow-y:auto;border:1px solid var(--border);border-radius:11px}table{border-collapse:separate;border-spacing:0;min-width:940px;width:100%;font-variant-numeric:tabular-nums}th{text-align:left;position:sticky;top:0;background:#233851;color:#f5f9ff;font-size:11px;letter-spacing:.02em;z-index:1}td,th{padding:11px 12px;border-bottom:1px solid #30445d}tbody tr:nth-child(even){background:#1b2a40}tbody tr:hover{background:#244161}.risk{color:#ff9cab;font-weight:750}.ok{color:#61d6ae}.warn{color:#f2cd86}.badge{display:inline-block;padding:3px 7px;border-radius:6px;background:#2a3b50;color:#dbebf9;font-size:11px}.note{padding:13px 15px;background:#253343;border:1px solid var(--border);color:#d0deeb;border-radius:11px;margin-top:18px}footer{color:var(--muted);font-size:11px;border-top:1px solid var(--border);margin-top:35px;padding-top:17px}#reportSearch{min-width:195px}
.gang-scroll{max-height:590px;overflow-y:auto;padding-right:8px;scrollbar-color:#45617c #19293d}
.gang-scroll .hrow{grid-template-columns:minmax(125px,200px) 1fr 88px}
.sharepie-layout{display:grid;grid-template-columns:minmax(175px,230px) minmax(0,1fr);gap:14px;align-items:center}
.sharepie-svg{width:100%;height:auto;max-width:230px}
.sharepie-legend{max-height:285px;overflow-y:auto;display:flex;flex-direction:column;gap:7px}
.sharepie-legend .item{display:flex;align-items:center;gap:8px;font-size:12px;color:var(--text);min-width:0}
.sharepie-legend .swatch{width:11px;height:11px;flex:none;border-radius:3px}
.sharepie-legend .item-label{overflow:hidden;text-overflow:ellipsis;white-space:nowrap;flex:1}
.sharepie-legend .item-val{flex:none;font-variant-numeric:tabular-nums;color:var(--muted)}
@media(max-width:650px){.sharepie-layout{grid-template-columns:1fr}.sharepie-svg{margin:0 auto}.gang-scroll .hrow{grid-template-columns:minmax(85px,130px) 1fr 58px}}
@media(max-width:980px){.cards{grid-template-columns:repeat(2,1fr)}.columns{grid-template-columns:1fr}.hrow{grid-template-columns:minmax(90px,150px) 1fr 40px}}@media(max-width:600px){main{padding:15px}h1{font-size:24px}header{flex-direction:column}.cards{grid-template-columns:repeat(2,minmax(0,1fr))}.card{padding:12px}.card .value{font-size:24px}.controlbar{gap:8px}.field{min-width:unset;flex:1 1 42%}.sectionhead{display:block}}
@media print{body{background:white;color:#152237}main{padding:10px}header,.panel,.card{break-inside:avoid;box-shadow:none}button,.controlbar{display:none}.panel,.card{background:white;color:#152237;border-color:#bec9d4}h2,h3,.card .value,.card .name{color:#152237}.smallnote,.muted,.hint{color:#46566b}}
</style></head>
<body><main>
<header><div><div class="eyebrow">Operational Intelligence / Board Reporting</div><h1>Workforce &amp; Operations</h1><div class="muted">Interactive executive report · generated __GENERATED__</div></div><div class="pill">INTERNAL · OFFLINE SNAPSHOT</div></header>
<div class="controlbar">
 <div class="field"><label for="team">WhatsApp group</label><select id="team"></select></div>
 <div class="field"><label for="period">Reporting dates</label><select id="period"><option value="all">All included dates</option><option value="7">Latest 7 included days</option><option value="14">Latest 14 included days</option><option value="30">Latest 30 included days</option><option value="custom">Custom range</option></select></div>
 <div class="field"><label for="from">From service / report date</label><input type="date" id="from"></div>
 <div class="field"><label for="to">To service / report date</label><input type="date" id="to"></div>
 <div class="field"><label for="reportSearch">Find a group / record</label><input type="search" id="reportSearch" placeholder="Search table rows"></div>
 <button id="reset">Reset filters</button><button id="download">Export filtered exceptions CSV</button>
</div>
<div class="scope">Exported scope: <b>__SCOPE__</b> · Dates in exception records are <b>night-shift service dates</b>, which may differ from the WhatsApp message timestamp. All charts use the embedded snapshot; this file does not update automatically.</div>
<div class="cards" id="cards"></div>
<div class="sectionhead"><div><div class="eyebrow">01 / Operational performance</div><h2>Staffing changes, no-shows &amp; shortages</h2><p>Independent event categories — totals are not unique employees.</p></div></div>
<div class="columns"><section class="panel"><h3>Four-category comparison · highest activity groups</h3><div id="legend" class="legend"></div><div id="exceptionBars"></div><div class="smallnote">A blank category may mean the value was not reported. Shortages are calculated per required role from final WhatsApp attendance and staffing plans.</div></section><section class="panel"><h3>Daily exception trend · service dates</h3><div id="exceptionTrend" class="svgwrap"></div><div class="smallnote">Daily counts use service dates. Unrecorded days are not represented as confirmed zero.</div></section></div>
<div class="columns" style="margin-top:13px"><section class="panel"><h3>Role-specific shortages · required vs present</h3><div id="roleBars"></div></section><section class="panel"><h3>Completeness &amp; reporting status</h3><div id="quality"></div></section></div>
<div class="sectionhead"><div><div class="eyebrow">02 / Source operations</div><h2>WhatsApp attendance &amp; safety</h2><p>Reported message values, not a deduplicated headcount of unique employees.</p></div></div>
<div class="columns"><section class="panel"><h3>Workforce role allocation · reported totals</h3><div id="attendanceBars"></div></section><section class="panel"><h3>Near-miss reporting trend · message dates</h3><div id="nearTrend" class="svgwrap"></div></section></div>
<div class="panel" style="margin-top:13px"><h3>Gang-wise attendance · stacked horizontal comparison</h3><div id="gangAttendanceLegend" class="legend"></div><div id="gangAttendance" class="gang-scroll"></div><p class="smallnote">Reported T002/T003, PWT, SPL and PL numbers by group. Late is an attendance indicator, not another staffing role; multiple message values are not deduplicated employees.</p></div>
<div class="columns" style="margin-top:13px"><section class="panel"><h3>Near misses percentage by group · pie chart</h3><div id="nearSharePie"></div><p class="smallnote">Share of reported near-miss values per group, not confirmed incident rates.</p></section><section class="panel"><h3>Workforce allocation percentage · pie chart</h3><div id="workforceSharePie"></div><p class="smallnote">Share of reported staffing-role values only; Late is excluded.</p></section></div>
<div class="sectionhead"><div><div class="eyebrow">03 / Evidence registers</div><h2>Staffing exceptions by group and night</h2><p id="exceptionsCount"></p></div></div>
<div class="panel"><div class="tablewrap"><table><thead><tr><th>Service date</th><th>Group</th><th>Before 15:00</th><th>After 15:00</th><th>No-shows</th><th>Required</th><th>Present</th><th>Shortage</th><th>Status</th></tr></thead><tbody id="exceptionsTable"></tbody></table></div><p class="smallnote">“—” means unknown or not submitted, rather than zero. Pending shortages must not be treated as confirmed staffing compliance.</p></div>
<div class="sectionhead"><div><h2>WhatsApp report evidence</h2><p id="reportsCount"></p></div></div>
<div class="panel"><div class="tablewrap"><table><thead><tr><th>Message date/time</th><th>Group</th><th>Sender</th><th>T002/T003</th><th>PWT</th><th>SPL</th><th>PL</th><th>Near miss</th><th>Late</th><th>No-shows reported</th></tr></thead><tbody id="reportsTable"></tbody></table></div></div>
<div class="note"><b>Interpretation:</b> Changes before / after 15:00 are operator-entered event counts. The WhatsApp message is the final actual attendance snapshot for a night shift. Changes and no-shows must not be subtracted again from that headcount. A no-show value is shown only when explicitly stated in WhatsApp. Excess staff in one role do not cancel shortages in another role. Unknowns stay unknown.</div>
<footer>Prepared from the locally stored Operational Intelligence database. This standalone HTML contains a point-in-time copy of report-level data and may contain internal operational information. Share only with authorised recipients. For the latest figures, re-export from the application.</footer>
</main>
<script>
'use strict';
const DATA = __PAYLOAD__;
const $ = id => document.getElementById(id);
const esc = s => String(s ?? '').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const fmt = v => v === null || v === undefined ? '—' : Number(v).toLocaleString('en-GB');
const COLOR={before:'#4d9ff8',after:'#18c4ac',no_shows:'#a78bfa',shortage:'#fb7185'};
const SOURCES=[['before','Before 15:00'],['after','After 15:00'],['no_shows','No Shows'],['shortage','Shortages']];
const uniqueDates=[...new Set([...DATA.exceptions.map(x=>x.date),...DATA.reports.map(x=>x.day)])].sort();
const newest=uniqueDates[uniqueDates.length-1] || '';
const names=['All Groups',...new Set([...DATA.exceptions.map(x=>x.group),...DATA.reports.map(x=>x.group)])];
$('team').innerHTML=names.map(n=>'<option value="'+esc(n)+'">'+esc(n)+'</option>').join('');
function minusDays(iso, n){const d=new Date(iso+'T12:00:00Z');d.setUTCDate(d.getUTCDate()-n);return d.toISOString().slice(0,10)}
function matchesDay(d){let p=$('period').value;if(p==='custom')return (!$('from').value||d>=$('from').value)&&(!$('to').value||d<=$('to').value);if(p!=='all'&&newest)return d>=minusDays(newest,Number(p)-1)&&d<=newest;return true}
function filtered(){let t=$('team').value.toLocaleLowerCase('en-GB');return {e:DATA.exceptions.filter(x=>(t==='all groups'||x.group.toLocaleLowerCase('en-GB')===t)&&matchesDay(x.date)),r:DATA.reports.filter(x=>(t==='all groups'||x.group.toLocaleLowerCase('en-GB')===t)&&matchesDay(x.day))}}
function total(arr,k){return arr.reduce((s,x)=>s+Number(x[k]??0),0)}
function card(name,value,sub){return '<div class="card"><div class="name">'+esc(name)+'</div><div class="value">'+esc(value)+'</div><div class="sub">'+esc(sub)+'</div></div>'}
function bars(rows,metrics,max=12){if(!rows.length)return '<div class="empty">No records for the selected period.</div>';const w=rows.slice(0,max);const largest=Math.max(1,...w.map(x=>metrics.reduce((a,m)=>a+Number(x[m]??0),0)));return w.map(x=>{const sum=metrics.reduce((a,m)=>a+Number(x[m]??0),0);const seg=metrics.map(m=>{const v=Number(x[m]??0);return v?'<div class="segment" style="width:'+(v/largest*100)+'%;background:'+COLOR[m]+'" title="'+esc(m)+': '+v+'"></div>':''}).join('');return '<div class="hrow"><span class="hlabel" title="'+esc(x.group)+'">'+esc(x.group)+'</span><div class="track">'+seg+'</div><span class="hvalue">'+sum+'</span></div>'}).join('')}
function trend(rows,fields,daysLimit=14){if(!rows.length)return '<div class="empty">No observations to chart.</div>';let d=[...new Set(rows.map(x=>x.date))].sort().slice(-daysLimit);const vals=d.map(day=>{let r=rows.filter(x=>x.date===day);return fields.map(f=>r.reduce((s,x)=>s+Number(x[f]??0),0))});const peak=Math.max(1,...vals.flat());const W=560,H=244,left=38,top=15,bottom=48,plotH=H-top-bottom, groupW=(W-left-8)/d.length, barW=Math.max(2,Math.min(12,groupW/(fields.length+1)));let svg='<svg viewBox="0 0 '+W+' '+H+'" role="img" aria-label="Daily trends">';for(let k=0;k<=4;k++){let y=top+plotH*k/4;svg+='<line x1="'+left+'" y1="'+y+'" x2="'+(W-4)+'" y2="'+y+'" stroke="#395068" stroke-dasharray="3 3"/><text x="4" y="'+(y+4)+'" fill="#abc0d7" font-size="10">'+Math.round(peak*(4-k)/4)+'</text>'}d.forEach((day,i)=>{fields.forEach((f,j)=>{let n=vals[i][j],height=n/peak*plotH,x=left+i*groupW+4+j*barW,y=top+plotH-height;svg+='<rect x="'+x+'" y="'+y+'" width="'+(barW-1)+'" height="'+height+'" fill="'+COLOR[f]+'"><title>'+esc(day)+' '+esc(f)+': '+n+'</title></rect>'});if(d.length<18||i%2===0)svg+='<text transform="translate('+(left+i*groupW+5)+','+(H-bottom+13)+') rotate(45)" fill="#b6c8da" font-size="10">'+esc(day.slice(5))+'</text>'});return svg+'</svg>'}
function singleTrend(rows){let byday={};rows.forEach(r=>{byday[r.day]=(byday[r.day]||0)+Number(r.near_miss??0)});const d=Object.keys(byday).sort().slice(-20);if(!d.length)return '<div class="empty">No received reports.</div>';const max=Math.max(1,...d.map(x=>byday[x]));let svg='<svg viewBox="0 0 540 228" role="img" aria-label="Near miss values by date">';d.forEach((day,i)=>{let x=30+i*25,h=byday[day]/max*160;svg+='<rect x="'+x+'" y="'+(174-h)+'" width="16" height="'+h+'" rx="3" fill="#f3c66b"><title>'+esc(day)+' · '+byday[day]+' near miss values</title></rect>';if(i%2===0)svg+='<text transform="translate('+x+',183) rotate(45)" fill="#abc0d7" font-size="10">'+esc(day.slice(5))+'</text>'});return svg+'</svg>'}
// v3.0.0: offline Workforce/Safety chart parity. Uses the existing filtered reports.
const GANG_FIELDS=[['t002_t003','T002/T003','#2980b9'],['pwt','PWT','#2ecc71'],['spl','SPL','#e67e22'],['pl','PL','#e74c3c'],['late','Late','#f1c40f']];
const SHARE_COLORS=['#d35400','#2980b9','#27ae60','#8e44ad','#f39c12','#16a085','#c0392b','#597cbb','#ae7ac0','#b9a045','#7a9e49','#e18a71'];
function sharePieHTML(items, colors, title){
 const data=items.filter(x=>Number(x.value)>0).sort((a,b)=>b.value-a.value||a.name.localeCompare(b.name));
 const sum=data.reduce((acc,x)=>acc+Number(x.value),0);
 if(!sum)return '<div class="empty">No positive reported values in the selected period.</div>';
 const cx=120,cy=120,R=93;let a=-Math.PI/2;
 const shapes=data.map((x,i)=>{let fraction=Number(x.value)/sum,delta=2*Math.PI*fraction,color=colors[i%colors.length];
  let shape='';if(fraction>=.999999){shape='<circle cx="120" cy="120" r="93" fill="'+color+'"/>'}
  else{const b=a+delta,p1=[cx+R*Math.cos(a),cy+R*Math.sin(a)],p2=[cx+R*Math.cos(b),cy+R*Math.sin(b)];
   const d='M '+cx+' '+cy+' L '+p1[0].toFixed(3)+' '+p1[1].toFixed(3)+' A '+R+' '+R+' 0 '+(delta>Math.PI?1:0)+' 1 '+p2[0].toFixed(3)+' '+p2[1].toFixed(3)+' Z';shape='<path d="'+d+'" fill="'+color+'" stroke="#152338" stroke-width="1"/>'}
  a+=delta;return '<g>'+shape+'<title>'+esc(x.name)+' · '+x.value+' ('+(100*fraction).toFixed(1)+'%)</title></g>'
 }).join('');
 const legend=data.map((x,i)=>'<div class="item" title="'+esc(x.name)+'"><span class="swatch" style="background:'+colors[i%colors.length]+'"></span><span class="item-label">'+esc(x.name)+'</span><span class="item-val">'+(100*x.value/sum).toFixed(1)+'%</span></div>').join('');
 return '<div class="sharepie-layout"><svg class="sharepie-svg" viewBox="0 0 240 240" role="img" aria-label="'+esc(title)+'">'+shapes+'</svg><div class="sharepie-legend">'+legend+'</div></div>';
}
function gangAttendanceHTML(rows){
 const groups=new Map();rows.forEach(r=>{if(!groups.has(r.group))groups.set(r.group,{});let sums=groups.get(r.group);GANG_FIELDS.forEach(([key])=>sums[key]=(sums[key]||0)+Number(r[key]??0))});
 const entries=[...groups.entries()].sort((a,b)=>a[0].localeCompare(b[0]));
 if(!entries.length)return '<div class="empty">No WhatsApp attendance records for the selected period.</div>';
 const peak=Math.max(1,...entries.map(([g,vals])=>GANG_FIELDS.reduce((s,[key])=>s+Number(vals[key]||0),0)));
 return entries.map(([group,vals])=>{const staff=['t002_t003','pwt','spl','pl'].reduce((s,key)=>s+Number(vals[key]||0),0);
  const segs=GANG_FIELDS.map(([key,label,color])=>{const v=Number(vals[key]||0);return v?'<div class="segment" style="width:'+(v/peak*100)+'%;background:'+color+'" title="'+esc(group+' · '+label+': '+v)+'"></div>':''}).join('');
  return '<div class="hrow"><span class="hlabel" title="'+esc(group)+'">'+esc(group)+'</span><div class="track">'+segs+'</div><span class="hvalue">'+fmt(staff)+' staff</span></div>';
 }).join('');
}
function refresh(){const {e,r}=filtered();const knownNo=e.filter(x=>x.no_shows!==null),knownShort=e.filter(x=>x.shortage!==null);const groups=new Set([...e.map(x=>x.group),...r.map(x=>x.group)]);const pending=e.filter(x=>x.shortage===null).length;
$('cards').innerHTML=[card('WhatsApp reports',r.length,'Received messages in selected scope'),card('Groups represented',groups.size,'Across reports and exception entries'),card('Changes before 15:00',e.some(x=>x.before!==null)?fmt(total(e,'before')):'—',e.filter(x=>x.before!==null).length+' submitted entries'),card('Changes after 15:00',e.some(x=>x.after!==null)?fmt(total(e,'after')):'—',e.filter(x=>x.after!==null).length+' submitted entries'),card('Explicit no-shows',knownNo.length?fmt(total(knownNo,'no_shows')):'—',knownNo.length+' attendance snapshots with counts'),card('Staffing shortages',knownShort.length?fmt(total(knownShort,'shortage')):'—',knownShort.length+' calculated nights'),card('Nights awaiting review',pending,'No confirmed shortage until report/plan is valid'),card('Near-miss values',fmt(total(r,'near_miss')),'Unverified reported values')].join('');
$('legend').innerHTML=SOURCES.map(([k,n])=>'<span><b style="background:'+COLOR[k]+'"></b>'+esc(n)+'</span>').join('');
let group={};e.forEach(x=>{let g=group[x.group]||(group[x.group]={group:x.group,before:0,after:0,no_shows:0,shortage:0});SOURCES.forEach(([k])=>g[k]+=Number(x[k]??0))});let g=Object.values(group).sort((a,b)=>SOURCES.reduce((s,[k])=>s+b[k]-a[k],0)||a.group.localeCompare(b.group));$('exceptionBars').innerHTML=bars(g,SOURCES.map(s=>s[0]));
$('exceptionTrend').innerHTML=trend(e,SOURCES.map(s=>s[0]));
const roles=['T002/T003','PWT','SPL','PL'];let role=Object.fromEntries(roles.map(k=>[k,{group:k,required:0,present:0,shortage:0}]));knownShort.forEach(x=>x.roles.forEach(y=>{const v=role[y.role];if(v){v.required+=Number(y.required||0);v.present+=Number(y.present||0);v.shortage+=Number(y.shortage||0)}}));const peak=Math.max(1,...roles.map(x=>role[x].required));$('roleBars').innerHTML=knownShort.length?roles.map(k=>{const v=role[k];return '<div class="hrow"><span>'+esc(k)+'</span><div class="track"><div class="segment" style="width:'+(v.present/peak*100)+'%;background:#18c4ac"></div><div class="segment" style="width:'+(v.shortage/peak*100)+'%;background:#fb7185"></div></div><span class="hvalue">'+v.shortage+' short</span></div>'}).join('')+'<div class="smallnote">Teal = present; coral = shortage, against planned role requirements. Surplus in a role may extend the teal bar.</div>':'<div class="empty">No calculable staffing snapshots in this period.</div>';
$('quality').innerHTML='<p><b>'+knownShort.length+'</b> nights with verified staffing calculations</p><p><b>'+pending+'</b> nights with shortages pending</p><p><b>'+e.filter(x=>x.before!==null&&x.after!==null).length+'</b> nights with both change counts entered</p><p><b>'+knownNo.length+'</b> nights with explicit WhatsApp no-show counts</p><p class="smallnote">Unknown values are not interpreted as zero. Reported dates may differ from service dates for overnight work.</p>';
const attendance=roles.map((k,i)=>({group:k,n:total(r,['t002_t003','pwt','spl','pl'][i])}));const mx=Math.max(1,...attendance.map(x=>x.n));$('attendanceBars').innerHTML=attendance.map((x,i)=>'<div class="hrow"><span>'+esc(x.group)+'</span><div class="track"><div class="segment" style="width:'+(x.n/mx*100)+'%;background:'+['#4d9ff8','#18c4ac','#f3c66b','#a78bfa'][i]+'"></div></div><span class="hvalue">'+fmt(x.n)+'</span></div>').join('')+'<p class="smallnote">Sums of message values; do not interpret as unique people across multiple reports.</p>';
$('nearTrend').innerHTML=singleTrend(r);
$('gangAttendanceLegend').innerHTML=GANG_FIELDS.map(([key,label,color])=>'<span><b style="background:'+color+'"></b>'+esc(label)+'</span>').join('');
$('gangAttendance').innerHTML=gangAttendanceHTML(r);
const nearGroups=new Map();r.forEach(x=>nearGroups.set(x.group,(nearGroups.get(x.group)||0)+Number(x.near_miss??0)));
$('nearSharePie').innerHTML=sharePieHTML([...nearGroups].map(([name,value])=>({name,value})),SHARE_COLORS,'Reported near-miss percentages by group');
$('workforceSharePie').innerHTML=sharePieHTML([['t002_t003','T002/T003'],['pwt','PWT'],['spl','SPL'],['pl','PL']].map(([key,name])=>({name,value:total(r,key)})),['#4d9ff8','#18c4ac','#f3c66b','#a78bfa'],'Workforce allocation percentage by reported role');

const q=$('reportSearch').value.toLocaleLowerCase('en-GB');const ex=e.filter(x=>!q||(x.group+' '+x.date+' '+x.status).toLocaleLowerCase('en-GB').includes(q));const reports=r.filter(x=>!q||(x.group+' '+x.day+' '+x.sender).toLocaleLowerCase('en-GB').includes(q));
$('exceptionsCount').textContent=ex.length+' of '+e.length+' entries displayed. Dates follow the shift service date.';$('reportsCount').textContent=reports.length+' of '+r.length+' reports displayed.';
$('exceptionsTable').innerHTML=ex.map(x=>'<tr><td>'+esc(x.date)+'</td><td>'+esc(x.group)+'</td><td>'+fmt(x.before)+'</td><td>'+fmt(x.after)+'</td><td>'+fmt(x.no_shows)+'</td><td>'+fmt(x.required)+'</td><td>'+fmt(x.available)+'</td><td class="'+(x.shortage>0?'risk':x.shortage===0?'ok':'warn')+'">'+fmt(x.shortage)+'</td><td><span class="badge" title="'+esc(x.reason)+'">'+esc(x.status)+'</span></td></tr>').join('')||'<tr><td colspan="9">No matching exceptions</td></tr>';
$('reportsTable').innerHTML=reports.map(x=>'<tr><td>'+esc(x.when)+'</td><td>'+esc(x.group)+'</td><td>'+esc(x.sender)+'</td>'+['t002_t003','pwt','spl','pl','near_miss','late','no_shows'].map(k=>'<td>'+fmt(x[k])+'</td>').join('')+'</tr>').join('')||'<tr><td colspan="10">No matching reports</td></tr>';
window.__reportRows={ex,reports};
}
['team','period','from','to'].forEach(id=>$(id).addEventListener('change',()=>{if(id==='from'||id==='to')$('period').value='custom';refresh()}));$('reportSearch').addEventListener('input',refresh);
$('reset').onclick=()=>{$('team').value='All Groups';$('period').value='all';$('from').value='';$('to').value='';$('reportSearch').value='';refresh()};
$('download').onclick=()=>{const cols=['date','group','before','after','no_shows','required','available','shortage','status','reason'];let rows=window.__reportRows?.ex||[];const csv=[cols.join(','),...rows.map(row=>cols.map(k=>'"'+String(row[k]??'').replace(/"/g,'""')+'"').join(','))].join('\r\n');const link=document.createElement('a');link.href=URL.createObjectURL(new Blob([csv],{type:'text/csv;charset=utf-8'}));link.download='staffing_exceptions_filtered.csv';link.click();setTimeout(()=>URL.revokeObjectURL(link.href),3000)};
refresh();
</script></body></html>
"""

def demo_data(days=42,seed=42):
    rng=random.Random(seed);n=0
    for i in range(days):
        d=date.today()-timedelta(days=days-i-1)
        for g in configuration()['groups']:
            for shift in (0,1):
                vals={'T002/T003':rng.randint(5,12),'PWT':rng.randint(2,6),'SPL':rng.randint(1,4),
                      'PL':rng.randint(0,2),'Near Miss':int(rng.random()<0.04), 'Late':rng.randint(0,2)}
                msg=' '.join(f'{k}: {v}' for k,v in vals.items())
                dt=datetime.combine(d,datetime.min.time()).replace(hour=7+shift*8,minute=rng.randint(0,40))
                n+=insert_report(g,'DEMO Generator',msg,dt,vals,origin='demo')
    audit('DEMO_SEED',f'{n} new labelled synthetic records')
    return n

init_db()


## Reporting parity — PDF and Interactive HTML

The export functions share the Executive Exceptions calculation. Early entries are shown even without WhatsApp messages; unknown values remain unknown. The HTML is a standalone snapshot, not a continuously refreshed live dashboard.

**Export workflow:** choose reporting period and group in the application, click **Export PDF** or **Interactive HTML**. The downloaded HTML includes all matching exception records and WhatsApp source rows with group/date filtering and an exception CSV export.


In [ ]:
# ---------------------------------------------------------------------------
# OPERATIONAL EXCEPTIONS / PLANNED STAFFING (native Python + SQLite)
# Added to the original notebook immediately after the core database cell.
# Excel input is read-only ZIP/XML; hidden worksheets and macros are ignored.
# ---------------------------------------------------------------------------
import hashlib as _ex_hashlib
import hmac as _ex_hmac
import json as _ex_json
import re as _ex_re
import secrets as _ex_secrets
import zipfile as _ex_zipfile
import xml.etree.ElementTree as _ex_et
from datetime import datetime as _ex_datetime, timedelta as _ex_timedelta
from pathlib import Path as _ex_Path

EX_ROLES = ('before',)  # Reuse Operator 1 account; historical Operator 2 is retained read-only.
EX_SHIFTS = ('Shift 1', 'Shift 2', 'Shift 3')
EX_BASIS = ('pre_adjustments', 'already_actual')
EX_METRIC_FIELDS = {'T002/T003': 't002_t003', 'PWT': 'pwt', 'SPL': 'spl', 'PL': 'pl'}
EX_WORKBOOK_NS = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
                  'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}


def init_exception_db():
    """Independent tables; leaves existing WhatsApp reports and its fingerprint alone."""
    with connect() as c:
        c.executescript('''
        CREATE TABLE IF NOT EXISTS staffing_plans (
          group_name TEXT NOT NULL, worksheet TEXT NOT NULL,
          effective_from TEXT NOT NULL, effective_to TEXT NOT NULL,
          components_json TEXT NOT NULL, planned_total INTEGER NOT NULL,
          workbook_name TEXT NOT NULL, source_cell TEXT NOT NULL,
          imported_at TEXT NOT NULL,
          PRIMARY KEY (group_name, worksheet, effective_from, effective_to)
        );
        CREATE INDEX IF NOT EXISTS idx_staffing_plans_dates
          ON staffing_plans(group_name, effective_from, effective_to);

        CREATE TABLE IF NOT EXISTS manual_staffing_plans (
          group_name TEXT NOT NULL COLLATE NOCASE,
          effective_from TEXT NOT NULL,
          components_json TEXT NOT NULL,
          planned_total INTEGER NOT NULL CHECK(planned_total > 0),
          updated_at TEXT NOT NULL,
          updated_by TEXT NOT NULL,
          PRIMARY KEY (group_name,effective_from)
        );
        CREATE INDEX IF NOT EXISTS idx_manual_staffing_effective
          ON manual_staffing_plans(group_name,effective_from);
        CREATE TABLE IF NOT EXISTS manual_staffing_plan_audit (
          id INTEGER PRIMARY KEY AUTOINCREMENT,
          changed_at TEXT NOT NULL,
          actor TEXT NOT NULL,
          operation TEXT NOT NULL,
          group_name TEXT NOT NULL,
          effective_from TEXT NOT NULL,
          old_state TEXT,
          new_state TEXT
        );
        CREATE TABLE IF NOT EXISTS exception_users (
          role TEXT PRIMARY KEY CHECK(role IN ('before','after')),
          display_name TEXT NOT NULL,
          password_salt TEXT NOT NULL, password_hash TEXT NOT NULL,
          created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS app_users (
          username TEXT PRIMARY KEY COLLATE NOCASE,
          display_name TEXT NOT NULL,
          account_role TEXT NOT NULL CHECK(account_role IN ('admin','operator')),
          password_salt TEXT NOT NULL, password_hash TEXT NOT NULL,
          recovery_salt TEXT NOT NULL, recovery_hash TEXT,
          failed_attempts INTEGER NOT NULL DEFAULT 0,
          locked_until TEXT,
          recovery_used INTEGER NOT NULL DEFAULT 0,
          enabled INTEGER NOT NULL DEFAULT 1,
          created_at TEXT NOT NULL, updated_at TEXT NOT NULL, last_login TEXT
        );
        CREATE TABLE IF NOT EXISTS exception_entries (
          group_name TEXT NOT NULL, service_date TEXT NOT NULL,
          shift_label TEXT NOT NULL CHECK(shift_label IN ('Shift 1','Shift 2','Shift 3')),
          before_count INTEGER CHECK(before_count >= 0),
          before_impact INTEGER,
          after_count INTEGER CHECK(after_count >= 0),
          after_impact INTEGER,
          no_shows INTEGER CHECK(no_shows >= 0),
          basis TEXT NOT NULL DEFAULT 'already_actual'
              CHECK(basis IN ('pre_adjustments','already_actual')),
          report_id INTEGER,
          before_by TEXT, before_at TEXT, after_by TEXT, after_at TEXT,
          revision INTEGER NOT NULL DEFAULT 0, updated_at TEXT NOT NULL,
          PRIMARY KEY (group_name, service_date, shift_label)
        );
        CREATE INDEX IF NOT EXISTS idx_exception_date ON exception_entries(service_date);
        CREATE UNIQUE INDEX IF NOT EXISTS idx_exception_snapshot_unique
          ON exception_entries(report_id) WHERE report_id IS NOT NULL;
        CREATE TABLE IF NOT EXISTS exception_sync_state (
          id TEXT PRIMARY KEY, revision INTEGER NOT NULL DEFAULT 0
        );
        INSERT OR IGNORE INTO exception_sync_state(id,revision) VALUES('plans',0);
        CREATE TABLE IF NOT EXISTS exception_entry_audit (
          id INTEGER PRIMARY KEY AUTOINCREMENT,
          changed_at TEXT NOT NULL, group_name TEXT NOT NULL,
          service_date TEXT NOT NULL, shift_label TEXT NOT NULL,
          actor_role TEXT NOT NULL, actor_name TEXT NOT NULL,
          old_state TEXT NOT NULL, new_state TEXT NOT NULL
        );
        ''')


def exception_users_ready():
    # Legacy Operator 1 login becomes the sole change-entry login.
    with connect() as c:
        return c.execute("SELECT 1 FROM exception_users WHERE role='before'").fetchone() is not None


def configure_exception_users(name, pin):
    # Create one operator login; preserve legacy accounts and history.
    name, pin = str(name).strip(), str(pin)
    if not name or len(name) > 100:
        raise ValueError('Enter an operator name (maximum 100 characters).')
    if not 6 <= len(pin) <= 128:
        raise ValueError('Use a passcode of 6 to 128 characters.')
    salt = _ex_secrets.token_bytes(16)
    digest = _ex_hashlib.pbkdf2_hmac('sha256', pin.encode('utf-8'), salt, 230000)
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        if c.execute("SELECT 1 FROM exception_users WHERE role='before'").fetchone():
            raise ValueError('Change-entry login already exists. Use its existing passcode.')
        c.execute('''INSERT INTO exception_users(role,display_name,password_salt,password_hash,created_at)
          VALUES('before',?,?,?,?)''',(name,salt.hex(),digest.hex(),stamp()))
    audit('CHANGE_OPERATOR_SETUP', 'Single changes operator account configured')


def authenticate_exception_user(role, pin):
    if role not in EX_ROLES:
        raise ValueError('Unknown operator role.')
    with connect() as c:
        row = c.execute('SELECT * FROM exception_users WHERE role=?', (role,)).fetchone()
    if row is None:
        raise PermissionError('The exception operator accounts are not configured.')
    digest = _ex_hashlib.pbkdf2_hmac('sha256', str(pin).encode('utf-8'),
                                     bytes.fromhex(row['password_salt']), 230000)
    if not _ex_hmac.compare_digest(digest, bytes.fromhex(row['password_hash'])):
        raise PermissionError('Incorrect passcode.')
    return row['display_name']


# ===========================================================================
# Local multi-user exception entry (does not require an external API).
# One-time recovery codes are revealed only at account creation/rotation.
# This is a local-workstation access control, not a remote authentication server.
# ===========================================================================
from datetime import timezone as _ex_timezone

_PASSWORD_ITERATIONS = 260000
# Volatile per-process sessions: tokens are never stored in SQLite or settings.json.
_APP_SESSIONS = {}
_SESSION_IDLE_MINUTES = 12 * 60


def app_session_info(token):
    info = _APP_SESSIONS.get(str(token))
    if not info:
        raise PermissionError('Your session has expired. Sign in again.')
    now = _ex_datetime.now(_ex_timezone.utc)
    if (now-info['last_used']).total_seconds() > _SESSION_IDLE_MINUTES*60:
        _APP_SESSIONS.pop(str(token),None)
        raise PermissionError('Session expired. Sign in again.')
    with connect() as c:
        row = c.execute('SELECT account_role,enabled,display_name FROM app_users WHERE username=?',
                        (info['username'],)).fetchone()
    if not row or not row['enabled']:
        _APP_SESSIONS.pop(str(token),None)
        raise PermissionError('Account disabled or unavailable. Sign in again.')
    info['last_used'] = now
    return {'username':info['username'],'display_name':row['display_name'],
            'role':row['account_role'],'token':str(token)}


def revoke_app_session(token):
    _APP_SESSIONS.pop(str(token),None)


def _revoke_user_sessions(username):
    for key,item in list(_APP_SESSIONS.items()):
        if item['username'].casefold()==str(username).casefold():
            _APP_SESSIONS.pop(key,None)

_USERNAME_PATTERN = _ex_re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]{2,31}\Z")


def _validate_username(name):
    name = str(name).strip()
    if not _USERNAME_PATTERN.fullmatch(name):
        raise ValueError('Username must be 3–32 characters: letters, numbers, dots, dashes or underscores.')
    return name


def _validate_password(password):
    password = str(password)
    if len(password) < 12 or len(password) > 128:
        raise ValueError('Password must be 12–128 characters long.')
    if password.strip() != password:
        raise ValueError('Password cannot begin or end with spaces.')
    return password


def _pbkdf2(value, salt):
    return _ex_hashlib.pbkdf2_hmac('sha256', value.encode('utf-8'), salt, _PASSWORD_ITERATIONS).hex()


def _new_credential(value):
    salt = _ex_secrets.token_bytes(16)
    return salt.hex(), _pbkdf2(value, salt)


def _new_recovery():
    code = _ex_secrets.token_hex(20)  # 160-bit random, kept out of logs.
    salt, digest = _new_credential(code)
    return code, salt, digest


def _normalise_recovery(code):
    return _ex_re.sub(r'[\s-]+','',str(code)).lower()


def app_users_ready():
    with connect() as c:
        return bool(c.execute('SELECT 1 FROM app_users LIMIT 1').fetchone())


def legacy_operator_migration_required():
    with connect() as c:
        old = c.execute("SELECT 1 FROM exception_users WHERE role='before'").fetchone()
        new = c.execute('SELECT 1 FROM app_users LIMIT 1').fetchone()
    return bool(old and not new)


def _make_app_user(c, username, display_name, role, password):
    username = _validate_username(username)
    display_name = str(display_name).strip()
    if not display_name or len(display_name)>100:
        raise ValueError('Enter a display name up to 100 characters.')
    if role not in ('admin','operator'):
        raise ValueError('Account role must be administrator or operator.')
    password = _validate_password(password)
    ps, ph = _new_credential(password)
    code, rs, rh = _new_recovery()
    now = stamp()
    c.execute('''INSERT INTO app_users
      (username,display_name,account_role,password_salt,password_hash,
       recovery_salt,recovery_hash,created_at,updated_at)
       VALUES(?,?,?,?,?,?,?,?,?)''',
       (username,display_name,role,ps,ph,rs,rh,now,now))
    return code


def bootstrap_admin_account(username, display_name, password, legacy_passcode=None):
    '''First-run administration; legacy Operator 1 passcode is required on upgrade.'''
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        if c.execute('SELECT 1 FROM app_users LIMIT 1').fetchone():
            raise PermissionError('An administrator already exists. Please sign in.')
        legacy = c.execute("SELECT * FROM exception_users WHERE role='before'").fetchone()
        if legacy:
            digest = _ex_hashlib.pbkdf2_hmac('sha256', str(legacy_passcode or '').encode('utf-8'),
                                     bytes.fromhex(legacy['password_salt']),230000)
            if not _ex_hmac.compare_digest(digest,bytes.fromhex(legacy['password_hash'])):
                raise PermissionError('The existing Operator 1 passcode is required to upgrade accounts.')
        code = _make_app_user(c,username,display_name,'admin',password)
    audit('APP_ADMIN_BOOTSTRAP','Administrator account created; credentials not logged')
    return code


def _require_admin(c, actor):
    username = app_session_info(actor)['username']
    current = c.execute('SELECT account_role,enabled FROM app_users WHERE username=?', (username,)).fetchone()
    if not current or not current['enabled'] or current['account_role']!='admin':
        raise PermissionError('An administrator account is required.')


def list_app_users():
    with connect() as c:
        return [dict(x) for x in c.execute('''SELECT username,display_name,account_role,enabled,
             last_login,created_at FROM app_users ORDER BY account_role,username COLLATE NOCASE''')]


def create_app_user(actor, username, display_name, role, password):
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        _require_admin(c,actor)
        code = _make_app_user(c,username,display_name,role,password)
    audit('APP_USER_CREATED',f'{app_session_info(actor)["username"]} added {username} ({role})')
    return code


def authenticate_app_user(username,password):
    name = str(username).strip()
    now = _ex_datetime.now(_ex_timezone.utc)
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        row = c.execute('SELECT * FROM app_users WHERE username=?', (name,)).fetchone()
        if not row:
            # Similar computational work for an unknown username.
            _pbkdf2(str(password),bytes(16))
            raise PermissionError('Invalid username or password.')
        if not row['enabled']:
            raise PermissionError('Account disabled. Contact an administrator.')
        if row['locked_until']:
            try: locked = _ex_datetime.fromisoformat(row['locked_until'])
            except ValueError: locked = now
            if locked>now:
                raise PermissionError('Too many sign-in attempts. Try again in 15 minutes or use recovery.')
        actual = _pbkdf2(str(password),bytes.fromhex(row['password_salt']))
        if not _ex_hmac.compare_digest(actual,row['password_hash']):
            attempts=int(row['failed_attempts'])+1
            until=(now+_ex_timedelta(minutes=15)).isoformat() if attempts>=5 else None
            c.execute('UPDATE app_users SET failed_attempts=?,locked_until=? WHERE username=?',
                      (attempts,until,name))
            raise_permission=True
        else:
            c.execute('''UPDATE app_users SET failed_attempts=0,locked_until=NULL,
                      last_login=? WHERE username=?''',(now.isoformat(),name))
            raise_permission=False
            session={'username':row['username'],'display_name':row['display_name'],
                     'role':row['account_role']}
    if raise_permission:
        raise PermissionError('Invalid username or password.')
    token=_ex_secrets.token_urlsafe(32)
    _APP_SESSIONS[token]={'username':session['username'],'last_used':now}
    session['token']=token
    audit('APP_USER_SIGN_IN',f"{session['username']} signed in to Exceptions")
    return session


def recover_app_password(username, recovery_code, new_password):
    '''Single-use, offline recovery. A successful reset rotates the recovery code.'''
    new_password=_validate_password(new_password)
    code=_normalise_recovery(recovery_code)
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        row=c.execute('SELECT * FROM app_users WHERE username=?',(str(username).strip(),)).fetchone()
        if not row or not row['enabled'] or not row['recovery_hash'] or row['recovery_used']:
            raise PermissionError('Recovery details are invalid. Contact an administrator.')
        digest=_pbkdf2(code,bytes.fromhex(row['recovery_salt']))
        if not _ex_hmac.compare_digest(digest,row['recovery_hash']):
            raise PermissionError('Recovery details are invalid. Contact an administrator.')
        ps,ph=_new_credential(new_password)
        next_code,rs,rh=_new_recovery()
        c.execute('''UPDATE app_users SET password_salt=?,password_hash=?,
            recovery_salt=?,recovery_hash=?,recovery_used=0,
            failed_attempts=0,locked_until=NULL,updated_at=? WHERE username=?''',
            (ps,ph,rs,rh,stamp(),row['username']))
    _revoke_user_sessions(row['username'])
    audit('APP_PASSWORD_RECOVERED',f"Offline recovery used for {row['username']}")
    return next_code


def rotate_user_recovery_code(actor, username):
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        _require_admin(c,actor)
        row=c.execute('SELECT username FROM app_users WHERE username=?',(username,)).fetchone()
        if not row:raise ValueError('Unknown user account.')
        code,rs,rh=_new_recovery()
        c.execute('''UPDATE app_users SET recovery_salt=?,recovery_hash=?,
                    recovery_used=0,updated_at=? WHERE username=?''',
                    (rs,rh,stamp(),row['username']))
    audit('APP_RECOVERY_ROTATED',f'{app_session_info(actor)["username"]} regenerated recovery for {username}')
    return code


def set_app_user_enabled(actor, username, enabled):
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        _require_admin(c,actor)
        row=c.execute('SELECT username,account_role,enabled FROM app_users WHERE username=?',(username,)).fetchone()
        if not row:raise ValueError('Unknown account.')
        if row['username'].casefold()==str(actor).casefold() and not enabled:
            raise ValueError('You cannot disable your own administrator account.')
        if not enabled and row['account_role']=='admin':
            admins=c.execute("SELECT COUNT(*) FROM app_users WHERE enabled=1 AND account_role='admin'").fetchone()[0]
            if admins<=1:raise ValueError('At least one enabled administrator must remain.')
        c.execute('UPDATE app_users SET enabled=?,updated_at=? WHERE username=?',
            (1 if enabled else 0,stamp(),row['username']))
    if not enabled:_revoke_user_sessions(username)
    audit('APP_USER_STATUS',f"{app_session_info(actor)['username']} {'enabled' if enabled else 'disabled'} {username}")


def link_final_attendance_as_user(session_token, group, service_date, report_id, expected_revision):
    principal=app_session_info(session_token)
    return link_final_attendance(principal['username'],group,service_date,
                                 'Shift 1',expected_revision,report_id)


def bulk_save_change_counts(actor, service_date, updates):
    '''Atomic multi-group changes. updates: [{group,revision,before?,after?}, ...].

    Unspecified stages remain NULL/unmodified; explicit zero means submitted zero.
    Any stale revision or invalid count aborts the *entire* batch.
    '''
    actor=app_session_info(actor)['username']
    day=parse_date(service_date).isoformat()
    active={x['name'].casefold() for x in group_registry() if x['enabled']}
    if not isinstance(updates,list) or len(updates)>1000:
        raise ValueError('Submit a list of up to 1000 group updates.')
    checked=[];seen=set()
    for item in updates:
        group=str(item.get('group','')).strip()
        if group.casefold() in seen:raise ValueError('Duplicate group in batch: '+group)
        seen.add(group.casefold())
        if group.casefold() not in active:raise ValueError('Inactive or unknown group: '+group)
        revision=item.get('revision')
        if type(revision) is not int or revision<0:raise ValueError('Invalid revision for '+group)
        fields={}
        for stage in ('before','after'):
            if stage in item:
                fields[stage]=_ex_validate_nonneg(item[stage],stage+' 15:00')
        if fields:checked.append((group,revision,fields))
    if not checked:return 0
    now=stamp()
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        changes=[]
        for group,expected,fields in checked:
            previous_row=c.execute('''SELECT * FROM exception_entries
                WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label='Shift 1' ''',
                (group,day)).fetchone()
            previous=dict(previous_row) if previous_row else {}
            current_revision=previous.get('revision',0)
            if expected!=current_revision:
                raise RuntimeError('Another user has changed '+group+' since you loaded it. Reload before saving.')
            next_fields={}
            for stage,value in fields.items():
                col=stage+'_count'
                if previous.get(col)!=value:
                    next_fields[col]=value
                    next_fields[stage+'_by']=actor
                    next_fields[stage+'_at']=now
            if not next_fields:continue
            if not previous_row:
                c.execute('''INSERT INTO exception_entries(group_name,service_date,shift_label,updated_at)
                     VALUES(?,?,'Shift 1',?)''',(group,day,now))
            assignments=','.join(k+'=?' for k in next_fields)
            values=list(next_fields.values())+[now,group,day,current_revision]
            updated=c.execute('''UPDATE exception_entries SET '''+assignments+''',
                basis='already_actual',updated_at=?,revision=revision+1
                WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label='Shift 1'
                AND revision=?''', values)
            if updated.rowcount != 1:
                raise RuntimeError('Concurrent edit detected for '+group)
            new=dict(c.execute('''SELECT * FROM exception_entries WHERE
                group_name=? COLLATE NOCASE AND service_date=? AND shift_label='Shift 1' ''',
                (group,day)).fetchone())
            c.execute('''INSERT INTO exception_entry_audit
              (changed_at,group_name,service_date,shift_label,actor_role,actor_name,old_state,new_state)
              VALUES(?,?,?,?,?,?,?,?)''',
              (now,group,day,'Shift 1','batch_changes',actor,
               _ex_json.dumps(previous,sort_keys=True),_ex_json.dumps(new,sort_keys=True)))
            changes.append(group)
    audit('CHANGES_BATCH_SAVED',f'{actor} submitted {len(changes)} group(s) for {day}')
    return len(changes)


def _ex_xlsx_rows(workbook_path):
    """Read visible worksheet cell values without loading/executing VBA macros."""
    from posixpath import normpath as _normpath
    path = _ex_Path(workbook_path)
    if path.suffix.lower() not in ('.xlsm', '.xlsx'):
        raise ValueError('Choose a .xlsm or .xlsx staffing tracker.')
    with _ex_zipfile.ZipFile(path) as z:
        all_names = set(z.namelist())
        shared = []
        if 'xl/sharedStrings.xml' in all_names:
            root = _ex_et.fromstring(z.read('xl/sharedStrings.xml'))
            shared = [''.join(n.text or '' for n in item.iter()
                              if n.tag.endswith('}t'))
                      for item in root.findall('m:si', EX_WORKBOOK_NS)]
        workbook = _ex_et.fromstring(z.read('xl/workbook.xml'))
        relroot = _ex_et.fromstring(z.read('xl/_rels/workbook.xml.rels'))
        relmap = {n.attrib['Id']: n.attrib['Target'] for n in relroot}
        for sh in workbook.findall('m:sheets/m:sheet', EX_WORKBOOK_NS):
            if sh.attrib.get('state', 'visible') != 'visible':
                continue
            name = sh.attrib['name']
            target = relmap[sh.attrib['{' + EX_WORKBOOK_NS['r'] + '}id']]
            xml_path = _normpath(target.lstrip('/') if target.startswith('/xl/') else
                                 ('xl/' + target) if not target.startswith('xl/') else target)
            if xml_path not in all_names:
                continue
            root = _ex_et.fromstring(z.read(xml_path))
            rows = {}
            for item in root.findall('.//m:sheetData/m:row/m:c', EX_WORKBOOK_NS):
                match = _ex_re.match(r'^([A-Z]+)(\d+)$', item.attrib.get('r', ''))
                if not match:
                    continue
                col, r = match.group(1), int(match.group(2))
                v = item.find('m:v', EX_WORKBOOK_NS)
                inline = item.find('m:is', EX_WORKBOOK_NS)
                value = v.text if v is not None else None
                if item.attrib.get('t') == 's' and value is not None:
                    value = shared[int(value)]
                elif item.attrib.get('t') == 'inlineStr' and inline is not None:
                    value = ''.join(inline.itertext())
                if value is not None:
                    rows.setdefault(r, {})[col] = value
            yield name, rows


def _ex_col_num(col):
    n = 0
    for c in col:
        n = n * 26 + (ord(c) - 64)
    return n


def _ex_as_count(val):
    try:
        numeric = float(val)
        return int(numeric) if numeric.is_integer() and 0 <= numeric <= 100000 else None
    except (TypeError, ValueError, OverflowError):
        return None


def _ex_date_from_serial(val):
    try:
        v = float(val)
        if 35000 <= v < 90000:
            return (_ex_datetime(1899, 12, 30) + _ex_timedelta(days=v)).date().isoformat()
    except (TypeError, ValueError, OverflowError):
        pass
    return None


def _ex_normalise_component(title):
    label = ' '.join(str(title or '').strip().upper().split())
    if 'T002' in label and 'T003' in label:
        return 'T002/T003'
    if label == 'PWT':
        return 'PWT'
    if label in ("SPL'S", 'SPL', 'SPLS'):
        return 'SPL'
    if label == 'PL':
        return 'PL'
    return label


def inspect_visible_staffing_plans(workbook_path):
    """Read planned staffing ONLY from numeric C cells of each visible PLANNED block.

    Each worksheet block has the gang title in column A, a 'PLANNED' header,
    C='SHIFT', and labelled staffing component rows in B/C. Its daily headers
    establish the dates for which the baseline is applicable. Blank C means
    unknown, NEVER zero; hidden worksheets are always ignored.
    """
    result, warnings = [], []
    for sheet, rows in _ex_xlsx_rows(workbook_path):
        for r in sorted(rows):
            header = rows[r]
            if str(header.get('A') or '').strip().upper() != 'PLANNED' or str(header.get('C') or '').strip().upper() != 'SHIFT':
                continue
            group = str(rows.get(r - 2, {}).get('A') or '').strip()
            if not group:
                warnings.append(f'{sheet}!A{r-2}: missing gang name')
                continue
            dates = []
            for col, val in header.items():
                if col not in ('A', 'B', 'C') and 4 <= _ex_col_num(col) <= 37:
                    day = _ex_date_from_serial(val)
                    if day:
                        dates.append(day)
            if not dates:
                warnings.append(f'{sheet}!D{r}: no dated shift columns')
                continue
            parts = {}
            source_cells = []
            for rr in range(r + 1, r + 12):
                line = rows.get(rr, {})
                raw_label = str(line.get('B') or '').strip()
                if not raw_label or any(key in raw_label.casefold() for key in
                                        ('change', 'no show', 'shortage', 'late', 'near miss')):
                    continue
                val = _ex_as_count(line.get('C'))
                if val is None:
                    continue
                name = _ex_normalise_component(raw_label)
                parts[name] = parts.get(name, 0) + val
                source_cells.append(f'C{rr}')
            if not parts:
                # A blank planned team is not a zero-staffing plan.
                continue
            result.append(dict(group_name=group, worksheet=sheet,
                               effective_from=min(dates), effective_to=max(dates),
                               components=parts, planned_total=sum(parts.values()),
                               source_cell=', '.join(source_cells)))
    return result, warnings


def import_visible_staffing_plans(workbook_path):
    plans, warnings = inspect_visible_staffing_plans(workbook_path)
    if not plans:
        raise ValueError('No visible worksheets with dated planned staffing in column C were found.')
    now = stamp()
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        for p in plans:
            c.execute('''INSERT INTO staffing_plans
                (group_name,worksheet,effective_from,effective_to,components_json,
                 planned_total,workbook_name,source_cell,imported_at)
                 VALUES(?,?,?,?,?,?,?,?,?)
                 ON CONFLICT(group_name,worksheet,effective_from,effective_to)
                 DO UPDATE SET components_json=excluded.components_json,
                   planned_total=excluded.planned_total,
                   workbook_name=excluded.workbook_name,
                   source_cell=excluded.source_cell,
                   imported_at=excluded.imported_at''',
                  (p['group_name'], p['worksheet'], p['effective_from'],
                   p['effective_to'], _ex_json.dumps(p['components'], sort_keys=True),
                   p['planned_total'], _ex_Path(workbook_path).name, p['source_cell'], now))
    audit('VISIBLE_PLANS_IMPORTED',
          f'{_ex_Path(workbook_path).name}: {len(plans)} nonblank plans, {len(warnings)} notices')
    return plans, warnings



def manual_plan_versions(group):
    """Versioned, locally configured requirements for one WhatsApp group."""
    with connect() as c:
        rows = c.execute('''SELECT * FROM manual_staffing_plans
            WHERE group_name=? COLLATE NOCASE ORDER BY effective_from DESC''', (group,)).fetchall()
    return [{**dict(row), 'components': _ex_json.loads(row['components_json'])} for row in rows]


def save_manual_staffing_plan(group, effective_from, components, actor='Settings'):
    """Save a date-effective, auditable plan; zero means a role is not required.

    A newer start date supersedes the preceding plan but does not rewrite
    historical staffing requirements. Workbook plans remain a dated fallback.
    """
    group = validated_group_title(group)
    registered = {r['name'].casefold() for r in group_registry()}
    if group.casefold() not in registered:
        raise ValueError('Save this WhatsApp group in Settings before configuring its staffing plan.')
    day = parse_date(effective_from).isoformat()
    if set(components) != set(EX_METRIC_FIELDS):
        raise ValueError('Provide exactly T002/T003, PWT, SPL and PL requirements.')
    clean = {}
    for role in EX_METRIC_FIELDS:
        raw = components[role]
        if isinstance(raw, bool) or (not isinstance(raw, int) and
            (not isinstance(raw, str) or not raw.strip().isascii() or not raw.strip().isdecimal())):
            raise ValueError(f'{role}: enter a whole number between 0 and 10000.')
        amount = int(raw)
        if amount < 0 or amount > 10000:
            raise ValueError(f'{role}: enter a whole number between 0 and 10000.')
        clean[role] = amount
    total = sum(clean.values())
    if total == 0:
        raise ValueError('The required total must be above zero. Use 0 for individual roles that are not required.')
    now = stamp()
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        old = c.execute('''SELECT * FROM manual_staffing_plans
            WHERE group_name=? COLLATE NOCASE AND effective_from=?''', (group,day)).fetchone()
        old_json = _ex_json.dumps(dict(old), sort_keys=True) if old else None
        new_json = _ex_json.dumps(clean, sort_keys=True)
        if old:
            c.execute('''UPDATE manual_staffing_plans
              SET components_json=?, planned_total=?, updated_at=?, updated_by=?
              WHERE group_name=? COLLATE NOCASE AND effective_from=?''',
              (new_json,total,now,str(actor),group,day))
        else:
            c.execute('''INSERT INTO manual_staffing_plans
              (group_name,effective_from,components_json,planned_total,updated_at,updated_by)
              VALUES(?,?,?,?,?,?)''',(group,day,new_json,total,now,str(actor)))
        c.execute('''INSERT INTO manual_staffing_plan_audit
          (changed_at,actor,operation,group_name,effective_from,old_state,new_state)
          VALUES(?,?,?,?,?,?,?)''',
          (now,str(actor),'UPDATED' if old else 'CREATED',group,day,old_json,new_json))
        c.execute("UPDATE exception_sync_state SET revision=revision+1 WHERE id='plans'")
    audit('MANUAL_PLAN_SAVED', f'{group} {day}: {total} planned (by {actor})')
    return {'group_name':group,'effective_from':day,'components':clean,'planned_total':total}


def delete_manual_staffing_plan(group, effective_from, actor='Settings'):
    """Delete only the exact manual version selected, never imported Excel plans."""
    day = parse_date(effective_from).isoformat()
    now = stamp()
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        old = c.execute('''SELECT * FROM manual_staffing_plans
            WHERE group_name=? COLLATE NOCASE AND effective_from=?''',(group,day)).fetchone()
        if old is None:
            raise ValueError('No manually saved plan starts on the selected date.')
        c.execute('''DELETE FROM manual_staffing_plans
            WHERE group_name=? COLLATE NOCASE AND effective_from=?''',(group,day))
        c.execute('''INSERT INTO manual_staffing_plan_audit
          (changed_at,actor,operation,group_name,effective_from,old_state,new_state)
          VALUES(?,?,?,?,?,?,?)''',
          (now,str(actor),'DELETED',group,day,_ex_json.dumps(dict(old),sort_keys=True),None))
        c.execute("UPDATE exception_sync_state SET revision=revision+1 WHERE id='plans'")
    audit('MANUAL_PLAN_DELETED', f'{group} {day} by {actor}')


def exception_plan(group, service_date):
    """Manual dated plans take priority; fall back to visible Excel worksheets."""
    day = parse_date(service_date).isoformat()
    with connect() as c:
        manual = c.execute('''SELECT * FROM manual_staffing_plans
            WHERE group_name=? COLLATE NOCASE AND effective_from<=?
            ORDER BY effective_from DESC LIMIT 1''',(group,day)).fetchone()
        if manual is not None:
            row = dict(manual)
            row['components'] = _ex_json.loads(row['components_json'])
            row['worksheet'] = 'Settings / manual plan'
            row['effective_to'] = 'Until superseded'
            row['source'] = 'Manual staffing planner'
            return row, ''
        matches = [dict(r) for r in c.execute('''
            SELECT * FROM staffing_plans WHERE group_name=? COLLATE NOCASE
              AND effective_from<=? AND effective_to>=?''',
            (group, day, day))]
    if not matches:
        return None, 'No staffing plan for this group/date. Use Settings → Manage Staffing Plans or import a current tracker.'
    # Never guess between two worksheets or overlapping Excel sources.
    if len(matches) != 1:
        return None, 'Multiple overlapping Excel planning sources. Set a manual plan in Settings to resolve this date.'
    row = matches[0]
    row['components'] = _ex_json.loads(row['components_json'])
    row['source'] = 'Visible Excel worksheet'
    unsupported = [v for v in row['components'] if v not in EX_METRIC_FIELDS]
    if unsupported:
        return None, f"Plan includes roles not captured by WhatsApp ({', '.join(unsupported)}). Set a manual plan with supported roles."
    return row, ''


def exception_reports(group, service_date):
    # Include next-morning reports for explicit night-shift selection only.
    day = parse_date(service_date)
    next_day = day + _ex_timedelta(days=1)
    with connect() as c:
        return [dict(r) for r in c.execute('''
           SELECT id,event_at,group_name,t002_t003,pwt,spl,pl,no_shows
           FROM reports WHERE group_name=? COLLATE NOCASE
           AND event_at>=? AND event_at<?
           AND (t002_t003 IS NOT NULL OR pwt IS NOT NULL
                OR spl IS NOT NULL OR pl IS NOT NULL)
           ORDER BY event_at DESC,id DESC''',
           (group, day.isoformat()+'T00:00:00', next_day.isoformat()+'T08:00:00'))]


def whatsapp_no_shows_for_snapshot(group, service_date, snapshot):
    # Use explicit count only; ambiguous multiple WhatsApp updates stay unknown.
    if snapshot.get('no_shows') is not None:
        return snapshot['no_shows']
    report_day = snapshot['event_at'][:10]
    service_day = parse_date(service_date).isoformat()
    with connect() as c:
        others = [r[0] for r in c.execute('''SELECT no_shows FROM reports
          WHERE group_name=? COLLATE NOCASE
            AND substr(event_at,1,10) IN (?,?)
            AND no_shows IS NOT NULL''', (group, report_day, service_day))]
    return others[0] if len(others)==1 else None


def exception_entry(group, service_date, shift_label='Shift 1'):
    with connect() as c:
        found = c.execute('''SELECT * FROM exception_entries
          WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=?''',
          (group, service_date, shift_label)).fetchone()
    return dict(found) if found else None


def _ex_validate_nonneg(n, field):
    if isinstance(n, bool) or not str(n).strip().isdigit() or len(str(n).strip()) > 6:
        raise ValueError(f'{field} must be a non-negative whole number.')
    value = int(n)
    if value > 100000:
        raise ValueError(f'{field} cannot exceed 100000.')
    return value


def _ex_validate_signed(n, field):
    val = str(n).strip()
    if not _ex_re.fullmatch(r'[+-]?\d{1,6}', val):
        raise ValueError(f'{field} must be a signed whole number, e.g. -2, 0, +1.')
    value = int(val)
    if abs(value) > 100000:
        raise ValueError(f'{field} magnitude cannot exceed 100000.')
    return value


def save_change_counts(actor, group, service_date, shift_label,
                       expected_revision, before_count, after_count, report_id=None):
    # Single operator edits both change counts; no-show history is untouched.
    if not str(actor).strip() or len(str(actor)) > 100:
        raise PermissionError('Sign in to record changes.')
    group = str(group).strip()
    registered = {x['name'].casefold() for x in group_registry() if x['enabled']}
    if group.casefold() not in registered:
        raise ValueError('Group is not active. Enable it under Settings → Group Registry.')
    day = parse_date(service_date).isoformat()
    if shift_label not in EX_SHIFTS:
        raise ValueError('Invalid shift.')
    if not isinstance(expected_revision,int) or expected_revision < 0:
        raise ValueError('Invalid entry revision.')
    before = _ex_validate_nonneg(before_count, 'Changes before 15:00')
    after = _ex_validate_nonneg(after_count, 'Changes after 15:00')
    if report_id not in (None,''):
        try: report_id = int(report_id)
        except (ValueError,TypeError): raise ValueError('Invalid WhatsApp report.')
        if report_id not in {x['id'] for x in exception_reports(group, day)}:
            raise ValueError('Selected WhatsApp report is not associated with this group/service window.')
    else:
        report_id = None
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        if report_id is not None:
            duplicate = c.execute('''SELECT shift_label,service_date FROM exception_entries
                WHERE report_id=? AND NOT (group_name=? COLLATE NOCASE
                AND service_date=? AND shift_label=?)''',
                (report_id,group,day,shift_label)).fetchone()
            if duplicate:
                raise ValueError('This WhatsApp report is already linked to another shift: '
                                 + duplicate['service_date'] + ' / ' + duplicate['shift_label'])
        old = c.execute('''SELECT * FROM exception_entries WHERE group_name=? COLLATE NOCASE
             AND service_date=? AND shift_label=?''', (group,day,shift_label)).fetchone()
        old_data = dict(old) if old else {}
        revision = int(old_data.get('revision', 0))
        if expected_revision != revision:
            raise RuntimeError('This entry has changed since you loaded it. Reload before submitting.')
        now = stamp()
        if old is None:
            c.execute('''INSERT INTO exception_entries(group_name,service_date,shift_label,updated_at)
                         VALUES(?,?,?,?)''',(group,day,shift_label,now))
        changed = c.execute('''UPDATE exception_entries SET before_count=?, after_count=?,
             before_impact=NULL, after_impact=NULL, basis='already_actual',report_id=?,
             before_by=?, before_at=?, after_by=?, after_at=?,
             updated_at=?, revision=revision+1
             WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=? AND revision=?''',
             (before,after,report_id,actor,now,actor,now,now,group,day,shift_label,revision))
        if changed.rowcount != 1:
            raise RuntimeError('Concurrent update detected. Reload before submitting.')
        new = dict(c.execute('''SELECT * FROM exception_entries WHERE group_name=? COLLATE NOCASE
          AND service_date=? AND shift_label=?''',(group,day,shift_label)).fetchone())
        c.execute('''INSERT INTO exception_entry_audit
         (changed_at,group_name,service_date,shift_label,actor_role,actor_name,old_state,new_state)
         VALUES(?,?,?,?,?,?,?,?)''',
         (now,group,day,shift_label,'changes',str(actor).strip(),
          _ex_json.dumps(old_data,sort_keys=True),_ex_json.dumps(new,sort_keys=True)))
    audit('CHANGE_COUNTS_EDIT',f'{group} {day} {shift_label} before={before},after={after}; {actor}')
    return new


def save_change_stage(actor, group, service_date, shift_label,
                      expected_revision, stage, count):
    """Save one deadline's change count without requiring a WhatsApp report.

    The other stage and any linked actual-attendance report are left intact.
    A submitted zero means a confirmed zero; an unsubmitted NULL is unknown.
    """
    if not str(actor).strip() or len(str(actor)) > 100:
        raise PermissionError('Sign in to record changes.')
    group = str(group).strip()
    active = {x['name'].casefold() for x in group_registry() if x['enabled']}
    if group.casefold() not in active:
        raise ValueError('Group is not active. Enable it under Settings.')
    day = parse_date(service_date).isoformat()
    if shift_label not in EX_SHIFTS:
        raise ValueError('Invalid shift.')
    if type(expected_revision) is not int or expected_revision < 0:
        raise ValueError('Invalid entry revision.')
    if stage not in ('before','after'):
        raise ValueError('Choose before or after 15:00.')
    value = _ex_validate_nonneg(count, 'Changes '+stage+' 15:00')
    # These are fixed, reviewed SQL column names, never passed by the user.
    field, by_field, at_field = {
        'before': ('before_count','before_by','before_at'),
        'after': ('after_count','after_by','after_at')
    }[stage]
    now = stamp()
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        old = c.execute("""SELECT * FROM exception_entries
            WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=?""",
            (group,day,shift_label)).fetchone()
        previous = dict(old) if old else {}
        revision = int(previous.get('revision',0))
        if revision != expected_revision:
            raise RuntimeError('Another edit was saved. Reload this shift before continuing.')
        if not old:
            c.execute("""INSERT INTO exception_entries(group_name,service_date,shift_label,updated_at)
                VALUES(?,?,?,?)""",(group,day,shift_label,now))
        cur = c.execute(f"""UPDATE exception_entries SET {field}=?, {by_field}=?, {at_field}=?,
            basis='already_actual',updated_at=?,revision=revision+1
            WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=?
              AND revision=?""",
            (value,str(actor).strip(),now,now,group,day,shift_label,revision))
        if cur.rowcount != 1:
            raise RuntimeError('Concurrent edit detected. Reload this shift.')
        updated = dict(c.execute("""SELECT * FROM exception_entries WHERE
            group_name=? COLLATE NOCASE AND service_date=? AND shift_label=?""",
            (group,day,shift_label)).fetchone())
        c.execute("""INSERT INTO exception_entry_audit
            (changed_at,group_name,service_date,shift_label,actor_role,actor_name,old_state,new_state)
            VALUES(?,?,?,?,?,?,?,?)""",
            (now,group,day,shift_label,'changes_'+stage,str(actor).strip(),
             _ex_json.dumps(previous,sort_keys=True),_ex_json.dumps(updated,sort_keys=True)))
    audit('CHANGE_STAGE_SAVED',f'{group} {day} {shift_label}: {stage}={value}, actor={actor}')
    return updated


def link_final_attendance(actor,group,service_date,shift_label,
                          expected_revision,report_id):
    """Explicitly link an ambiguous/after-midnight WhatsApp snapshot after it arrives."""
    if not str(actor).strip():
        raise PermissionError('Sign in to link attendance.')
    day=parse_date(service_date).isoformat()
    group=str(group).strip()
    if group.casefold() not in {x['name'].casefold() for x in group_registry() if x['enabled']}:
        raise ValueError('Group is not active.')
    if shift_label not in EX_SHIFTS:
        raise ValueError('Invalid shift.')
    if type(expected_revision) is not int or expected_revision < 0:
        raise ValueError('Invalid revision.')
    try: report_id=int(report_id)
    except (TypeError,ValueError): raise ValueError('Select an actual WhatsApp attendance report.')
    if report_id not in {r['id'] for r in exception_reports(group,day)}:
        raise ValueError('Report does not match this group/night-shift service window.')
    now=stamp()
    with connect() as c:
        c.execute('BEGIN IMMEDIATE')
        old=c.execute("""SELECT * FROM exception_entries
            WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=?""",
            (group,day,shift_label)).fetchone()
        previous=dict(old) if old else {}
        if expected_revision != int(previous.get('revision',0)):
            raise RuntimeError('This record has changed. Reload before linking a report.')
        collision=c.execute("""SELECT service_date,shift_label FROM exception_entries
            WHERE report_id=? AND NOT (group_name=? COLLATE NOCASE
            AND service_date=? AND shift_label=?) LIMIT 1""",
            (report_id,group,day,shift_label)).fetchone()
        if collision:
            raise ValueError('This WhatsApp report is already linked to '
                             +collision['service_date']+' '+collision['shift_label'])
        if not old:
            c.execute("""INSERT INTO exception_entries(group_name,service_date,shift_label,updated_at)
                VALUES(?,?,?,?)""",(group,day,shift_label,now))
        cur=c.execute("""UPDATE exception_entries
            SET report_id=?,basis='already_actual',updated_at=?,revision=revision+1
            WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=? AND revision=?""",
            (report_id,now,group,day,shift_label,expected_revision))
        if cur.rowcount!=1:
            raise RuntimeError('Concurrent attendance edit detected. Reload.')
        updated=dict(c.execute("""SELECT * FROM exception_entries
            WHERE group_name=? COLLATE NOCASE AND service_date=? AND shift_label=?""",
            (group,day,shift_label)).fetchone())
        c.execute("""INSERT INTO exception_entry_audit
            (changed_at,group_name,service_date,shift_label,actor_role,actor_name,old_state,new_state)
            VALUES(?,?,?,?,?,?,?,?)""",
            (now,group,day,shift_label,'attendance_link',str(actor).strip(),
             _ex_json.dumps(previous,sort_keys=True),_ex_json.dumps(updated,sort_keys=True)))
    audit('FINAL_ATTENDANCE_LINKED',f'{group} {day} {shift_label} linked report {report_id}')
    return updated


def calculate_exception(group, service_date, shift_label='Shift 1'):
    """Calculate shortage by *role* from the actual WhatsApp snapshot.

    A recorded change is an event, not automatically one employee missing.
    The WhatsApp message is actual attendance, so never re-subtract operator
    changes or no-shows (which would double-count absent staff).

    Role shortages are summed; surplus in a different skill category does not
    hide a safety-critical vacancy. Operator change logs are independent
    of availability; an identifiable WhatsApp report is sufficient to show
    a calculated shortage without waiting for manual change counts.
    """
    plan, plan_error = exception_plan(group, service_date)
    record = exception_entry(group, service_date, shift_label)
    snapshots = exception_reports(group, service_date)
    result = {'group': group, 'service_date': service_date, 'shift': shift_label,
              'plan_total': plan['planned_total'] if plan else None,
              'plan_components': plan['components'] if plan else {},
              'before_count': record['before_count'] if record else None,
              'after_count': record['after_count'] if record else None,
              'no_shows': None,  # WhatsApp only; historical manual values are ignored.
              'staffing_reported': None, 'available': None, 'shortage': None,
              'role_breakdown': [], 'group_total_gap': None,
              'basis': 'already_actual',
              'report_id': record['report_id'] if record else None,
              'status': 'Pending', 'reason': '',
              'revision': record['revision'] if record else 0}
    if plan_error:
        result['reason'] = plan_error
        return result
    if record and record['report_id'] is not None:
        matching = [r for r in snapshots if r['id'] == record['report_id']]
        if not matching:
            result['reason'] = 'Selected WhatsApp report is missing or belongs to another date.'
            return result
        snapshot = matching[0]
    elif len(snapshots)==1 and snapshots[0]['event_at'][:10] == service_date:
        # Automatic matching is safe only with exactly one calendar-day report
        # and no post-midnight competing report or other shift assignment.
        with connect() as c:
            other_shift = c.execute('''SELECT 1 FROM exception_entries
                WHERE group_name=? COLLATE NOCASE AND service_date=?
                AND shift_label<>? LIMIT 1''', (group,service_date,shift_label)).fetchone()
            assigned_elsewhere = c.execute('''SELECT 1 FROM exception_entries
                WHERE report_id=? AND NOT (group_name=? COLLATE NOCASE
                AND service_date=? AND shift_label=?) LIMIT 1''',
                (snapshots[0]['id'],group,service_date,shift_label)).fetchone()
        if other_shift or assigned_elsewhere:
            result['reason'] = 'Report already assigned to another shift. Select the correct final report.'
            return result
        snapshot = snapshots[0]
    elif not snapshots:
        result['reason'] = 'No WhatsApp attendance report for this group and date.'
        return result
    else:
        result['reason'] = 'Select the final attendance message for this night shift (the message may be after midnight).'
        return result

    needed = [(role, EX_METRIC_FIELDS[role]) for role, required in plan['components'].items() if required > 0]
    missing = [role for role, field in needed if snapshot[field] is None]
    if missing:
        result['reason'] = 'WhatsApp attendance is missing required role(s): ' + ', '.join(missing)
        return result
    role_breakdown = []
    for role, field in EX_METRIC_FIELDS.items():
        planned = int(plan['components'].get(role, 0))
        raw_actual = snapshot[field]
        # A role not required by the plan can be blank. It cannot substitute
        # for a missing required role; show it as 0 only for total display.
        present = int(raw_actual) if raw_actual is not None else 0
        if present < 0:
            result['reason'] = f'Invalid negative WhatsApp headcount for {role}.'
            return result
        role_breakdown.append({'role': role, 'required': planned, 'present': present,
                               'shortage': max(0, planned - present),
                               'surplus': max(0, present - planned)})
    available = sum(r['present'] for r in role_breakdown)
    shortage = sum(r['shortage'] for r in role_breakdown)
    complete = bool(record and record['before_count'] is not None
                    and record['after_count'] is not None)
    reported_no_shows = whatsapp_no_shows_for_snapshot(group, service_date, snapshot)
    result.update(staffing_reported=available, available=available,
                  shortage=shortage, role_breakdown=role_breakdown,
                  no_shows=reported_no_shows,
                  group_total_gap=max(0, int(plan['planned_total']) - available),
                  status='Complete' if complete else 'Attendance',
                  reason=('Attendance captured; both change counts submitted.' if complete else
                          'Attendance captured; change counts have not yet both been entered.'),
                  report_id=snapshot['id'])
    return result


def exception_dashboard(start=None, end=None, group='All Groups'):
    """Include unreviewed WhatsApp days so missing manual submissions are visible."""
    start_iso = parse_date(start).isoformat() if start else None
    end_iso = parse_date(end).isoformat() if end else None
    conditions, params = [], []
    if start_iso:
        conditions.append('event_at>=?')
        params.append(start_iso + 'T00:00:00')
    if end_iso:
        conditions.append('event_at<?')
        params.append((_ex_datetime.fromisoformat(end_iso) + _ex_timedelta(days=1)).date().isoformat()+'T00:00:00')
    if group != 'All Groups':
        conditions.append('group_name=? COLLATE NOCASE')
        params.append(group)
    report_where = ' WHERE ' + ' AND '.join(conditions) if conditions else ''
    conditions2, params2 = [], []
    if start_iso:
        conditions2.append('service_date>=?'); params2.append(start_iso)
    if end_iso:
        conditions2.append('service_date<=?'); params2.append(end_iso)
    if group != 'All Groups':
        conditions2.append('group_name=? COLLATE NOCASE'); params2.append(group)
    manual_where = ' WHERE ' + ' AND '.join(conditions2) if conditions2 else ''
    with connect() as c:
        # One after-midnight report linked to the previous shift must not be
        # double-counted as an independent following-day staffing snapshot.
        report_condition = ' AND ' if report_where else ' WHERE '
        reports_sql = ('''SELECT DISTINCT r.group_name,substr(r.event_at,1,10)
           FROM reports r''' + report_where + report_condition +
           ''' (r.t002_t003 IS NOT NULL OR r.pwt IS NOT NULL
                OR r.spl IS NOT NULL OR r.pl IS NOT NULL)
             AND NOT EXISTS (SELECT 1 FROM exception_entries e
               WHERE e.report_id=r.id AND e.service_date<>substr(r.event_at,1,10))''')
        reports = [tuple(x) for x in c.execute(reports_sql, params)]
        manual = [tuple(x) for x in c.execute('SELECT group_name,service_date,shift_label '
                    'FROM exception_entries' + manual_where, params2)]
    manual_by_day = {(g.casefold(), d) for g, d, shift in manual}
    keys = set(manual)
    for g, d in reports:
        if (g.casefold(), d) not in manual_by_day:
            keys.add((g, d, 'Shift 1'))
    # Retain historical data from renamed/paused groups; new active groups
    # appear in the editor immediately even before their first report.
    rows = [calculate_exception(g, d, shift) for g, d, shift in
            sorted(keys, key=lambda x:(x[1], x[0].casefold(), x[2]), reverse=True)]
    calculable = [r for r in rows if r['status'] in ('Complete','Attendance')]
    return dict(rows=rows,
                changes_before=sum(r['before_count'] for r in rows if r['before_count'] is not None),
                changes_after=sum(r['after_count'] for r in rows if r['after_count'] is not None),
                changes_before_known=sum(r['before_count'] is not None for r in rows),
                changes_after_known=sum(r['after_count'] is not None for r in rows),
                no_shows=sum(r['no_shows'] for r in calculable if r['no_shows'] is not None),
                no_shows_known=sum(r['no_shows'] is not None for r in calculable),
                shortage=sum(r['shortage'] for r in calculable),
                short_shifts=sum(r['shortage'] > 0 for r in calculable),
                verified=len(calculable),
                change_logs_complete=sum(r['status']=='Complete' for r in calculable),
                pending=len(rows)-len(calculable),
                awaiting_whatsapp=sum(r['reason'].startswith('No WhatsApp attendance report') for r in rows),
                awaiting_manual_changes=sum(r['status']=='Attendance' for r in rows))



def service_date_has_records(service_date):
    """True for a persisted exception entry or WhatsApp staffing report.

    Deliberately excludes the group registry: 40 registered groups are *not*
    40 records. A manual entry may precede the final WhatsApp report.
    """
    day = parse_date(service_date).isoformat()
    with connect() as c:
        if c.execute('SELECT 1 FROM exception_entries WHERE service_date=? LIMIT 1',
                     (day,)).fetchone():
            return True
        return bool(c.execute("""SELECT 1 FROM reports r
            WHERE substr(r.event_at,1,10)=?
              AND (r.t002_t003 IS NOT NULL OR r.pwt IS NOT NULL
                   OR r.spl IS NOT NULL OR r.pl IS NOT NULL)
              AND NOT EXISTS (
                SELECT 1 FROM exception_entries e
                WHERE e.report_id=r.id AND e.service_date<>?
              ) LIMIT 1""", (day, day)).fetchone())


def workforce_exception_chart_rows(rows, limit=10):
    """Four independently reported KPI categories by group for the chart.

    A missing no-show count is not converted into an asserted zero. Only
    present counts contribute to the bar; the chart labels the coverage caveat.
    Category totals are incident counts, NOT a unique-person headcount.
    """
    grouped={}
    for r in rows:
        name=str(r['group'])
        g=grouped.setdefault(name, {'group':name, 'before':0, 'after':0,
                                    'no_shows':0, 'shortage':0, 'no_shows_known':0})
        for field, source in [('before','before_count'),('after','after_count'),
                              ('no_shows','no_shows'),('shortage','shortage')]:
            value=r.get(source)
            if value is not None:
                g[field]+=max(0,int(value))
                if field=='no_shows':g['no_shows_known']+=1
    result=sorted(grouped.values(),
                  key=lambda g:(-(g['before']+g['after']+g['no_shows']+g['shortage']),
                                g['group'].casefold()))
    return result[:max(0,int(limit))]


def exception_data_signature():
    with connect() as c:
        a = c.execute('SELECT COUNT(*),COALESCE(SUM(revision),0) FROM exception_entries').fetchone()
        b = c.execute("SELECT revision FROM exception_sync_state WHERE id='plans'").fetchone()
    return tuple(a) + tuple(b)


init_exception_db()


## 2. Legacy adapter and collector diagnostics

Read-only helpers and monitoring state.

In [ ]:
# ---- Compatibility: analytics functions share one module in this build ----
import sys
core = sys.modules[__name__]

# ---- Optional browser monitoring ----
"""Optional live WhatsApp Web adapter. No official messaging API used.
Requires user-authorised browser login; WhatsApp UI selectors can change.
"""
import asyncio, logging, re, threading
from datetime import datetime, timedelta
from pathlib import Path

LOG=logging.getLogger('monitor')


def whatsapp_datetime(pretext):
    '''Parse the timestamp supplied by WhatsApp's message metadata.

    Refuse an unrecognised timestamp instead of silently misdating a report.
    '''
    match = re.search(r"\[([^\]]+)\]", pretext or "")
    if not match or ',' not in match.group(1):
        return None
    time_part, date_part = (v.strip() for v in match.group(1).split(',', 1))
    value = date_part.casefold().strip()
    if value == 'today':
        day = datetime.now().date()
    elif value == 'yesterday':
        day = datetime.now().date() - timedelta(days=1)
    else:
        # Compatibility entry point: delegate numeric dates to the *same*
        # parser used by the original WhatsApp engine. Never parse WhatsApp
        # metadata using the CSV (DD/MM/YYYY) reader.
        try:
            uk_date, normal_time = parse_whatsapp_timestamp(pretext)
            return core.normalise_datetime(uk_date, normal_time)
        except ValueError:
            return None
    try:
        return core.normalise_datetime(day, time_part)
    except ValueError:
        return None


def collector_diagnostics():
    with core.connect() as con:
        con.execute('''CREATE TABLE IF NOT EXISTS collection_diagnostics (
          id INTEGER PRIMARY KEY CHECK(id=1), updated_at TEXT,
          group_name TEXT, visible_messages INTEGER DEFAULT 0,
          metric_messages INTEGER DEFAULT 0, saved INTEGER DEFAULT 0,
          duplicate INTEGER DEFAULT 0, missing_timestamp INTEGER DEFAULT 0,
          last_note TEXT DEFAULT '')''')
        con.execute('INSERT OR IGNORE INTO collection_diagnostics(id) VALUES (1)')
        return dict(con.execute('SELECT * FROM collection_diagnostics WHERE id=1').fetchone())


def save_collector_diagnostics(group, counters, note=''):
    with core.connect() as con:
        con.execute('''CREATE TABLE IF NOT EXISTS collection_diagnostics (
          id INTEGER PRIMARY KEY CHECK(id=1), updated_at TEXT,
          group_name TEXT, visible_messages INTEGER DEFAULT 0,
          metric_messages INTEGER DEFAULT 0, saved INTEGER DEFAULT 0,
          duplicate INTEGER DEFAULT 0, missing_timestamp INTEGER DEFAULT 0,
          last_note TEXT DEFAULT '')''')
        con.execute('''INSERT INTO collection_diagnostics
          (id,updated_at,group_name,visible_messages,metric_messages,saved,duplicate,missing_timestamp,last_note)
          VALUES(1,?,?,?,?,?,?,?,?)
          ON CONFLICT(id) DO UPDATE SET updated_at=excluded.updated_at,
          group_name=excluded.group_name, visible_messages=excluded.visible_messages,
          metric_messages=excluded.metric_messages,saved=excluded.saved,
          duplicate=excluded.duplicate,missing_timestamp=excluded.missing_timestamp,
          last_note=excluded.last_note''',
          (core.stamp(), group, *(counters.get(k,0) for k in
           ('visible','metrics','saved','duplicate','missing_timestamp')), note[:300]))


## 3. Original WhatsApp parser and history catch-up

Incoming WhatsApp uses MDY; staging CSV and Excel are explicitly DMY.

In [ ]:
# ---- Original, user-provided WhatsApp processing engine ----
# Extracted from WhatsApp_Integration_FIXED (Reporting Module)(1).ipynb.
# Collection, parsing, and original CSV/Excel writes are preserved.
# Only the initial URL and proactive group sweep interval are adjusted.
import os
import asyncio
import tkinter as tk
from openpyxl import load_workbook, Workbook
from playwright.async_api import async_playwright

BASE_DIR = str(ROOT)
_legacy_config = ROOT / 'config.json'
try:
    _legacy_paths = json.loads(_legacy_config.read_text(encoding='utf-8'))
except (OSError, ValueError):
    _legacy_paths = {}
LOCAL_FILE_PATH = str(_legacy_paths.get('excel_path') or (DATA / 'WhatsAppData.xlsx'))
STAGING_CSV_PATH = str(_legacy_paths.get('csv_path') or (DATA / 'WhatsAppData_staging.csv'))
TARGET_GROUPS = list(configuration()['groups'])
recently_logged_fingerprints = set()
is_running = False
browser_context = None
playwright_instance = None
worker_loop = None

def load_saved_groups():
    global TARGET_GROUPS
    if os.path.exists(GROUPS_CONFIG_PATH):
        try:
            with open(GROUPS_CONFIG_PATH, "r", encoding="utf-8") as f:
                saved = [line.strip() for line in f if line.strip()]
                if saved:
                    TARGET_GROUPS = saved
                    logging.info(f"📂 Loaded {len(TARGET_GROUPS)} target groups from config file.")
                    return
        except Exception as e:
            logging.warning(f"⚠️ Could not load saved groups config: {e}")
    logging.info("📋 Config file not found or empty. Using default group lists.")

def save_groups_to_disk():
    try:
        with open(GROUPS_CONFIG_PATH, "w", encoding="utf-8") as f:
            for group in TARGET_GROUPS:
                f.write(f"{group}\n")
        logging.info("💾 Target group layout successfully saved to disk.")
    except Exception as e:
        logging.error(f"❌ Failed to auto-save groups configuration: {e}")

def load_existing_fingerprints():
    global recently_logged_fingerprints
    if os.path.exists(STAGING_CSV_PATH):
        try:
            with open(STAGING_CSV_PATH, mode='r', newline='', encoding='utf-8') as f:
                reader = csv.reader(f)
                header = next(reader, None)
                if not header:
                    return
                for row in reader:
                    if len(row) >= 5:
                        layout_token = f"{row[2]}_{row[3]}_{row[0]}_{row[1]}_{row[4]}"
                        recently_logged_fingerprints.add(layout_token)
            logging.info(f"💾 Synchronized {len(recently_logged_fingerprints)} historic records into deduplication stack.")
        except Exception as e:
            logging.warning(f"⚠️ Could not load history pipeline: {e}")

def parse_metrics_from_text(text):
    clean_text = " ".join(text.split())
    extracted_values = {}
    t_match = re.search(r"T002\s*[\/\\\-_]\s*T003[:\s\-]*(\d+)", clean_text, re.IGNORECASE)
    extracted_values["T002/T003"] = int(t_match.group(1)) if t_match else ""
    for key in ["PWT", "SPL", "PL", "Near Miss", "Late"]:
        match = re.search(rf"\b{key}[:\s\-]*(\d+)", clean_text, re.IGNORECASE)
        extracted_values[key] = int(match.group(1)) if match else ""
    
    # Capture even a standalone WhatsApp no-show update (the full message is
    # preserved in the existing CSV, and SQLite parses its explicit count).
    found_any = any(extracted_values[k] != "" for k in extracted_values)
    if not found_any and parse_explicit_no_shows(text) is not None:
        found_any = True
    return extracted_values, found_any

# WhatsApp Web's message metadata in this user's installation is US-style
# MONTH/DAY/YEAR. We verified this against stored records captured on
# 10 November 2026: WhatsApp's 11/10/2026 was incorrectly saved as 11 October.
# The staging CSV uses the independent, *strict* UK DAY/MONTH/YEAR convention.
# Do NOT set this to DMY just to make the CSV look British: that swaps the
# meaning of incoming WhatsApp dates. If WhatsApp's own interface is changed
# to a British date locale later, change this input setting to "DMY".
WHATSAPP_NUMERIC_DATE_ORDER = "MDY"
CSV_DATE_OUTPUT_FORMAT = "%d/%m/%Y"
assert WHATSAPP_NUMERIC_DATE_ORDER in {"MDY", "DMY"}


def parse_whatsapp_timestamp(pre_text):
    """Return (DD/MM/YYYY, HH:MM:SS) from WhatsApp message metadata.

    Do not substitute today's date when a historical timestamp cannot be read.
    That would silently convert yesterday's message to a report for today.
    """
    match = re.search(r"\[([^\]]+)\]", pre_text or "")
    if not match:
        raise ValueError("Missing WhatsApp message timestamp metadata")
    parts = match.group(1).split(',', 1)
    if len(parts) != 2:
        raise ValueError(f"Timestamp lacks a date: {match.group(1)!r}")
    raw_time, raw_date = (part.strip() for part in parts)
    today = datetime.now().date()
    date_value = raw_date.casefold()
    if date_value == "today":
        day = today
    elif date_value == "yesterday":
        day = today - timedelta(days=1)
    elif re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}", raw_date):
        day = datetime.strptime(raw_date, "%Y-%m-%d").date()
    elif re.fullmatch(r"\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}", raw_date):
        # Source input order must be explicit. Guessing from the numbers is
        # unreliable when both components are <=12 (e.g. 11/10/2026).
        # Here MDY 11/10/2026 = 10 November, not 11 October.
        a, b, year = re.split(r"[/.-]", raw_date)
        fmt = "%m/%d/%Y" if WHATSAPP_NUMERIC_DATE_ORDER == "MDY" else "%d/%m/%Y"
        if len(year) == 2:
            fmt = fmt.replace("%Y", "%y")
        day = datetime.strptime("/".join((a, b, year)), fmt).date()
    elif date_value in {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}:
        weekday = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"].index(date_value)
        days_ago = (today.weekday() - weekday) % 7
        day = today - timedelta(days=days_ago)
    else:
        raise ValueError(f"Unsupported WhatsApp date: {raw_date!r}")

    # WhatsApp often supplies time as 9:05 AM or 21:05; normalise it as well.
    time_text = re.sub(r"\s+", " ", raw_time.upper()).strip()
    parsed_time = None
    for fmt in ("%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"):
        try:
            parsed_time = datetime.strptime(time_text, fmt).time()
            break
        except ValueError:
            continue
    if parsed_time is None:
        raise ValueError(f"Unsupported WhatsApp time: {raw_time!r}")
    return day.strftime(CSV_DATE_OUTPUT_FORMAT), parsed_time.strftime("%H:%M:%S")

def append_to_local_excel(group, sender, message, metrics, actual_date, actual_time):
    # CSV/Excel invariant: dates are ALWAYS DD/MM/YYYY, including if an
    # external caller provided a date object or ISO date string.
    actual_date = parse_date(actual_date).strftime(CSV_DATE_OUTPUT_FORMAT)
    row_data = [actual_date, actual_time, group, sender, message, metrics.get("T002/T003", ""), metrics.get("PWT", ""), metrics.get("SPL", ""), metrics.get("PL", ""), metrics.get("Near Miss", 0), metrics.get("Late", 0)]
    csv_headers = ["Date", "Time", "Group", "Sender", "Full Message", "T002/T003", "PWT", "SPL", "PL",  "Near Miss", "Late"]
    
    file_exists = os.path.exists(STAGING_CSV_PATH)
    try:
        parent_csv_dir = os.path.dirname(STAGING_CSV_PATH)
        if parent_csv_dir and not os.path.exists(parent_csv_dir):
            os.makedirs(parent_csv_dir, exist_ok=True)
            
        with open(STAGING_CSV_PATH, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(csv_headers)
            writer.writerow(row_data)
        logging.info("📝 Message safely staged to CSV backup.")
    except Exception as e:
        logging.error(f"❌ Critical CSV Staging Write Error: {e}")
        return
    
    try:
        parent_excel_dir = os.path.dirname(LOCAL_FILE_PATH)
        if parent_excel_dir and not os.path.exists(parent_excel_dir):
            os.makedirs(parent_excel_dir, exist_ok=True)
            
        if not os.path.exists(LOCAL_FILE_PATH):
            wb = Workbook()
            sheet = wb.active
            sheet.append(csv_headers)
            wb.save(LOCAL_FILE_PATH)
        wb = load_workbook(LOCAL_FILE_PATH)
        sheet = wb.active
        sheet.append(row_data)
        wb.save(LOCAL_FILE_PATH)
        logging.info("📊 Excel file synchronized straight to master layout.")
    except PermissionError:
        logging.warning("⚠️ Master Excel is OPEN. Data is safe inside temporary CSV pipeline.")
    except Exception as e:
        logging.error(f"❌ Local Excel Write Error: {e}")

async def process_live_feed(messages, group_name):
    global recently_logged_fingerprints
    # Process every currently loaded message. The previous last-10 slice
    # silently dropped previous-day messages in busy groups.
    for msg in messages:
        try:
            text_el = await msg.query_selector("span.selectable-text") or                       await msg.query_selector(".selectable-text span") or                       await msg.query_selector(".copyable-text") or                       await msg.query_selector("[class*='message-text']")
            if not text_el:
                continue
            message_text = (await text_el.inner_text()).strip()
            if not message_text:
                continue
                
            parent_div = await msg.query_selector("div.copyable-text")
            sender = "Unknown Sender"
            pre_text = await parent_div.get_attribute("data-pre-plain-text") if parent_div else None
            if pre_text:
                try:
                    msg_date, msg_time = parse_whatsapp_timestamp(pre_text)
                except ValueError as err:
                    logging.warning("Skipping message with invalid timestamp in %s: %s", group_name, err)
                    continue
            else:
                logging.debug("Skipping message without WhatsApp timestamp in %s", group_name)
                continue
            if "]" in pre_text:
                sender = pre_text.split("]")[-1].replace(":", "").strip()
            if sender == "Unknown Sender":
                if await msg.query_selector(".message-out") or await msg.query_selector("[class*='message-out']"):
                    sender = "You"
                else:
                    name_container = await msg.query_selector("span[dir='auto']")
                    if name_container:
                        sender = await name_container.inner_text()
            
            layout_token = f"{group_name}_{sender}_{msg_date}_{msg_time}_{message_text}"
            if layout_token in recently_logged_fingerprints:
                continue
            metrics_dict, values_detected = parse_metrics_from_text(message_text)
            if values_detected:
                logging.info(f"🎯 Metrics Found: Group '{group_name}' | From '{sender}'\n   Text: {message_text}")
                append_to_local_excel(group_name, sender, message_text, metrics_dict, msg_date, msg_time)
                # Mark processed only after the CSV/database writer returned.
                recently_logged_fingerprints.add(layout_token)
                if len(recently_logged_fingerprints) > 500:
                    recently_logged_fingerprints.pop()
        except Exception:
            logging.exception("WhatsApp extraction failed for group %s", group_name)

async def catch_up_recent_history(page, group_name, lookback_days=2, max_scrolls=22):
    """Scroll backwards through a group's virtualised history on startup.

    WhatsApp only renders a window of messages; merely opening an unread chat
    often exposes the newest items, not yesterday's report. Scan each window,
    page upward, and stop once earlier-than-lookback items are reached.
    This is a bounded catch-up, not an unbounded full-history download.
    """
    cutoff = datetime.now().date() - timedelta(days=lookback_days)
    seen_windows = set()
    for step in range(max_scrolls + 1):
        messages = await page.query_selector_all("#main [data-testid='msg-container']")
        if not messages:
            messages = await page.query_selector_all("div[data-testid='msg-container']")
        if messages:
            await process_live_feed(messages, group_name)

        # Detect oldest visible message to avoid scrolling across years.
        oldest = None
        for msg in messages[:5]:
            pre = await msg.query_selector("div.copyable-text[data-pre-plain-text]")
            if not pre:
                continue
            metadata = await pre.get_attribute('data-pre-plain-text')
            try:
                dt, _ = parse_whatsapp_timestamp(metadata)
                parsed = datetime.strptime(dt, "%d/%m/%Y").date()
                oldest = parsed if oldest is None else min(oldest, parsed)
            except ValueError:
                pass
        if oldest is not None and oldest < cutoff:
            logging.info("History catch-up reached %s for %s", oldest, group_name)
            break

        # Scroll the actual conversation pane, not the contacts sidebar.
        position = await page.evaluate("""() => {
          const msg = document.querySelector('#main [data-testid="msg-container"]');
          if (!msg) return null;
          let p = msg.parentElement;
          while (p && p.id !== 'app') {
            if (p.scrollHeight > p.clientHeight + 40 && p.clientHeight > 80) {
              const before = p.scrollTop;
              p.scrollTop = Math.max(0, before - Math.max(300, p.clientHeight * 0.85));
              return {before, after:p.scrollTop, height:p.scrollHeight};
            }
            p = p.parentElement;
          }
          return null;
        }""")
        if position is None:
            logging.warning("No scrollable history panel found for %s", group_name)
            break
        await asyncio.sleep(0.45)
        # Virtualised lists can return a temporary scrollTop of zero while loading.
        # Use message fingerprints to recognise a genuinely stagnant history.
        first = await page.query_selector('#main [data-testid="msg-container"] div.copyable-text[data-pre-plain-text]')
        key = await first.get_attribute('data-pre-plain-text') if first else None
        if key in seen_windows and position['after'] <= 0:
            break
        seen_windows.add(key)
    await page.evaluate("""() => {
      const msg = document.querySelector('#main [data-testid="msg-container"]');
      let p = msg?.parentElement;
      while (p && p.id !== 'app') {
        if (p.scrollHeight > p.clientHeight + 40 && p.clientHeight > 80) {
          p.scrollTop = p.scrollHeight;
          break;
        }
        p = p.parentElement;
      }
    }""")
    logging.info("Completed previous-day catch-up for %s (%d scan windows)", group_name, step + 1)


# ==============================================================================


### Verify date handling before starting WhatsApp

Checks US-format WhatsApp metadata converts to UK-format output.

In [ ]:
# Date-format smoke test — runs without WhatsApp login or database edits.
assert WHATSAPP_NUMERIC_DATE_ORDER == "MDY"
assert parse_whatsapp_timestamp("[10:44, 11/10/2026] Team:") == ("10/11/2026", "10:44:00")
assert parse_whatsapp_timestamp("[18:38, 11/09/2026] Team:") == ("09/11/2026", "18:38:00")
assert parse_whatsapp_timestamp("[12:56, 11/06/2026] Team:") == ("06/11/2026", "12:56:00")
assert normalise_datetime("10/11/2026", "13:15:00") == "2026-11-10T13:15:00"
print("Date tests passed: incoming MDY -> staging DMY -> SQLite ISO")

## 4. Browser monitoring engine

Existing browser collection workflow, unchanged except for startup format diagnostic.

In [ ]:
# SECTION 3: SCRAPER ENGINE INTERNALS (THREAD ISOLATED)
# ==============================================================================
def trigger_start_engine(window_reference):
    global is_running, worker_loop
    if is_running:
        return
    if not TARGET_GROUPS:
        logging.error("❌ Action Interrupted: Tracking array filters cannot be blank.")
        return

    is_running = True
    window_reference.btn_start.configure(state="disabled")
    window_reference.btn_terminate.configure(state="normal")
    window_reference.update_status_led_threadsafe("#2ecc71", "Active")
    window_reference.update_contextual_glow_threadsafe("green")

    def run_loop(loop):
        asyncio.set_event_loop(loop)
        loop.run_until_complete(start_scraping_engine(window_reference))

    worker_loop = asyncio.new_event_loop()
    t = threading.Thread(target=run_loop, args=(worker_loop,), daemon=True)
    t.start()

def trigger_stop_engine(window_reference):
    global is_running, worker_loop
    if not is_running:
        return
    is_running = False
    logging.info("⏳ Sending clean up kill-stack events to background threads...")
    if worker_loop and worker_loop.is_running():
        asyncio.run_coroutine_threadsafe(stop_scraping_engine_internal(window_reference), worker_loop)

async def stop_scraping_engine_internal(window_reference):
    global browser_context, playwright_instance, is_running
    is_running = False
    try:
        if browser_context:
            await browser_context.close()
            browser_context = None
        if playwright_instance:
            await playwright_instance.stop()
            playwright_instance = None
    except Exception:
        pass
    finally:
        window_reference.update_status_led_threadsafe("#e74c3c", "Stopped")
        window_reference.update_contextual_glow_threadsafe("default")
        try:
            window_reference.btn_start.configure(state="normal")
            window_reference.btn_terminate.configure(state="disabled")
        except Exception:
            pass
        logging.info("🛑 Engine structural systems offline.")

async def start_scraping_engine(window_reference):
    global is_running, browser_context, playwright_instance
    logging.info("🚀 Allocating Chromium Playwright engine tasks...")
    logging.info("Date formats: WhatsApp incoming=%s, staging CSV/Excel=DD/MM/YYYY",
                 WHATSAPP_NUMERIC_DATE_ORDER)
    try:
        playwright_instance = await async_playwright().start()
        browser_context = await playwright_instance.chromium.launch_persistent_context(
            user_data_dir=os.path.join(BASE_DIR, "whatsapp_session"),
            headless=False,
            args=["--disable-blink-features=AutomationControlled"]
        )
        
        pages_list = browser_context.pages
        page = pages_list[0] if pages_list else await browser_context.new_page()
        
        await page.goto("https://web.whatsapp.com")
        logging.info("Waiting for WhatsApp Web... Please scan QR code if prompted.")
        
        try:
            await page.wait_for_selector("div[data-testid='chat-list']", timeout=90000)
            logging.info("Connected successfully! Anti-lock data parser active...")
        except Exception:
            logging.warning("⚠️ Timeout or alternative layout hook during initialization check.")
            await page.wait_for_selector("[id='app']", timeout=30000)
            logging.info("Connected via application frame checkpoint.")

        await asyncio.sleep(3)
        
        # Track rotation timing (e.g., proactive sweep every 30 seconds)
        last_proactive_sweep = datetime.now() - timedelta(seconds=6)
        sweep_interval_seconds = 5
        backfilled_groups = set()  # reset on each application monitoring session 
        
        while is_running:
            try:
                # --- METHOD 1: PROCESS THE CURRENTLY OPEN CHAT ---
                active_header_el = await page.query_selector("div[data-testid='conversation-header']") or \
                                   await page.query_selector("[class*='conversation-header']")
                if active_header_el:
                    title_el = await active_header_el.query_selector("span[title]")
                    if title_el:
                        active_group_name = await title_el.get_attribute("title")
                        if active_group_name in TARGET_GROUPS:
                            all_messages = await page.query_selector_all("div[data-testid='msg-container']") or \
                                           await page.query_selector_all("div[role='row']") or \
                                           await page.query_selector_all("[class*='message-']")
                            if all_messages:
                                await process_live_feed(all_messages, active_group_name)
                                
                # --- METHOD 2: REACT TO GREEN UNREAD BADGES ---
                unread_chats = await page.query_selector_all("span[aria-label*='unread message']") or \
                               await page.query_selector_all("[aria-label*='unread']")
                for chat in unread_chats:
                    chat_row = await chat.evaluate_handle("el => el.closest('div[role=\\'row\\']') || el.closest('[class*=\\'row\\']')")
                    if not chat_row:
                        continue
                    group_title_el = await chat_row.query_selector("span[title]")
                    if not group_title_el:
                        continue
                    group_name = await group_title_el.get_attribute("title")
                    if group_name in TARGET_GROUPS:
                        logging.info(f"📬 [Reactive Badge] Parsing downstream unread stack in group '{group_name}'...")
                        await chat_row.click()
                        await asyncio.sleep(1.5)
                        all_messages = await page.query_selector_all("div[data-testid='msg-container']") or \
                                       await page.query_selector_all("div[role='row']")
                        if all_messages:
                            await process_live_feed(all_messages, group_name)

                # --- METHOD 3: PROACTIVE ROTATION SWEEP ---
                # Runs automatically if the time interval has elapsed, tracking what was missed by phone views
                time_since_last_sweep = (datetime.now() - last_proactive_sweep).total_seconds()
                if time_since_last_sweep >= sweep_interval_seconds:
                    logging.info(f"🔄 [Proactive Sweep] Cycling target groups to catch up on items read externally...")
                    
                    for group_to_check in TARGET_GROUPS:
                        # Find the chat item in the left-side list by searching for its title text attribute
                        chat_row_el = await page.query_selector(f"span[title='{group_to_check}']")
                        if chat_row_el:
                            list_item_clickable = await chat_row_el.evaluate_handle(
                                "el => el.closest('div[role=\\'row\\']') || el.closest('[class*=\\'row\\']') || el"
                            )
                            if list_item_clickable:
                                await list_item_clickable.click()
                                await asyncio.sleep(1.5) # Give the chat pane a brief window to load
                                
                                all_messages = await page.query_selector_all("div[data-testid='msg-container']") or \
                                               await page.query_selector_all("div[role='row']")
                                if all_messages:
                                    await process_live_feed(all_messages, group_to_check)
                                if group_to_check not in backfilled_groups:
                                    # One bounded previous-day scan per group at start.
                                    # The normal live scanning loop is unchanged.
                                    await catch_up_recent_history(page, group_to_check)
                                    backfilled_groups.add(group_to_check)
                    
                    last_proactive_sweep = datetime.now() # Reset our interval clock tracker
                    
            except Exception as loop_err:
                logging.debug(f"Engine iteration warning: {loop_err}")
                pass
            
            await asyncio.sleep(1)
            
    except Exception as e:
        logging.error(f"❌ Core Parser Engine Crash Loop: {e}")
    finally:
        await stop_scraping_engine_internal(window_reference)

# ==============================================================================


## 5. SQLite bridge and monitor controls

The original CSV writer remains the first stage, followed by a SQLite insert.

In [ ]:
# SECTION 4: CUSTOMTKINTER INTERACTIVE GRAPHICAL DASHBOARD WINDOW
# ==============================================================================



# Minimal compatibility bridge: never modify the user's original parser or
# browser navigation. The existing CSV/Excel writer is called first, and the
# *same* parsed record is mirrored into SQLite for the executive dashboards.
_original_append_to_local_excel = append_to_local_excel
_record_sink = None

def append_to_local_excel(group, sender, message, metrics, actual_date, actual_time):
    _original_append_to_local_excel(group, sender, message, metrics, actual_date, actual_time)
    try:
        when = normalise_datetime(actual_date, actual_time)
        added = insert_report(group, sender, message, when, metrics, origin='original_whatsapp_engine')
        if added:
            # A successfully parsed message confirms that an authenticated
            # WhatsApp conversation was accessible (even on alternate layouts).
            if is_running and monitor_state()['status'] != 'Connected':
                set_monitor_state('Connected', success=True)
            if _record_sink is not None:
                _record_sink(('reports_saved', 1, group))
            save_collector_diagnostics(group, {'visible': 1, 'metrics': 1, 'saved': 1, 'duplicate': 0, 'missing_timestamp': 0}, 'Original engine wrote report; bridged to dashboard')
            LOG.info('Bridged WhatsApp report into live SQLite dashboard: %s', group)
    except Exception as exc:
        LOG.exception('Report was seen by original parser but SQLite bridge failed: %s', exc)
        if _record_sink is not None:
            _record_sink('LIVE BRIDGE ERROR: ' + str(exc))

class _NoopButton:
    def configure(self, **kwargs):
        pass

class _OriginalWindowAdapter:
    # Only the UI functions that the original engine calls are adapted.
    btn_start = _NoopButton()
    btn_terminate = _NoopButton()
    def __init__(self, emit):
        self.emit = emit
    def update_status_led_threadsafe(self, color, state):
        # The original engine invokes this upon starting and stopping. "Active"
        # is not proof WhatsApp is authenticated; only mark green on verified login.
        if state == 'Stopped':
            set_monitor_state('Disconnected')
        self.emit('WhatsApp engine: ' + state)
    def update_contextual_glow_threadsafe(self, color):
        pass
    def update_status_led(self, color, state):
        self.emit('WhatsApp engine: ' + state)
    def update_contextual_glow(self, color):
        pass

class _OriginalLogBridge(logging.Handler):
    def __init__(self, sink):
        super().__init__(logging.INFO)
        self.sink = sink
    def emit(self, record):
        message = record.getMessage()
        if 'Connected successfully!' in message:
            # The original engine detected the authenticated chat list.
            set_monitor_state('Connected', success=True)
            self.sink('WhatsApp authenticated: live monitoring connected')
        elif 'Connected via application frame' in message:
            # App frame alone also exists at the QR login screen. Do not claim
            # a connection based only on that fallback selector.
            self.sink('WhatsApp loaded; chat authentication not yet verified')
        elif 'Core Parser Engine Crash' in message:
            set_monitor_state('Error', message)
            self.sink('Original engine error: ' + message)

class WhatsAppMonitor:
    """Thin adapter around original trigger_start_engine/trigger_stop_engine."""
    def __init__(self, callback=None):
        global _record_sink
        self.callback = callback or (lambda msg: None)
        _record_sink = self.callback
        self.window = _OriginalWindowAdapter(self.callback)
        self.log_handler = _OriginalLogBridge(self.callback)
        logging.getLogger().addHandler(self.log_handler)

    @property
    def active(self):
        return bool(is_running)

    def start(self):
        global TARGET_GROUPS
        if is_running:
            return False
        TARGET_GROUPS = list(configuration()['groups'])
        if not TARGET_GROUPS:
            self.callback('Configure at least one WhatsApp group first')
            return False
        # Prevent redownloading all historical rows; previously captured CSV
        # messages can be reconciled through the idempotent SQLite importer.
        set_monitor_state('Starting')
        self.callback('Starting original WhatsApp engine for: ' + ', '.join(TARGET_GROUPS))
        load_existing_fingerprints()
        trigger_start_engine(self.window)
        return True

    def stop(self):
        if is_running:
            set_monitor_state('Disconnecting')
            trigger_stop_engine(self.window)
            self.callback('Stopping original WhatsApp engine')
        else:
            set_monitor_state('Disconnected')


def reconcile_original_csv():
    """Catch up original CSV records on application start, idempotently."""
    if not Path(STAGING_CSV_PATH).is_file():
        return 0
    added, dupes, errors = import_csv(STAGING_CSV_PATH, quarantine_legacy_dates=True)
    if errors:
        LOG.warning('Legacy CSV reconciliation: %d invalid rows; first: %s',len(errors),errors[:2])
    return added


## 6. Multi-user daily control — streamlined v3.0.0

Account administration is under **Settings**. In **Exceptions**, a service date without records shows an empty state; use **Start entries for this date** if you need to record changes before WhatsApp attendance arrives. The Executive chart uses four colours and labels all values as event counts, not unique personnel.


In [ ]:
# ---- Local CustomTkinter executive desktop UI ----
"""Local desktop executive dashboard. Run: python app.py"""
import logging, queue, tkinter as tk
import json, os, re, sys, subprocess, webbrowser, uuid
from datetime import date, datetime, timedelta
from pathlib import Path
from tkinter import filedialog, messagebox, ttk

try:
    import customtkinter as ctk
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
except ImportError as exc:
    raise SystemExit(f'Missing dependency: {exc}. Install with: pip install -r requirements.txt')


# All UI colours come from one palette so Tk, ttk and Matplotlib stay in sync.
PALETTES = {
    'Dark': dict(NAV='#0D1626', BG='#101A2B', PANEL='#1B2B41', BORDER='#2D4059',
                 MUTED='#A4B6C9', WHITE='#F1F6FE', BLUE='#56B9FA',
                 TEAL='#54D7BA', AMBER='#FFCB78', RED='#FF8B92'),
    'Light': dict(NAV='#E9F0F7', BG='#F5F8FC', PANEL='#FFFFFF', BORDER='#D3DFEB',
                  MUTED='#53667C', WHITE='#1C3049', BLUE='#1264AB',
                  TEAL='#13765F', AMBER='#9C6509', RED='#B63246'),
}

def apply_palette(mode):
    """Update global colour tokens; rebuild GUI only on user-initiated toggle."""
    colors = PALETTES['Light' if mode == 'Light' else 'Dark']
    globals().update(colors)
    ctk.set_appearance_mode('light' if mode == 'Light' else 'dark')
    return 'Light' if mode == 'Light' else 'Dark'

ctk.set_default_color_theme('blue')
apply_palette(configuration().get('theme', 'Dark'))

class Application(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title(f'Operational Intelligence v{APP_VERSION} | Executive Command Centre')
        self.geometry('1480x880');self.minsize(1100,710)
        # Jupyter may configure the root logger at WARNING; enable the original
        # collector's INFO connection events so the status LED is trustworthy.
        root_logger = logging.getLogger()
        root_logger.setLevel(logging.INFO)
        if not any(isinstance(h, logging.FileHandler) and
                   Path(h.baseFilename).resolve() == (DATA / 'collector.log').resolve()
                   for h in root_logger.handlers):
            root_logger.addHandler(logging.FileHandler(DATA / 'collector.log', encoding='utf-8'))
        self.theme_mode = apply_palette(configuration().get('theme', 'Dark'))
        self.configure(fg_color=BG)
        self.queue=queue.Queue();self.monitor=WhatsAppMonitor(self.queue.put)
        # A previous process may have terminated unexpectedly while its
        # SQLite status was Connected. This new process starts disconnected.
        if not is_running:
            core.set_monitor_state('Disconnected')
        self.page='Overview';self._chart=None;self._rows=[];self._last_sig=None;self._last_report_day=None;self._refresh_pending=False;self._next_fallback_check=0.0;self._next_daily_check=0.0;self._next_connection_led_check=0.0;self._live_widgets={};self._visible_record_ids=();self._last_live_render=0.0
        self.period=ctk.StringVar(value='All time');self.group=ctk.StringVar(value='All Groups')
        self.custom_start=ctk.StringVar(value=(date.today()-timedelta(days=6)).isoformat());self.custom_end=ctk.StringVar(value=date.today().isoformat())
        self.status_text=ctk.StringVar(value='Local database ready')
        self.exception_session=None
        self._exception_draft_dirty=False
        self._exception_started_date=None
        self._show_settings_accounts=False
        self._plan_source_mtime=None
        self._plan_import_busy=False
        self._next_plan_check=0.0
        self.exception_group=ctk.StringVar(value=core.configuration()['groups'][0] if core.configuration()['groups'] else 'Gang 1')
        self.exception_day=ctk.StringVar(value=date.today().isoformat())
        self._reconcile_busy=False
        self._nav_compact=False
        self._nav_render_job=None
        self._nav_generation=0
        self._loading_progress=None
        self._last_view_refresh='Not refreshed yet'
        self._layout();self._navigate('Overview');self.after(100,self._poll);self.after(250,self._reconcile_legacy)
        self.protocol('WM_DELETE_WINDOW',self._close)

    def _reconcile_legacy(self):
        def work():
            self._reconcile_busy=True
            try:
                count = reconcile_original_csv()
                self.queue.put(('reports_saved', count, 'original CSV reconciliation'))
            except Exception as exc:
                self.queue.put('Original CSV reconciliation error: ' + str(exc))
            finally:
                self._reconcile_busy=False
        threading.Thread(target=work, daemon=True).start()

    def label(self,parent,text,size=13,color=WHITE,bold=False,**kwargs):
        return ctk.CTkLabel(parent,text=text,font=ctk.CTkFont(family='Segoe UI',size=size,weight='bold' if bold else 'normal'),text_color=color,**kwargs)
    def button(self,parent,text,command,color=None,width=120,**kw):
        # Explicit green/red buttons retain white labels even in light mode.
        custom = color is not None and color != 'transparent'
        color = color if color is not None else ('#D9EAF8' if self.theme_mode == 'Light' else '#254C72')
        fg_text = '#FFFFFF' if custom else WHITE
        if color == 'transparent': fg_text = WHITE
        return ctk.CTkButton(parent,text=text,command=command,fg_color=color,
                             text_color=fg_text, hover_color=('#C0D9ED' if self.theme_mode == 'Light' and not custom else '#346A98'),
                             font=ctk.CTkFont(family='Segoe UI',size=12,weight='bold'),
                             corner_radius=10,width=width,height=36,**kw)
    def panel(self,parent,**kwargs):return ctk.CTkFrame(parent,fg_color=PANEL,corner_radius=14,border_width=1,border_color=BORDER,**kwargs)
    def _table_colors(self):
        """Accessible executive table palette, coordinated with the active theme."""
        if self.theme_mode == 'Light':
            return dict(background='#F4F8FE', alternate='#EAF2FB',
                        foreground='#203850', heading='#243D5B', heading_ink='#F9FCFF',
                        heading_hover='#315778', border='#C9D9E9', selected='#246A9B',
                        selected_ink='#FFFFFF', scrollbar='#7196B7',
                        scrollbar_hover='#457DA7', track='#DFE9F5', editor='#FFFFFF')
        return dict(background='#14273B', alternate='#1B3149',
                    foreground='#E8F3FF', heading='#27415D', heading_ink='#F4F9FF',
                    heading_hover='#315575', border='#36506A', selected='#276E9D',
                    selected_ink='#FFFFFF', scrollbar='#557894',
                    scrollbar_hover='#7BA4C8', track='#162A40', editor='#203A54')

    def _style_table(self, name, rowheight=34, fontsize=10):
        """Style a Tk table with consistent headings, selection and themed scrollbars.

        Tk's native Treeview remains keyboard and screen-reader friendly; changing
        a ttk style does not rebuild the data model or replace its edit bindings.
        """
        colors=self._table_colors()
        style=ttk.Style(self)
        if style.theme_use()!='clam':
            style.theme_use('clam')
        table_style=f'{name}.Treeview'
        style.configure(table_style,
                        font=('Segoe UI',fontsize),rowheight=rowheight,
                        background=colors['background'],foreground=colors['foreground'],
                        fieldbackground=colors['background'],borderwidth=0,
                        relief='flat',lightcolor=colors['background'],darkcolor=colors['background'])
        style.map(table_style,
                  background=[('selected',colors['selected'])],
                  foreground=[('selected',colors['selected_ink'])])
        head_style=f'{table_style}.Heading'
        style.configure(head_style,
                        background=colors['heading'],foreground=colors['heading_ink'],
                        font=('Segoe UI',fontsize,'bold'),padding=(12,12),
                        borderwidth=0,relief='flat',
                        lightcolor=colors['heading'],darkcolor=colors['heading'])
        style.map(head_style,
                  background=[('active',colors['heading_hover']),
                              ('pressed',colors['heading_hover'])],
                  foreground=[('active',colors['heading_ink'])])
        for axis in ('Vertical','Horizontal'):
            name_scroll=f'Ops.{axis}.TScrollbar'
            style.configure(name_scroll,
                background=colors['scrollbar'],troughcolor=colors['track'],
                arrowcolor=colors['heading_ink'],bordercolor=colors['track'],
                lightcolor=colors['scrollbar'],darkcolor=colors['scrollbar'],
                gripcount=0,relief='flat',borderwidth=0,
                width=12)
            style.map(name_scroll,
                background=[('active',colors['scrollbar_hover']),
                            ('pressed',colors['scrollbar_hover'])])
        return table_style

    def _stripe_rows(self, tree):
        """Apply subtle alternating backgrounds without destroying status tags."""
        palette=self._table_colors()
        tree.tag_configure('ops_even',background=palette['background'])
        tree.tag_configure('ops_odd',background=palette['alternate'])
        for index, item in enumerate(tree.get_children('')):
            existing=tuple(t for t in tree.item(item,'tags') if t not in ('ops_even','ops_odd'))
            stripe='ops_odd' if index%2 else 'ops_even'
            if tree.item(item,'tags')!=(stripe,)+existing:
                tree.item(item,tags=(stripe,)+existing)
        # Hover is visual only: it does not change selection, values or editor bindings.
        if not getattr(tree,'_ops_hover_bound',False):
            tree._ops_hover_bound=True
            tree._ops_hover_id=None
            tree.tag_configure('ops_hover',background=('#D8E8F7' if self.theme_mode=='Light' else '#274866'))
            def clear_hover():
                old=tree._ops_hover_id
                if old and tree.exists(old):
                    tree.item(old,tags=tuple(x for x in tree.item(old,'tags') if x!='ops_hover'))
                tree._ops_hover_id=None
            def hover(event):
                row=tree.identify_row(event.y)
                if row==tree._ops_hover_id:return
                clear_hover()
                if row and tree.exists(row):
                    tree.item(row,tags=tuple(x for x in tree.item(row,'tags') if x!='ops_hover')+('ops_hover',))
                    tree._ops_hover_id=row
            tree.bind('<Motion>',hover,add='+')
            tree.bind('<Leave>',lambda _:clear_hover(),add='+')

    def _empty_state(self,parent,title,description,action_label=None,action=None,kind='neutral',compact=False):
        """Accessible distinction between missing data, pending data, and clear filters."""
        accent={'neutral':BLUE,'pending':AMBER,'success':TEAL,'error':RED}.get(kind,BLUE)
        container=self.panel(parent)
        container.pack(fill='x',pady=(7,16))
        header=ctk.CTkFrame(container,fg_color='transparent')
        header.pack(fill='x',padx=18,pady=(14 if compact else 20,5))
        glyph={'neutral':'○','pending':'◷','success':'✓','error':'!'} .get(kind,'○')
        self.label(header,glyph,25,accent,True).pack(side='left',padx=(0,12))
        self.label(header,title,15 if compact else 18,WHITE,True).pack(side='left')
        self.label(container,description,11 if compact else 12,MUTED,wraplength=940,
                   justify='left').pack(anchor='w',padx=20,pady=(0,12))
        if action_label and action:
            self.button(container,action_label,action,width=225).pack(anchor='w',padx=20,pady=(0,18))
        else:
            ctk.CTkFrame(container,height=5,fg_color='transparent').pack()
        return container

    def _show_loading_state(self,page):
        """Show an actual animated Tk loading state while navigation is queued."""
        for widget in self.content.winfo_children():widget.destroy()
        card=self.panel(self.content)
        card.pack(fill='x',pady=(24,14))
        self.label(card,'OPENING  /  '+page.upper(),12,BLUE,True).pack(anchor='w',padx=22,pady=(22,8))
        self.label(card,'Preparing your dashboard',19,WHITE,True).pack(anchor='w',padx=22)
        self.label(card,'Loading saved views and charts. Your operational records are unchanged.',
                   12,MUTED).pack(anchor='w',padx=22,pady=(5,16))
        self._loading_progress=ctk.CTkProgressBar(card,mode='indeterminate',height=7,
                                                    progress_color=BLUE,fg_color=BORDER)
        self._loading_progress.pack(fill='x',padx=22,pady=(0,24))
        self._loading_progress.start()

    def _finish_loading(self):
        if self._loading_progress is not None:
            try:self._loading_progress.stop()
            except Exception:pass
            self._loading_progress=None

    def _toggle_compact_sidebar(self):
        self._nav_compact=not self._nav_compact
        compact=self._nav_compact
        self.sidebar.configure(width=76 if compact else 228)
        self.brand_label.configure(text='◈' if compact else '◈   OPERATIONS')
        self.brand_subtitle.pack_forget()
        if not compact:self.brand_subtitle.pack(anchor='w',padx=22,pady=(0,32),before=next(iter(self.nav.values())))
        self.collection_label.pack_forget()
        if not compact:self.collection_label.pack(anchor='w',padx=22,pady=(35,8),before=self.sidebar_start)
        self.collection_hint.pack_forget()
        if not compact:self.collection_hint.pack(padx=16,pady=9,after=self.sidebar_stop)
        self.nav_toggle.configure(text='»' if compact else '«')
        for name,btn in self.nav.items():
            btn.configure(text=(self.nav_icons[name] if compact else
                         f'{self.nav_icons[name]}    {name}'),width=52 if compact else 190,
                         anchor='center' if compact else 'w')
            btn.pack_configure(padx=9 if compact else 17)
        self.sidebar_start.configure(text='▶' if compact else '▶  Start WhatsApp',
                                     width=52 if compact else 188)
        self.sidebar_stop.configure(text='■' if compact else '■  Stop monitoring',
                                    width=52 if compact else 188)
        self.brand_label.pack_configure(padx=18 if compact else 21)

    def _layout(self):
        self.grid_columnconfigure(1,weight=1);self.grid_rowconfigure(0,weight=1)
        side=ctk.CTkFrame(self,fg_color=NAV,corner_radius=0,width=228)
        self.sidebar=side
        side.grid(row=0,column=0,sticky='nsew');side.grid_propagate(False)
        self.nav_toggle=self.button(side,'«',self._toggle_compact_sidebar,width=35,color='transparent')
        self.nav_toggle.pack(anchor='e',padx=10,pady=(12,3))
        self.brand_label=self.label(side,'◈   OPERATIONS',19,BLUE,True)
        self.brand_label.pack(anchor='w',padx=21,pady=(10,4))
        self.brand_subtitle=self.label(side,'INTELLIGENCE  /  DESKTOP',10,MUTED,True)
        self.brand_subtitle.pack(anchor='w',padx=22,pady=(0,32))
        self.nav_icons={'Overview':'◈','Workforce':'▤','Executive':'▦','Exceptions':'✎',
                        'Safety & Alerts':'⚑','Reports':'▣','System Health':'◉','Settings':'⚙'}
        self.nav={}
        for name in self.nav_icons:
            b=self.button(side,f'{self.nav_icons[name]}    {name}',
                          lambda n=name:self._navigate(n),width=190,color='transparent',anchor='w')
            b.pack(padx=17,pady=5);self.nav[name]=b
        self.collection_label=self.label(side,'COLLECTION CONTROL',10,MUTED,True)
        self.collection_label.pack(anchor='w',padx=22,pady=(35,8))
        self.sidebar_start=self.button(side,'▶  Start WhatsApp',self._start,color='#166653',width=188)
        self.sidebar_start.pack(pady=5)
        self.sidebar_stop=self.button(side,'■  Stop monitoring',self._stop,color='#69424D',width=188)
        self.sidebar_stop.pack(pady=5)
        self.collection_hint=self.label(side,'Requires WhatsApp Web sign-in',10,MUTED,wraplength=180)
        self.collection_hint.pack(padx=16,pady=9)
        self.main=ctk.CTkFrame(self,fg_color=BG,corner_radius=0)
        self.main.grid(row=0,column=1,sticky='nsew');self.main.grid_columnconfigure(0,weight=1)
        bar=ctk.CTkFrame(self.main,fg_color='transparent')
        bar.grid(row=0,column=0,sticky='ew',padx=22,pady=(17,7))
        headings=ctk.CTkFrame(bar,fg_color='transparent')
        headings.pack(side='left',fill='x',expand=True)
        self.heading=self.label(headings,'Operational Overview',25,WHITE,True)
        self.heading.pack(anchor='w')
        self.page_context=self.label(headings,'OPERATIONS  /  LIVE INTELLIGENCE',10,MUTED,True)
        self.page_context.pack(anchor='w',pady=(2,0))
        self.connection_led=self.label(bar,'●  DISCONNECTED',11,RED,True)
        self.connection_led.pack(side='right',padx=8)
        self.button(bar,'☀  Light' if self.theme_mode=='Dark' else '☾  Dark',
                    self._toggle_theme,width=95).pack(side='right',padx=10)
        self._update_connection_led()
        self.content=ctk.CTkScrollableFrame(self.main,fg_color='transparent')
        self.content.grid(row=1,column=0,sticky='nsew',padx=20,pady=8)
        self.main.grid_rowconfigure(1,weight=1)
        foot=ctk.CTkFrame(self.main,fg_color=NAV,height=37,corner_radius=0)
        foot.grid(row=2,column=0,sticky='ew')
        self.label(foot,'●',12,TEAL,True).pack(side='left',padx=(15,7),pady=8)
        self.label(foot,'Desktop ready',10,MUTED,textvariable=self.status_text,
                   width=550,anchor='w').pack(side='left',padx=1,pady=7)
        self.label(foot,'LOCAL-FIRST  •  SQLITE',10,TEAL,True).pack(side='right',padx=16)
        if self._nav_compact:
            self._nav_compact=False
            self._toggle_compact_sidebar()

    def _update_connection_led(self):
        """Paint green only for an active, authenticated WhatsApp monitor."""
        if not hasattr(self, 'connection_led') or not self.connection_led.winfo_exists():
            return
        state = core.monitor_state()['status']
        connected = bool(self.monitor.active and state == 'Connected')
        label = '●  CONNECTED' if connected else ('●  CONNECTING' if self.monitor.active else '●  DISCONNECTED')
        color = TEAL if connected else RED
        if self.connection_led.cget('text') != label or self.connection_led.cget('text_color') != color:
            self.connection_led.configure(text=label, text_color=color)

    def _toggle_theme(self):
        self._set_theme('Light' if self.theme_mode == 'Dark' else 'Dark')

    def _set_theme(self, next_mode):
        """Persist the preference and rebuild only on a manual theme change.

        Live ingestion still updates labels and charts in place; no flicker.
        """
        if next_mode == self.theme_mode:
            return
        cfg = core.configuration()
        cfg['theme'] = next_mode
        try:
            core.save_configuration(cfg)
        except Exception as exc:
            messagebox.showerror('Theme settings', str(exc))
            return
        self.theme_mode = apply_palette(next_mode)
        old_page = self.page
        for widget in self.winfo_children():
            widget.destroy()
        self.configure(fg_color=BG)
        self._layout()
        self._navigate(old_page)

    def _navigate(self,page):
        if (self.page=='Exceptions' and page!='Exceptions'
                and getattr(self,'_exception_draft_dirty',False)):
            if not messagebox.askyesno('Unsaved staffing changes',
                    'Some group counts have not been saved. Discard these drafts and leave the page?'):
                return
            self._exception_draft_dirty=False
        self.page=page
        self.heading.configure(text={'Overview':'Operational Overview','Reports':'Reporting Centre',
            'Executive':'Executive Command Centre',
            'Workforce':'Workforce & Attendance','Safety & Alerts':'Safety & Risk Intelligence',
            'System Health':'System Health & Audit','Settings':'Local Settings',
            'Staffing Plans':'Staffing Planning','Exceptions':'Staffing Exceptions & Changes'}[page])
        selected='#D1E5F7' if self.theme_mode=='Light' else '#23496E'
        for name,btn in self.nav.items():
            btn.configure(fg_color=selected if name==page else 'transparent')
        # Schedule the expensive view creation to allow the loading indicator
        # to paint before Matplotlib and database work begins.
        self._nav_generation+=1
        generation=self._nav_generation
        if self._nav_render_job is not None:
            try:self.after_cancel(self._nav_render_job)
            except Exception:pass
        self._show_loading_state(page)
        self.status_text.set('Opening '+page+'…')
        self._nav_render_job=self.after(75,lambda:self._finish_navigation(page,generation))

    def _finish_navigation(self,page,generation):
        if generation!=self._nav_generation or page!=self.page:return
        self._nav_render_job=None
        try:
            self._render()
            with core.connect() as con:
                row=con.execute('SELECT COUNT(*),MAX(id) FROM reports').fetchone()
            self._last_sig=(tuple(row),core.exception_data_signature(),self.page)
            self.status_text.set(page+' ready · '+datetime.now().strftime('%H:%M:%S'))
        except Exception as exc:
            logging.exception('Unable to open page %s',page)
            for widget in self.content.winfo_children():widget.destroy()
            self._empty_state(self.content,'Unable to display '+page,
                    'The view could not be opened: '+str(exc),
                    'Try refreshing',lambda:self._navigate(page),kind='error')
            self.status_text.set('View error: '+str(exc))

    def _scope(self,parent,show_exports=True):
        row=ctk.CTkFrame(parent,fg_color='transparent');row.pack(fill='x',pady=(0,15))
        self.label(row,'Reporting period',12,MUTED).pack(side='left',padx=(0,8))
        ctk.CTkOptionMenu(row,values=['Today','Yesterday','7 days','14 days','30 days','90 days','All time','Custom'],variable=self.period,
                          command=lambda v:self._render(),width=115,fg_color=PANEL,text_color=WHITE,
                          button_color=('#B8D4ED' if self.theme_mode == 'Light' else '#254C72')).pack(side='left',padx=(0,18))
        self.label(row,'Team',12,MUTED).pack(side='left',padx=(0,8))
        values=['All Groups']+sorted(set(core.groups())|set(core.configuration()['groups']))
        if self.group.get() not in values:self.group.set('All Groups')
        ctk.CTkOptionMenu(row,values=values,variable=self.group,command=lambda v:self._render(),width=190,
                          fg_color=PANEL,text_color=WHITE,
                          button_color=('#B8D4ED' if self.theme_mode == 'Light' else '#254C72')).pack(side='left')
        self.button(row,'↻  Refresh',self._render,width=95).pack(side='right',padx=5)
        if show_exports:
            self.button(row,'Interactive HTML',self._html,width=130).pack(side='right',padx=5)
            self.button(row,'Export PDF',self._pdf,width=100).pack(side='right',padx=5)
        custom=ctk.CTkFrame(parent,fg_color='transparent');custom.pack(fill='x',pady=(0,12))
        self.label(custom,'From',11,MUTED).pack(side='left',padx=(0,4))
        ctk.CTkEntry(custom,textvariable=self.custom_start,placeholder_text='YYYY-MM-DD',width=110).pack(side='left',padx=(0,6))
        self.label(custom,'To',11,MUTED).pack(side='left',padx=(0,4))
        ctk.CTkEntry(custom,textvariable=self.custom_end,placeholder_text='YYYY-MM-DD',width=110).pack(side='left',padx=(0,7))
        self.button(custom,'Apply dates',self._apply_custom_dates,width=94).pack(side='left',padx=(0,10))
        return row

    def _apply_custom_dates(self):
        try:
            a=core.parse_date(self.custom_start.get())
            b=core.parse_date(self.custom_end.get())
            if a>b:raise ValueError('Start date must not be later than end date.')
        except ValueError as exc:
            messagebox.showerror('Invalid custom date range',str(exc)+'\nUse YYYY-MM-DD or DD/MM/YYYY.');return
        self.period.set('Custom');self._render()

    def _date_bounds(self):
        label=self.period.get()
        if label=='Custom':
            try:
                a=core.parse_date(self.custom_start.get());b=core.parse_date(self.custom_end.get())
                if a>b:return date(9999,1,1),date(1000,1,1)
                return a,b
            except ValueError:return date(9999,1,1),date(1000,1,1)
        # Today and Yesterday are exact local calendar days, not rolling 24-hour windows.
        if label=='Today':
            return date.today(), date.today()
        if label=='Yesterday':
            yesterday = date.today() - timedelta(days=1)
            return yesterday, yesterday
        if label=='All time':return None,None
        return date.today()-timedelta(days=int(label.split()[0])-1),date.today()

    def _date_range(self):return self._date_bounds()[0]

    def _selected_rows(self):
        a,b=self._date_bounds()
        return core.filtered(a,b,self.group.get())

    def _reset_view_filters(self):
        self.period.set('All time')
        self.group.set('All Groups')
        self._render()

    def _kpi(self,parent,title,value,sub,color=BLUE):
        p=self.panel(parent);p.pack(side='left',fill='both',expand=True,padx=5)
        ctk.CTkFrame(p,fg_color=color,height=4,corner_radius=3).pack(fill='x',padx=16,pady=(12,1))
        self.label(p,title.upper(),11,MUTED,True).pack(anchor='w',padx=16,pady=(11,5))
        value_label=self.label(p,str(value),30,color,True)
        value_label.pack(anchor='w',padx=16)
        sub_label=self.label(p,sub,11,MUTED)
        sub_label.pack(anchor='w',padx=16,pady=(0,14))
        return value_label,sub_label
    def _title(self,parent,name,detail=None,top_padding=18):
        self.label(parent,name,17,WHITE,True).pack(anchor='w',pady=(top_padding,4))
        if detail:self.label(parent,detail,11,MUTED).pack(anchor='w',pady=(0,7))

    def _render(self):
        """Rebuild only on navigation/filter changes, never on live-message events."""
        # Explicit page/filter navigation may rebuild; automatic updates never do.
        self._finish_loading()
        self._live_widgets={}
        if hasattr(self,'page_context') and self.page_context.winfo_exists():
            if self.page in ('Overview','Executive','Workforce','Safety & Alerts','Reports'):
                detail=f'{self.period.get().upper()}  /  {self.group.get().upper()}'
            elif self.page=='Exceptions':
                detail=f'NIGHT SERVICE  /  {self.exception_day.get()}'
            else:
                detail='OPERATIONS  /  CONFIGURATION & CONTROL'
            self.page_context.configure(text=detail)
        if self.exception_session is not None:
            try:self.exception_session=core.app_session_info(self.exception_session['token'])
            except PermissionError:self.exception_session=None
        for w in self.content.winfo_children():w.destroy()
        self._chart=None
        if self.page in ('Overview','Executive','Workforce','Safety & Alerts','Reports'):
            self._scope(self.content,show_exports=self.page not in ('Overview','Reports'))
            self._rows=self._selected_rows()
        if self.page=='Overview':self._overview()
        elif self.page=='Reports':self._reports_centre()
        elif self.page=='Executive':self._executive()
        elif self.page=='Workforce':self._workforce()
        elif self.page=='Safety & Alerts':self._safety()
        elif self.page=='Exceptions':self._exceptions()
        elif self.page=='System Health':self._health()
        elif self.page=='Settings':
            if self.exception_session is None:
                self._title(self.content,'Settings access',
                            'Sign in as an administrator to manage accounts, groups and staffing plans.')
                self._exception_login()
            elif self.exception_session.get('role')!='admin':
                self._title(self.content,'Administrator access required',
                            'Your operator account can enter changes but cannot manage users or settings.')
                self.button(self.content,'Sign in as administrator',self._settings_switch_account,width=215)
            else:self._settings()
        elif self.page=='Staffing Plans':
            if self.exception_session is None or self.exception_session.get('role')!='admin':
                self._title(self.content,'Administrator access required',
                            'Sign in to Exceptions as an administrator to manage staffing plans.')
                self.button(self.content,'Go to Exceptions sign-in',lambda:self._navigate('Exceptions'),width=220)
            else:self._staffing_plan_settings()

    # ------------------------------------------------------------------
    # OPERATIONAL OVERVIEW - read-only presentation of existing v3.0.0 KPIs
    # ------------------------------------------------------------------
    def _overview(self):
        # Keep the overview focused on the data; the page title and global filters
        # already identify the section, so no second Consolidated View hero card.
        # Keep the heading immediately below the date filters. The empty-data
        # notice is created after the KPI summary, not above the heading.
        self._title(self.content,'Operational snapshot',
                    'WhatsApp attendance and safety indicators, alongside independently entered staffing-change counts.',
                    top_padding=5)
        row1=ctk.CTkFrame(self.content,fg_color='transparent');row1.pack(fill='x',pady=(0,9))
        self._live_widgets['ov_reports']=self._kpi(row1,'Captured reports','-','WhatsApp records',BLUE)
        self._live_widgets['ov_teams']=self._kpi(row1,'Groups reporting','-','Unique groups in scope',TEAL)
        self._live_widgets['ov_near']=self._kpi(row1,'Near misses','-','Reported message totals',RED)
        self._live_widgets['ov_late']=self._kpi(row1,'Late reported','-','Reported attendance exceptions',AMBER)
        row2=ctk.CTkFrame(self.content,fg_color='transparent');row2.pack(fill='x',pady=(0,8))
        self._live_widgets['ov_before']=self._kpi(row2,'Changes before 15:00','-','Known operator entries',BLUE)
        self._live_widgets['ov_after']=self._kpi(row2,'Changes after 15:00','-','Known operator entries',TEAL)
        self._live_widgets['ov_no']=self._kpi(row2,'No shows','-','Explicit WhatsApp count only',AMBER)
        self._live_widgets['ov_short']=self._kpi(row2,'Staff shortages','-','Actual attendance below plan',RED)
        self._live_widgets['ov_status']=self.label(self.content,'Loading operational summary...',11,MUTED)
        self._live_widgets['ov_status'].pack(anchor='w',padx=8,pady=(2,8))
        # Reserve space for an empty/pending notice only when necessary. This
        # maintains a compact gap between filters and Operational snapshot.
        empty_holder=ctk.CTkFrame(self.content,fg_color='transparent')
        empty_holder.pack(fill='x')
        self._live_widgets['ov_empty_holder']=empty_holder

        charts=ctk.CTkFrame(self.content,fg_color='transparent');charts.pack(fill='x',pady=(12,2))
        mix=self.panel(charts);mix.pack(side='left',fill='both',expand=True,padx=(4,8))
        self.label(mix,'WORKFORCE ALLOCATION',13,WHITE,True).pack(anchor='w',padx=16,pady=(13,2))
        self.label(mix,'Reported role totals • shares of attendance observations',10,MUTED).pack(anchor='w',padx=16)
        self._live_widgets['ov_mix']=self._overview_canvas(mix,(5.0,3.3))
        ex=self.panel(charts);ex.pack(side='left',fill='both',expand=True,padx=(5,4))
        self.label(ex,'CHANGES, NO-SHOWS & SHORTAGES',13,WHITE,True).pack(anchor='w',padx=16,pady=(13,2))
        self.label(ex,'Four independent event categories • top eight groups',10,MUTED).pack(anchor='w',padx=16)
        self._live_widgets['ov_ex_chart']=self._overview_canvas(ex,(6.0,3.3))

        strip=ctk.CTkFrame(self.content,fg_color='transparent');strip.pack(fill='x',pady=(10,0))
        chart=self.panel(strip);chart.pack(side='left',fill='both',expand=True,padx=(4,8))
        self.label(chart,'NEAR-MISS REPORTING TREND',13,WHITE,True).pack(anchor='w',padx=16,pady=(12,0))
        self._live_widgets['ov_trend']=self._overview_canvas(chart,(6.0,2.9))
        attention=self.panel(strip);attention.pack(side='left',fill='both',expand=True,padx=(5,4))
        self.label(attention,'MANAGEMENT ATTENTION',13,WHITE,True).pack(anchor='w',padx=16,pady=(13,8))
        self._live_widgets['ov_attention']=[]
        for _ in range(4):
            lab=self.label(attention,'Checking current records…',12,MUTED,wraplength=470,justify='left')
            lab.pack(anchor='w',padx=17,pady=8)
            self._live_widgets['ov_attention'].append(lab)
        self.button(attention,'Review safety alerts',lambda:self._navigate('Safety & Alerts'),width=195).pack(anchor='w',padx=17,pady=(12,16))

        self._title(self.content,'Latest WhatsApp evidence','Double-click an entry to inspect the original captured message. Staffing changes are recorded separately in Exceptions.')
        frame=self.panel(self.content);frame.pack(fill='x',pady=(0,13))
        self._style_table('OverviewRecent',rowheight=34,fontsize=10)
        cols=('Date/time','Group','Sender','People present','Near misses','Late')
        tbl=ttk.Treeview(frame,columns=cols,show='headings',height=7,style='OverviewRecent.Treeview')
        for col in cols:
            tbl.heading(col,text=col)
            tbl.column(col,anchor='w',width=185 if col in ('Date/time','Group','Sender') else 115)
        tbl.grid(row=0,column=0,sticky='nsew',padx=(12,0),pady=(12,0))
        frame.grid_columnconfigure(0,weight=1)
        bar=ttk.Scrollbar(frame,orient='vertical',command=tbl.yview,style='Ops.Vertical.TScrollbar')
        bar.grid(row=0,column=1,sticky='ns',padx=(0,12),pady=(12,0))
        scroll=ttk.Scrollbar(frame,orient='horizontal',command=tbl.xview,style='Ops.Horizontal.TScrollbar')
        scroll.grid(row=1,column=0,sticky='ew',padx=(12,0),pady=(0,12))
        tbl.configure(yscrollcommand=bar.set,xscrollcommand=scroll.set)
        self._live_widgets['ov_recent']=tbl
        self._live_widgets['ov_mapping']={}
        def show_evidence(_):
            key=tbl.focus()
            item=self._live_widgets.get('ov_mapping',{}).get(key)
            if item:
                messagebox.showinfo('Original WhatsApp evidence',
                    f"Time: {item['event_at']}\nGroup: {item['group_name']}\nSender: {item.get('sender') or '-'}\n\n{item.get('message') or ''}")
        tbl.bind('<Double-1>',show_evidence)
        self._update_overview_in_place()

    def _overview_canvas(self,parent,dimensions):
        fig=Figure(figsize=dimensions,dpi=85,facecolor=PANEL)
        ax=fig.add_subplot(111)
        ax.set_facecolor(PANEL)
        canvas=FigureCanvasTkAgg(fig,master=parent)
        canvas.get_tk_widget().pack(fill='both',expand=True,padx=12,pady=(3,12))
        return fig,ax,canvas

    def _draw_overview_workforce(self,ax,rows):
        ax.clear();ax.set_facecolor(PANEL)
        values=core.workforce_totals(rows)
        core.original_style_pie(ax,['T002/T003','PWT','SPL','PL'],values,
                                ['#38BDF8','#2DD4BF','#A78BFA','#F4B35A'],
                                dark=self.theme_mode=='Dark')

    def _draw_overview_exceptions(self,ax,entries):
        ax.clear();ax.set_facecolor(PANEL)
        groups=core.workforce_exception_chart_rows(entries,limit=8)
        if not groups:
            ax.text(.5,.5,'No staffing exceptions in this selection',ha='center',va='center',
                    color=MUTED,transform=ax.transAxes,fontsize=11)
            ax.set_xticks([]);ax.set_yticks([])
            return
        y=list(range(len(groups)))
        left=[0]*len(y)
        for key,title,color in [('before','Before 15:00','#2980B9'),('after','After 15:00','#16A085'),
                                ('no_shows','No shows','#9B59B6'),('shortage','Shortages','#E74C3C')]:
            values=[g[key] for g in groups]
            ax.barh(y,values,left=left,height=.61,label=title,color=color)
            left=[a+b for a,b in zip(left,values)]
        ax.set_yticks(y,labels=[g['group'] for g in groups])
        ax.invert_yaxis();ax.tick_params(colors=WHITE,labelsize=8)
        ax.xaxis.grid(True,color=BORDER,alpha=.35);ax.set_axisbelow(True)
        ax.legend(loc='lower center',bbox_to_anchor=(.5,1.01),ncol=2,
                  fontsize=8,frameon=False,labelcolor=WHITE)
        for edge in ('top','right'):ax.spines[edge].set_visible(False)
        ax.spines['left'].set_color(BORDER);ax.spines['bottom'].set_color(BORDER)
        ax.set_xlabel('Event totals (not unique people)',color=MUTED,fontsize=8)

    def _update_overview_in_place(self):
        if self.page!='Overview' or 'ov_reports' not in self._live_widgets:return
        rows=self._selected_rows()
        self._rows=rows
        a,b=self._date_bounds()
        ex=core.exception_dashboard(a,b,self.group.get())
        summary=core.summary(rows,core.configuration()['groups'])
        empty_holder=self._live_widgets.get('ov_empty_holder')
        if empty_holder is not None and empty_holder.winfo_exists():
            for child in empty_holder.winfo_children():child.destroy()
            if not rows and not ex['rows']:
                self._empty_state(empty_holder,'No operational data for the selected period',
                   'No WhatsApp attendance or saved staffing-change entries match these filters. '
                   'Select another period or group to view operational KPIs.',
                   'Show all dates & groups',self._reset_view_filters,kind='neutral',compact=True)
            elif not rows:
                self._empty_state(empty_holder,'Staffing changes recorded - attendance pending',
                   'Operator changes may have been saved, but no matching final WhatsApp attendance has arrived. '
                   'Shortages cannot be confirmed from change counts alone.',kind='pending',compact=True)
        safety_ok,total,safety_pct=core.safety_completeness(rows)
        self._update_kpi_in_place('ov_reports',f'{len(rows):,}',
                                  f'{safety_ok} report(s) with complete safety fields')
        self._update_kpi_in_place('ov_teams',str(summary['teams']),
                                  f"{len(summary['missing_groups'])} configured groups not represented")
        self._update_kpi_in_place('ov_near',str(summary['near_miss']),'Reported values, not verified distinct incidents')
        self._update_kpi_in_place('ov_late',str(summary['late']),'Reported attendance exceptions')
        self._update_kpi_in_place('ov_before',str(ex['changes_before']) if ex['changes_before_known'] else '-',
                                  f"{ex['changes_before_known']} entry/entries")
        self._update_kpi_in_place('ov_after',str(ex['changes_after']) if ex['changes_after_known'] else '-',
                                  f"{ex['changes_after_known']} entry/entries")
        self._update_kpi_in_place('ov_no',str(ex['no_shows']) if ex['no_shows_known'] else '-',
                                  f"{ex['no_shows_known']} explicit WhatsApp count(s)")
        self._update_kpi_in_place('ov_short',str(ex['shortage']) if ex['verified'] else '-',
                                  f"{ex['verified']} attendance calculation(s); {ex['pending']} pending")
        status=(f"{ex['verified']} staffing records with attendance • {ex['pending']} pending • "
                f"{ex['awaiting_whatsapp']} awaiting WhatsApp • "
                f"Safety fields: {safety_pct:.0f}% complete" if total else
                f"{ex['verified']} staffing records with attendance • {ex['pending']} pending • No WhatsApp reports in this view")
        self._configure_text(self._live_widgets.get('ov_status'),status)
        messages=[(RED,f"{len(core.alerts('Open'))} open alerts (all dates)"),
                  (AMBER,f"{ex['pending']} exception record(s) still awaiting attendance/plan"),
                  (BLUE,f"{len(summary['missing_groups'])} configured group(s) not represented"),
                  (TEAL,f"Last WhatsApp capture: {summary['last_event'][:16] if summary['last_event'] else 'No matching reports'}")]
        for label,(col,txt) in zip(self._live_widgets.get('ov_attention',[]),messages):
            self._configure_text(label,'●  '+txt)
            if label.winfo_exists():label.configure(text_color=col)
        for key,fn,data in [('ov_mix',self._draw_overview_workforce,rows),
                            ('ov_ex_chart',self._draw_overview_exceptions,ex['rows']),
                            ('ov_trend',lambda ax,rs:core.plot_near_miss_calendar(ax,rs,dark=self.theme_mode=='Dark'),rows)]:
            if key in self._live_widgets:
                fig,ax,canvas=self._live_widgets[key]
                fn(ax,data)
                # Some plotting helpers call ax.clear(), restoring Matplotlib's
                # default white axes face colour. Reinstate the active CTk theme
                # after every redraw, including empty charts and live refreshes.
                fig.set_facecolor(PANEL)
                ax.set_facecolor(PANEL)
                fig.tight_layout(pad=1.8)
                if key=='ov_ex_chart':fig.subplots_adjust(left=.24,top=.79,bottom=.20)
                canvas.draw_idle()
        tree=self._live_widgets.get('ov_recent')
        if tree is not None and tree.winfo_exists():
            mapping=self._live_widgets['ov_mapping'];mapping.clear()
            for old in tree.get_children():tree.delete(old)
            for r in rows[:12]:
                role_counts=[r.get(key) for key in ('t002_t003','pwt','spl','pl')]
                headcount=sum(int(v or 0) for v in role_counts) if any(v is not None for v in role_counts) else '-'
                values=(str(r['event_at']).replace('T',' ')[:16],r['group_name'],r.get('sender') or '-',
                        headcount,'-' if r.get('near_miss') is None else r['near_miss'],
                        '-' if r.get('late') is None else r['late'])
                item=tree.insert('','end',values=values);mapping[item]=r
            self._stripe_rows(tree)

    # ------------------------------------------------------------------
    # REPORTING CENTRE - existing PDF/HTML/CSV/XLSX export functions
    # ------------------------------------------------------------------
    def _report_catalog_path(self):
        folder=core.DATA/'reports'
        folder.mkdir(parents=True,exist_ok=True)
        return folder/'report_catalog.json'

    def _register_local_report(self,path):
        """Store display metadata only; never copy or modify exported reports."""
        report=Path(path).expanduser().resolve()
        if not report.is_file():return
        index=self._report_catalog_path()
        try:
            records=json.loads(index.read_text(encoding='utf-8')) if index.exists() else []
            if not isinstance(records,list):records=[]
        except (OSError,ValueError):
            records=[]
        records=[r for r in records if isinstance(r,dict) and r.get('path')!=str(report)]
        records.append({'path':str(report),'created':datetime.now().isoformat(timespec='seconds'),
                        'scope':f'{self.group.get()} / {self.period.get()}'})
        temp=index.with_suffix('.tmp')
        temp.write_text(json.dumps(records[-1000:],indent=2),encoding='utf-8')
        temp.replace(index)

    def _report_catalog(self):
        """Combine built-in automatic report folder with existing local exports."""
        folder=self._report_catalog_path().parent
        try: indexed=json.loads((folder/'report_catalog.json').read_text(encoding='utf-8'))
        except (OSError,ValueError):indexed=[]
        if not isinstance(indexed,list):indexed=[]
        source_paths={}
        for item in indexed:
            if isinstance(item,dict) and isinstance(item.get('path'),str):
                source_paths[item['path']]=item.get('scope','Saved export')
        for ext in ('*.pdf','*.html','*.xlsx','*.csv'):
            for path in folder.glob(ext):source_paths.setdefault(str(path.resolve()),'Automatic / report library')
        found=[]
        for raw_path,scope in source_paths.items():
            try:
                path=Path(raw_path).expanduser().resolve()
                if not path.is_file() or path.suffix.lower() not in ('.pdf','.html','.xlsx','.csv'):continue
                info=path.stat()
                found.append({'path':path,'name':path.name,'type':path.suffix.lower().lstrip('.').upper(),
                              'scope':str(scope),'size':info.st_size,'modified':info.st_mtime})
            except OSError:continue
        return sorted(found,key=lambda item:(-item['modified'],item['name'].lower()))

    @staticmethod
    def _open_local_report(path):
        path=Path(path).expanduser().resolve()
        if not path.is_file():raise FileNotFoundError(f'This report no longer exists: {path}')
        if path.suffix.lower() not in ('.pdf','.html','.csv','.xlsx'):
            raise ValueError('Unsupported report file type.')
        if sys.platform.startswith('win'):
            os.startfile(str(path))
        elif sys.platform=='darwin':
            subprocess.Popen(['open',str(path)])
        else:
            subprocess.Popen(['xdg-open',str(path)])

    def _generate_library_export(self,format_name):
        """Generate an additional copy in the Reports library; use v3.0.0 exports."""
        suffix={'PDF':'pdf','HTML':'html','Excel':'xlsx','CSV':'csv'}[format_name]
        a,b=self._date_bounds()
        group=self.group.get()
        slug=re.sub(r'[^A-Za-z0-9_-]+','_',group).strip('_')[:32] or 'all_groups'
        stamp=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:5]
        output=self._report_catalog_path().parent/f'operational_{slug}_{stamp}.{suffix}'
        self.status_text.set(f'Generating {format_name} for {group}…')
        try:
            rows=self._selected_rows()
            if format_name in ('PDF','HTML'):
                ex=core.exception_dashboard(a,b,group)['rows']
                fn=core.export_pdf if format_name=='PDF' else core.export_interactive_html
                fn(output,rows,scope=f'{group} / {self.period.get()}',exception_rows=ex)
            elif format_name=='Excel':core.export_excel(output,rows)
            else:core.export_csv(output,rows)
            self._register_local_report(output)
            self.status_text.set(f'{format_name} created • {output.name}')
            callback=self._live_widgets.get('report_library_refresh')
            if callback and self.page=='Reports':callback()
            messagebox.showinfo('Report generated',
                f'{format_name} saved in the report library:\n{output}\n\nOpen it using the report table below.')
        except Exception as exc:
            logging.exception('Report generation error')
            messagebox.showerror('Report generation failed',str(exc))

    def _reports_centre(self):
        hero=self.panel(self.content);hero.pack(fill='x',pady=(0,16))
        self.label(hero,'▣  REPORTING CENTRE',12,BLUE,True).pack(anchor='w',padx=20,pady=(17,5))
        self.label(hero,'One place for executive packs and operational extracts',19,WHITE,True).pack(anchor='w',padx=20)
        self.label(hero,'Use the reporting filters above. PDF and HTML include the existing v3.0.0 charts, staffing exceptions and evidence. '
                        'Excel and CSV contain filtered WhatsApp records. Files remain local to this computer.',
                   11,MUTED,wraplength=1020,justify='left').pack(anchor='w',padx=20,pady=(6,17))
        self._title(self.content,'Create a report','Generated reports are saved under data/reports and appear in the library below.')
        row=ctk.CTkFrame(self.content,fg_color='transparent');row.pack(fill='x',pady=(0,12))
        for title,short,description,color in [
            ('Executive PDF','PDF','Board-ready charts and staffing detail',BLUE),
            ('Interactive HTML','HTML','Self-contained interactive charts and filters',TEAL),
            ('Excel extract','Excel','Filtered WhatsApp data',AMBER),
            ('CSV extract','CSV','Filtered WhatsApp data',BLUE)]:
            tile=self.panel(row);tile.pack(side='left',fill='both',expand=True,padx=5)
            self.label(tile,title,14,WHITE,True).pack(anchor='w',padx=14,pady=(13,4))
            self.label(tile,description,10,MUTED,wraplength=205,justify='left').pack(anchor='w',padx=14,pady=(0,10))
            self.button(tile,'Generate '+short,lambda x=short:self._generate_library_export(x),
                        width=145).pack(anchor='w',padx=14,pady=(0,14))
        self._title(self.content,'Report library','Double-click to open a file in its default desktop application. The library also lists previous exports that are still accessible.')
        searchbar=ctk.CTkFrame(self.content,fg_color='transparent');searchbar.pack(fill='x',pady=(2,9))
        search=ctk.StringVar(value='')
        category=ctk.StringVar(value='All formats')
        box=ctk.CTkEntry(searchbar,textvariable=search,width=310,placeholder_text='Search report names or scopes…')
        box.pack(side='left',padx=(0,10))
        option=ctk.CTkOptionMenu(searchbar,values=['All formats','PDF','HTML','XLSX','CSV'],variable=category,
                                 width=135,fg_color=PANEL,text_color=WHITE,
                                 button_color=('#B8D4ED' if self.theme_mode=='Light' else '#254C72'))
        option.pack(side='left')
        self._live_widgets['report_catalog_info']=self.label(searchbar,'',11,MUTED)
        self._live_widgets['report_catalog_info'].pack(side='right',padx=7)
        frame=self.panel(self.content);frame.pack(fill='x',pady=(0,9))
        self._style_table('ReportLibrary',rowheight=35,fontsize=10)
        columns=('file','kind','scope','modified','size')
        table=ttk.Treeview(frame,columns=columns,show='headings',height=12,style='ReportLibrary.Treeview')
        for key,label,width in [('file','File name',290),('kind','Format',80),('scope','Saved scope',240),
                                 ('modified','Last modified',160),('size','Size',100)]:
            table.heading(key,text=label);table.column(key,width=width,minwidth=65,anchor='w')
        table.grid(row=0,column=0,sticky='nsew',padx=(12,0),pady=(12,0));frame.grid_columnconfigure(0,weight=1)
        vs=ttk.Scrollbar(frame,orient='vertical',command=table.yview,style='Ops.Vertical.TScrollbar')
        vs.grid(row=0,column=1,sticky='ns',padx=(0,12),pady=(12,0))
        hs=ttk.Scrollbar(frame,orient='horizontal',command=table.xview,style='Ops.Horizontal.TScrollbar')
        hs.grid(row=1,column=0,sticky='ew',padx=(12,0),pady=(0,12))
        table.configure(yscrollcommand=vs.set,xscrollcommand=hs.set)
        empty_library=ctk.CTkFrame(self.content,fg_color='transparent')
        empty_library.pack(fill='x',pady=(0,4))
        mapping={}
        def selected():
            key=table.focus()
            if not key or key not in mapping:
                messagebox.showinfo('Select a report','Select a report from the library first.')
                return None
            return mapping[key]['path']
        def open_report(event=None):
            path=selected()
            if path is None:return
            try:self._open_local_report(path)
            except Exception as exc:messagebox.showerror('Unable to open report',str(exc))
        def open_folder():
            try:self._open_local_report_folder()
            except Exception as exc:messagebox.showerror('Unable to open report folder',str(exc))
        def refresh(*_):
            query=search.get().strip().casefold()
            wanted=category.get()
            records=self._report_catalog()
            visible=[r for r in records if (wanted=='All formats' or r['type']==wanted)
                     and (not query or query in r['name'].casefold() or query in r['scope'].casefold())]
            for old in table.get_children():table.delete(old)
            mapping.clear()
            for r in visible:
                values=(r['name'],r['type'],r['scope'],
                        datetime.fromtimestamp(r['modified']).strftime('%d %b %Y  %H:%M'),
                        f"{r['size']/1024:,.1f} KB")
                key=table.insert('','end',values=values);mapping[key]=r
            self._stripe_rows(table)
            for child in empty_library.winfo_children():child.destroy()
            if not records:
                self._empty_state(empty_library,'No reports saved yet',
                    'Generate your first executive PDF or interactive HTML above. Files generated later will appear automatically after refresh.',
                    'Generate executive PDF',lambda:self._generate_library_export('PDF'),kind='neutral',compact=True)
            elif not visible:
                self._empty_state(empty_library,'No reports match the current search',
                    'Adjust the search text or format filter; saved reports have not been removed.',
                    'Clear search',lambda:(search.set(''),category.set('All formats'),refresh()),kind='neutral',compact=True)
            self._configure_text(self._live_widgets.get('report_catalog_info'),
                                 f'{len(visible)} shown / {len(records)} available')
            if not records:
                self.status_text.set('No saved reports found. Generate a report above to get started.')
        search.trace_add('write',refresh)
        option.configure(command=lambda _:refresh())
        table.bind('<Double-1>',open_report)
        controls=ctk.CTkFrame(self.content,fg_color='transparent');controls.pack(fill='x',pady=(0,13))
        self.button(controls,'Open selected',open_report,width=145).pack(side='left',padx=(0,8))
        self.button(controls,'Refresh library',refresh,width=145).pack(side='left',padx=(0,8))
        self.button(controls,'Open reports folder',open_folder,width=175).pack(side='left',padx=(0,8))
        def add_existing():
            selected_files=filedialog.askopenfilenames(title='Add existing reports to the local library',
                filetypes=[('Report documents','*.pdf *.html *.xlsx *.csv'),('All files','*.*')])
            if not selected_files:return
            count=0
            for path in selected_files:
                if Path(path).suffix.lower() not in ('.pdf','.html','.xlsx','.csv'):continue
                try:self._register_local_report(path);count+=1
                except Exception as exc:logging.warning('Could not index report %s: %s',path,exc)
            refresh()
            self.status_text.set(f'{count} existing report(s) added to the library')
        self.button(controls,'Add existing files',add_existing,width=155).pack(side='left')
        self.label(controls,'Reports may contain confidential operational information; share only with authorised users.',
                   10,AMBER,wraplength=410).pack(side='right',padx=5)
        self._live_widgets['report_library_refresh']=refresh
        refresh()

    def _open_local_report_folder(self):
        folder=self._report_catalog_path().parent
        if sys.platform.startswith('win'):os.startfile(str(folder))
        elif sys.platform=='darwin':subprocess.Popen(['open',str(folder)])
        else:subprocess.Popen(['xdg-open',str(folder)])

    def _executive(self):
        if not self._rows:
            self._live_widgets['executive_empty']=self._empty_state(self.content,
                'No WhatsApp attendance in this view',
                'No WhatsApp reports match this date and group selection. Existing staffing-change '
                'entries can still be displayed below. Choose a wider period or another group.',
                'Show all dates & groups',self._reset_view_filters,kind='pending',compact=True)
        s=core.summary(self._rows,core.configuration()['groups'])
        prior=core.compare(self._rows,*self._date_bounds())
        with core.connect() as con:
            all_count=con.execute('SELECT COUNT(*) FROM reports').fetchone()[0]
        self._live_widgets['counter']=self.label(self.content,
            f'LIVE DATA  •  {s["reports"]:,} displayed / {all_count:,} in database  •  Period: {self.period.get()}  •  Team: {self.group.get()}',
            12,TEAL if s['reports'] else AMBER)
        self._live_widgets['counter'].pack(anchor='w',pady=(0,14))
        k=ctk.CTkFrame(self.content,fg_color='transparent');k.pack(fill='x')
        note=(f"{prior['reports']:+.1f}% vs previous period" if prior and prior['reports'] is not None else 'Comparison unavailable')
        self._live_widgets['report_kpi']=self._kpi(k,'Captured reports',f"{s['reports']:,}",note)
        self._live_widgets['team_kpi']=self._kpi(k,'Teams reporting',s['teams'],f"{len(s['missing_groups'])} configured team(s) not represented",TEAL)
        self._live_widgets['near_kpi']=self._kpi(k,'Near-miss reported',s['near_miss'],'Unverified message values',RED)
        complete,total,pct=core.safety_completeness(self._rows)
        self._live_widgets['complete_kpi']=self._kpi(k,'Safety data completeness',f'{pct:.1f}%' if pct is not None else '-',f'{complete} of {total} reports have both fields',TEAL)
        second=ctk.CTkFrame(self.content,fg_color='transparent');second.pack(fill='both',pady=(18,0))
        chart=self.panel(second);chart.pack(side='left',fill='both',expand=True,padx=(5,8))
        self.label(chart,'NEAR-MISS REPORTS - DAILY TREND',13,WHITE,True).pack(anchor='w',padx=14,pady=(12,0))
        self._safety_chart(chart,self._rows)
        distribution=self.panel(second,width=290);distribution.pack(side='left',fill='both',padx=5)
        self.label(distribution,'NEAR-MISS SHARE BY GANG',12,WHITE,True).pack(anchor='w',padx=10,pady=(12,2))
        per_gang=core.near_miss_by_group(self._rows)
        self._pie_chart(distribution,list(per_gang),list(per_gang.values()),
                        ['#d35400','#2980b9','#27ae60','#8e44ad','#f39c12'] * max(1,len(per_gang)),key='safety_pie')
        attention=self.panel(second,width=285);attention.pack(side='right',fill='both',padx=5)
        self.label(attention,'MANAGEMENT ATTENTION',13,WHITE,True).pack(anchor='w',padx=15,pady=(13,10))
        issues=[(RED,f"{len(core.alerts('Open'))} open alerts"),
                (AMBER,f"{s['missing_safety']} reports missing safety values" if s['missing_safety'] else 'Safety values complete in received reports'),
                (BLUE,f"{len(s['missing_groups'])} teams without a captured report"),
                (TEAL,f"Last captured: {s['last_event'][:16] if s['last_event'] else 'No records'}")]
        self._live_widgets['issues']=[]
        for color,text in issues:
            lbl=self.label(attention,'●  '+text,12,color,wraplength=260,justify='left')
            lbl.pack(anchor='w',padx=16,pady=9)
            self._live_widgets['issues'].append(lbl)
        self.button(attention,'Review alerts',lambda:self._navigate('Safety & Alerts'),width=220).pack(padx=15,pady=(15,20))
        # Forecast models remain available elsewhere; the Executive tab intentionally
        # contains only actual reports, safety indicators and recent activity.
        self._exception_executive_panel()
        self._report_table(self.content,self._rows[:80],limit=80,title='Recent WhatsApp report evidence')

    def _pie_chart(self,parent,labels,values,colors,key=None):
        fig=Figure(figsize=(3.4,3.2),dpi=85,facecolor=PANEL)
        ax=fig.add_subplot(111);ax.set_facecolor(PANEL)
        core.original_style_pie(ax,labels,values,colors,dark=self.theme_mode == 'Dark')
        ax.set_facecolor(PANEL)
        fig.tight_layout(pad=1.2)
        canvas=FigureCanvasTkAgg(fig,master=parent);canvas.draw()
        canvas.get_tk_widget().pack(fill='both',expand=True,padx=5,pady=5)
        if key:self._live_widgets[key]=(fig,ax,canvas)
        return fig,ax,canvas

    def _safety_chart(self,parent,rows):
        fig=Figure(figsize=(6.5,3.2),dpi=85,facecolor=PANEL)
        ax=fig.add_subplot(111);ax.set_facecolor(PANEL)
        self._draw_safety_bars(ax,rows)
        fig.tight_layout(pad=1.5)
        canvas=FigureCanvasTkAgg(fig,master=parent);canvas.draw()
        canvas.get_tk_widget().pack(fill='both',expand=True,padx=6,pady=6)
        self._live_widgets['safety_chart']=(fig,ax,canvas)

    @staticmethod
    def _draw_safety_bars(ax,rows):
        """Reported dates only: dynamic daily near-miss chart with DD-MM labels."""
        core.plot_near_miss_calendar(ax, rows, dark=True)
        ax.set_facecolor(PANEL)

    def _gang_attendance_chart(self,parent,rows):
        """Horizontal STACKED bars by group, including all monitored groups.

        Each segment represents a reported quantity (T002/T003, PWT, SPL,
        PL or Late), not a unique person count. Late is an attendance
        exception: stacking it enables compact comparison across 40+ gangs,
        but the resulting bar length must NOT be interpreted as headcount.

        The parent is a vertically scrollable CTkScrollableFrame, so the
        canvas height expands with the number of gangs instead of dropping
        groups or making their labels unreadable.
        """
        groups = core.gang_attendance(rows)
        holder = self.panel(parent)
        holder.pack(fill='x', pady=(0, 12))
        self.label(holder, 'GANG-WISE ATTENDANCE - STACKED HORIZONTAL COMPARISON',
                   13, WHITE, True).pack(anchor='w', padx=14, pady=(12, 0))
        self.label(holder,
                   'Segments show reported values; Late is an exception count, not unique headcount.',
                   11, MUTED).pack(anchor='w', padx=14, pady=(2, 8))
        names = sorted(groups, key=str.casefold)
        if not names:
            self.label(holder,'○  No gang attendance received for these filters',
                       13,WHITE,True).pack(anchor='w',padx=16,pady=(8,5))
            self.label(holder,'Attendance comparisons will appear when a matching WhatsApp report is captured.',
                       11,MUTED).pack(anchor='w',padx=16,pady=(0,18))
            return
        # Give every gang a readable row while letting the page scroll vertically.
        # The width is sized *by Tk's actual panel width*, not a hard-coded 10-inch
        # Matplotlib figure. This prevents clipping and unused space on resize.
        row_pixels = 28
        height_px = max(340, 145 + row_pixels * max(1, len(names)))
        fig = Figure(figsize=(9, height_px / 100), dpi=100, facecolor=PANEL)
        ax = fig.add_subplot(111)
        ax.set_facecolor(PANEL)
        fields = ['t002_t003', 'pwt', 'spl', 'pl', 'late']
        labels = ['T002/T003', 'PWT', 'SPL', 'PL', 'Late']
        colors = ['#2980b9', '#2ecc71', '#e67e22', '#e74c3c', '#f1c40f']
        if names:
            y_positions = list(range(len(names)))
            left = [0] * len(names)
            for field, label, color in zip(fields, labels, colors):
                values = [groups[name][field] for name in names]
                ax.barh(y_positions, values, left=left, height=0.68,
                        label=label, color=color)
                left = [a + b for a, b in zip(left, values)]
            ax.set_yticks(y_positions)
            ax.set_yticklabels(names)
            ax.invert_yaxis()
            ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.005),
                      facecolor=PANEL, labelcolor=WHITE, fontsize=9,
                      ncol=5, frameon=False)
            ax.set_xlim(left=0)
        else:
            ax.text(.5, .5, 'No attendance reports in selected period',
                    ha='center', va='center', color=MUTED,
                    transform=ax.transAxes)
        ax.tick_params(colors=MUTED, labelsize=9)
        for spine in ax.spines.values():
            spine.set_color(BORDER)
        ax.grid(axis='x', color='#38475C', alpha=.35)
        ax.set_axisbelow(True)
        # Reserve a flexible margin for group names and room for the legend.
        # Axes stay within the figure at any application window width.
        max_name_len = max((len(str(name)) for name in names), default=12)
        label_margin_px = min(230, max(105, max_name_len * 7 + 14))
        chart_height = height_px
        top_margin_px = 75
        bottom_margin_px = 38
        canvas = FigureCanvasTkAgg(fig, master=holder)
        chart_widget = canvas.get_tk_widget()
        chart_widget.pack(fill='x', expand=False, padx=12, pady=(3, 12))

        resize_job = [None]
        previous_width = [0]

        def fit_chart_to_panel(width):
            """Resize the existing figure rather than destroying the canvas."""
            if not chart_widget.winfo_exists():
                return
            available_width = max(320, int(width))
            if abs(available_width - previous_width[0]) < 3:
                return
            previous_width[0] = available_width
            dpi = fig.get_dpi()
            fig.set_size_inches(available_width / dpi, chart_height / dpi,
                                forward=False)
            # Pixel-based margins retain predictable label/legend clearance.
            fig.subplots_adjust(
                left=min(.42, label_margin_px / available_width),
                right=.985,
                bottom=bottom_margin_px / chart_height,
                top=1 - top_margin_px / chart_height,
            )
            chart_widget.configure(width=available_width, height=chart_height)
            canvas.draw_idle()

        def on_panel_resize(event):
            # Coalesce bursts of Configure events; never redraw the whole page.
            if resize_job[0] is not None:
                holder.after_cancel(resize_job[0])
            width = max(320, event.width - 24)
            resize_job[0] = holder.after(60, lambda w=width: fit_chart_to_panel(w))

        holder.bind('<Configure>', on_panel_resize, add='+')
        # First paint uses Tk's real width, available once layout has settled.
        holder.after_idle(lambda: fit_chart_to_panel(holder.winfo_width() - 24))

    def _interactive_workforce_analysis(self, parent, rows):
        """Optional analytical drill-down below the existing attendance chart.

        This panel is intentionally self-contained: it reads the already filtered
        SQLite result rows and does not modify WhatsApp collection, saved data,
        the existing stacked chart, or the workforce allocation pie chart.
        Chart updates reuse the same Matplotlib canvas, avoiding screen flicker.
        """
        holder = self.panel(parent)
        holder.pack(fill='x', pady=(8, 12))
        self.label(holder, 'INTERACTIVE WORKFORCE ANALYSIS', 13, WHITE, True).pack(
            anchor='w', padx=14, pady=(13, 2))
        self.label(holder, 'Explore reported values by gang. Near misses and late reports are indicators, not unique workforce headcount.',
                   11, MUTED, wraplength=960, justify='left').pack(
            anchor='w', padx=14, pady=(0, 11))
        if not rows:
            self.label(holder,'○  No reports available for drill-down',13,WHITE,True).pack(
                anchor='w',padx=16,pady=(4,4))
            self.label(holder,'Change reporting filters or import attendance before using interactive analysis.',
                       11,MUTED).pack(anchor='w',padx=16,pady=(0,18))
            return

        # Controls affect only the analysis below; global date/team filters above
        # still determine the eligible records used throughout the application.
        controls = ctk.CTkFrame(holder, fg_color='transparent')
        controls.pack(fill='x', padx=14, pady=(0, 6))
        metric_options = {
            'Workforce Total': 'workforce',
            'T002/T003': 't002_t003',
            'PWT': 'pwt', 'SPL': 'spl', 'PL': 'pl',
            'Late Reported': 'late',
            'Near Misses': 'near_miss',
        }
        metric_var = ctk.StringVar(value='Workforce Total')
        gang_var = ctk.StringVar(value='All Gangs')
        sort_var = ctk.StringVar(value='Highest first')
        view_var = ctk.StringVar(value='Top 10')
        plot_state = {'rows': rows}

        for title, variable, options, width in (
            ('Analyse', metric_var, list(metric_options), 170),
            ('Gang', gang_var, ['All Gangs'], 195),
            ('Sort', sort_var, ['Highest first', 'Lowest first'], 130),
            ('Show', view_var, ['Top 10', 'Bottom 10', 'All Gangs'], 125),
        ):
            cell = ctk.CTkFrame(controls, fg_color='transparent')
            cell.pack(side='left', padx=(0, 12))
            self.label(cell, title, 11, MUTED).pack(anchor='w', pady=(0, 3))
            menu = ctk.CTkOptionMenu(
                cell, variable=variable, values=options, width=width,
                fg_color=PANEL, text_color=WHITE,
                button_color=('#B8D4ED' if self.theme_mode == 'Light' else '#254C72'))
            menu.pack()
            if title == 'Gang':
                gang_menu = menu

        status = self.label(holder, '', 11, MUTED)
        status.pack(anchor='w', padx=16, pady=(4, 3))
        fig = Figure(figsize=(8.8, 3.9), dpi=90, facecolor=PANEL)
        ax = fig.add_subplot(111)
        canvas = FigureCanvasTkAgg(fig, master=holder)
        chart_widget = canvas.get_tk_widget()
        chart_widget.pack(fill='x', expand=False, padx=12, pady=(2, 12))

        # IMPORTANT: FigureCanvasTkAgg automatically matches figure dimensions
        # to its Tk canvas when <Configure> fires. The old code overwrote the
        # figure's width with a constant 8.8 inches on EACH filter update,
        # even though Tk stretched the widget across the full panel. That
        # mismatch made the figure image repeat side by side (two apparent
        # graphs). We keep ONE figure, ONE axes and ONE Tk canvas throughout.
        def keep_chart_width_in_sync(event):
            if event.width < 100 or event.height < 100:
                return
            # Matplotlib's own Tk resize callback runs first and sets its
            # bitmap to the current widget dimensions. Adjust margins only.
            label_pixels = plot_state.get('label_margin_pixels', 140)
            left = min(.42, max(.105, label_pixels / event.width))
            fig.subplots_adjust(
                left=left, right=.985, top=.965,
                bottom=max(.065, min(.17, 34 / event.height)),
            )
            canvas.draw_idle()

        chart_widget.bind('<Configure>', keep_chart_width_in_sync, add='+')
        colors = {'workforce': BLUE, 'late': '#D5A33B', 'near_miss': '#D35400',
                  't002_t003': '#2980b9', 'pwt': '#2ecc71',
                  'spl': '#e67e22', 'pl': '#e74c3c'}

        def refresh_chart(*_):
            # Missing numeric values are not treated as validated zeros by the
            # parser. This chart sums only present numerical report values.
            aggregate = {}
            for report in plot_state['rows']:
                name = report.get('group_name') or 'Unknown gang'
                sums = aggregate.setdefault(name, {name: 0 for name in metric_options.values()})
                for field in ('t002_t003', 'pwt', 'spl', 'pl', 'late', 'near_miss'):
                    value = report.get(field)
                    if isinstance(value, (float, int)):
                        sums[field] += value
                sums['workforce'] = sum(sums[field] for field in ('t002_t003', 'pwt', 'spl', 'pl'))

            choices = ['All Gangs'] + sorted(aggregate, key=str.casefold)
            gang_menu.configure(values=choices)
            if gang_var.get() not in choices:
                gang_var.set('All Gangs')
                return  # StringVar trace runs refresh once with the new choice.
            field = metric_options[metric_var.get()]
            items = [(name, values[field]) for name, values in aggregate.items()
                     if gang_var.get() == 'All Gangs' or name == gang_var.get()]
            items.sort(key=lambda item: (item[1], item[0].casefold()),
                       reverse=sort_var.get() == 'Highest first')
            if gang_var.get() == 'All Gangs':
                if view_var.get() == 'Top 10':
                    items = sorted(items, key=lambda item: (-item[1], item[0].casefold()))[:10]
                elif view_var.get() == 'Bottom 10':
                    items = sorted(items, key=lambda item: (item[1], item[0].casefold()))[:10]
            items.sort(key=lambda item: (item[1], item[0].casefold()),
                       reverse=sort_var.get() == 'Highest first')
            # Adjustable figure height keeps gang labels readable without
            # cropping, including when the user chooses 'All Gangs'.
            # For many gangs, enlarge HEIGHT only. Never reset figure width:
            # Tk owns that dimension and resizing it to 8.8in causes repeated
            # chart images in a wider CustomTkinter panel.
            desired_height = max(310, 105 + 30 * max(1, len(items)))
            plot_state['label_margin_pixels'] = min(
                240, max(110, max((len(name) for name, _ in items), default=12) * 7 + 26)
            )
            ax.clear()
            ax.set_facecolor(PANEL)
            if items:
                names, values = zip(*items)
                ax.barh(range(len(items)), values, color=colors[field], height=.65)
                ax.set_yticks(range(len(items)))
                ax.set_yticklabels(names, fontsize=9)
                ax.invert_yaxis()
                ax.set_xlim(left=0)
                status.configure(text=f'{len(items)} gang(s) shown  •  {metric_var.get()}  •  {sum(values):,.0f} reported units in view')
            else:
                ax.text(.5, .5, 'No matching reports for these filters',
                        transform=ax.transAxes, ha='center', va='center', color=MUTED)
                status.configure(text='No matching gang reports for the selected reporting period.')
            ax.tick_params(axis='both', colors=MUTED, labelsize=9)
            ax.grid(axis='x', color='#38475C', alpha=.3)
            ax.set_axisbelow(True)
            for spine in ax.spines.values(): spine.set_color(BORDER)
            # Tk's Configure event will resize the Matplotlib bitmap to
            # EXACTLY the canvas width, so filtering never tiles a stale image.
            # Reuse the existing canvas instead of creating another one.
            chart_widget.configure(height=desired_height)
            # On the first render Tk may not yet have measured the panel;
            # subsequent Configure events handle sizing without flicker.
            if chart_widget.winfo_width() > 100:
                w = chart_widget.winfo_width()
                fig.subplots_adjust(
                    left=min(.42, max(.105, plot_state['label_margin_pixels'] / w)),
                    right=.985, top=.965,
                    bottom=max(.065, min(.17, 34 / desired_height)),
                )
            canvas.draw_idle()

        # Draw only the analytics panel on interaction; other page components
        # never get destroyed or recreated by these controls.
        for variable in (metric_var, gang_var, sort_var, view_var):
            variable.trace_add('write', refresh_chart)

        def update_from_db():
            if holder.winfo_exists():
                plot_state['rows'] = self._selected_rows()
                refresh_chart()
        self._live_widgets['analysis_refresh'] = update_from_db
        refresh_chart()

    def _workforce_charts(self,parent,rows):
        if not rows:
            self._empty_state(parent,'Workforce charts awaiting attendance',
                'No workforce allocation percentages or role totals can be calculated until '
                'WhatsApp staffing values have been received.',kind='pending',compact=True)
            return
        names=['T002/T003','PWT','SPL','PL'];v=core.workforce_totals(rows)
        colors=['#2980b9','#2ecc71','#e67e22','#e74c3c']
        wrap=ctk.CTkFrame(parent,fg_color='transparent');wrap.pack(fill='x',pady=(8,15))
        a=self.panel(wrap);a.pack(side='left',fill='both',expand=True,padx=5)
        self.label(a,'WORKFORCE VOLUMES + LATE REPORTS',13,WHITE,True).pack(anchor='w',padx=13,pady=(12,2))
        fig=Figure(figsize=(6.1,3.2),dpi=85,facecolor=PANEL);ax=fig.add_subplot(111);ax.set_facecolor(PANEL)
        bar_names=names+['Late']
        bar_values=v+[sum((r['late'] or 0) for r in rows)]
        bars=ax.bar(bar_names,bar_values,color=colors+['#f1c40f'],width=.5)
        for b in bars:
            if b.get_height():ax.annotate(f'{b.get_height():.0f}',(b.get_x()+b.get_width()/2,b.get_height()),
                                          xytext=(0,3),textcoords='offset points',ha='center',color=WHITE,fontsize=8)
        ax.tick_params(colors=MUTED,labelsize=9);ax.grid(axis='y',color='#38475C',alpha=.35)
        for spine in ax.spines.values():spine.set_color(BORDER)
        fig.tight_layout(pad=1.5)
        c=FigureCanvasTkAgg(fig,master=a);c.draw();c.get_tk_widget().pack(fill='both',expand=True,padx=5,pady=5)
        b=self.panel(wrap);b.pack(side='left',fill='both',expand=True,padx=5)
        self.label(b,'WORKFORCE ALLOCATION PERCENTAGE',13,WHITE,True).pack(anchor='w',padx=13,pady=(12,2))
        self._pie_chart(b,names,v,colors)

    def _plot(self,parent,daily,type='line'):
        fig=Figure(figsize=(7,3.2),dpi=90,facecolor=PANEL)
        ax=fig.add_subplot(111);ax.set_facecolor(PANEL)
        keys=list(sorted(daily));values=[daily[k] for k in keys]
        if not keys: ax.text(0.5,0.5,'No reports in this period',color=MUTED,ha='center',va='center',transform=ax.transAxes)
        else:
            if type=='bar':ax.bar(range(len(keys)),values,color=BLUE)
            else:ax.plot(range(len(keys)),values,color=BLUE,linewidth=2.5);ax.fill_between(range(len(keys)),values,color=BLUE,alpha=.12)
            step=max(1,len(keys)//6)
            ax.set_xticks(list(range(0,len(keys),step)))
            ax.set_xticklabels([keys[i][5:] for i in range(0,len(keys),step)],rotation=0)
        ax.tick_params(colors=MUTED,labelsize=9)
        for spine in ax.spines.values():spine.set_color(BORDER)
        ax.grid(axis='y',color='#38475C',alpha=.5)
        fig.tight_layout(pad=2)
        canvas=FigureCanvasTkAgg(fig,master=parent)
        canvas.draw();canvas.get_tk_widget().pack(fill='both',expand=True,padx=8,pady=7)
        self._chart=canvas
        if self.page=='Executive': self._live_widgets['trend_chart']=(fig,ax,canvas)

    def _report_table(self,parent,rows,limit=100,title="Recent reports"):
        """Show raw-message sender plus structured metrics; double-click for evidence."""
        self._title(parent, f'{title} ({min(len(rows),limit)} shown)',
                    'Double-click a row to inspect its original message and parsed values.')
        frame=self.panel(parent);frame.pack(fill='x',pady=(0,16))
        if not rows:
            # The table and scrollbars use grid() within `frame`.  Put the
            # empty-data labels in their own child container so their pack()
            # calls do not mix geometry managers in `frame`.
            empty_holder=ctk.CTkFrame(frame,fg_color='transparent')
            empty_holder.grid(row=2,column=0,columnspan=2,sticky='ew',padx=14,pady=(0,12))
            empty_title=self.label(empty_holder,'○  No report evidence for this selection',13,WHITE,True)
            empty_title.pack(anchor='w',padx=1,pady=(13,3))
            empty_help=self.label(empty_holder,'Records will appear after the next matching WhatsApp message. '
                'Try widening the reporting period or checking collection status.',
                11,MUTED,wraplength=920)
            empty_help.pack(anchor='w',padx=1,pady=(0,8))
            if self.page=='Executive':
                self._live_widgets['report_empty_titles']=(empty_title,empty_help)
        # Sender already exists in SQLite; show it alongside report metrics.
        show_late=self.page in ('Workforce','Safety & Alerts')
        cols=('Date / time','Team','Sender','T002/T003','PWT','SPL','PL','Near Miss') + (('Late',) if show_late else ())
        self._style_table('Exec',rowheight=34,fontsize=10)
        tree=ttk.Treeview(frame,columns=cols,show='headings',height=min(11,max(4,len(rows))),style='Exec.Treeview')
        for c in cols:
            tree.heading(c,text=c)
            tree.column(c,width=175 if c in ('Date / time','Team','Sender') else 103,anchor='w')
        # Separate horizontal/vertical scrolling prevents column clipping on laptops.
        tree.grid(row=0,column=0,sticky='nsew',padx=(14,0),pady=(8,0))
        frame.grid_columnconfigure(0,weight=1)
        scrollbar=ttk.Scrollbar(frame,orient='vertical',command=tree.yview,
                                style='Ops.Vertical.TScrollbar')
        scrollbar.grid(row=0,column=1,sticky='ns',padx=(0,14),pady=(8,0))
        horizontal=ttk.Scrollbar(frame,orient='horizontal',command=tree.xview,
                                 style='Ops.Horizontal.TScrollbar')
        horizontal.grid(row=1,column=0,sticky='ew',padx=(14,0),pady=(0,14))
        tree.configure(yscrollcommand=scrollbar.set,xscrollcommand=horizontal.set)
        mapping={}
        for r in rows[:limit]:
            vals=[r['event_at'].replace('T',' '),r['group_name'],r.get('sender') or '-']+[r[k] if r[k] is not None else '-' for k in (('t002_t003','pwt','spl','pl','near_miss','late') if show_late else ('t002_t003','pwt','spl','pl','near_miss'))]
            item=tree.insert('',tk.END,values=vals);mapping[item]=r
        self._stripe_rows(tree)
        def detail(_):
            key=tree.focus()
            if key and key in mapping:
                r=mapping[key]
                messagebox.showinfo('Report evidence',f"Time: {r['event_at']}\nGroup: {r['group_name']}\nSender: {r['sender']}\nOrigin: {r['origin']}\n\n{r['message']}")
        tree.bind('<Double-1>',detail)
        if self.page=='Executive':
            self._live_widgets['reports_tree']=tree
            self._live_widgets['reports_mapping']=mapping

    def _workforce(self):
        if not self._rows:
            self._empty_state(self.content,'Attendance not available for this selection',
                'No group attendance has been captured for the selected dates. '
                'Charts and staffing volumes require a WhatsApp report; missing data is not zero.',
                'Show all dates & groups',self._reset_view_filters,kind='pending',compact=True)
        s=core.summary(self._rows,core.configuration()['groups'])
        self.label(self.content,'Reporting coverage is message-based, not a verified employee headcount.',12,MUTED).pack(anchor='w',pady=(0,15))
        cards=ctk.CTkFrame(self.content,fg_color='transparent');cards.pack(fill='x')
        for key,name in [('t002_t003','T002/T003'),('pwt','PWT'),('spl','SPL'),('pl','PL')]:
            self._kpi(cards,name,s['staffing'][key],'Sum of reported message values',TEAL)
        # Late remains a KPI on Workforce AND appears on Safety & Alerts.
        # It is an attendance exception; do not include it in allocation pie.
        self._kpi(cards, 'Late reported', s['late'],
                  'Reported attendance exceptions', AMBER)
        # Separate five coverage cards visually from the attendance chart.
        # Use a fixed-height transparent frame so both themes retain the same gap.
        coverage_gap = ctk.CTkFrame(self.content, fg_color='transparent', height=28)
        coverage_gap.pack(fill='x')
        coverage_gap.pack_propagate(False)
        # The old 'number of reports per team' chart was not an attendance
        # measure. Replace it with metric-by-gang grouped bars.
        self._gang_attendance_chart(self.content,self._rows)
        self._interactive_workforce_analysis(self.content,self._rows)
        self._workforce_charts(self.content,self._rows)
        self._title(self.content,'What-if workforce scenario','Enter independent assumptions; figures are not inferred from WhatsApp messages.')
        sc=self.panel(self.content);sc.pack(fill='x',pady=8)
        controls=ctk.CTkFrame(sc,fg_color='transparent');controls.pack(fill='x',padx=14,pady=14)
        vals=[]
        for title,default in [('Required workforce',50),('Scheduled workforce',48),('Absence %',8),('Extra contractors',4)]:
            cell=ctk.CTkFrame(controls,fg_color='transparent');cell.pack(side='left',padx=8)
            self.label(cell,title,11,MUTED).pack(anchor='w')
            ent=ctk.CTkEntry(cell,width=125);ent.insert(0,str(default));ent.pack(pady=5);vals.append(ent)
        result=self.label(sc,'',15,BLUE,True);result.pack(anchor='w',padx=22,pady=(0,16))
        def update(*_):
            try:
                d=core.scenarios(*(float(v.get()) for v in vals))
                result.configure(text=f"Effective supply: {d['effective']:.1f}  |  Gap: {d['gap']:.1f}  |  Surplus: {d['surplus']:.1f}  |  Coverage: {d['coverage']:.1f}%",text_color=RED if d['gap']>0 else TEAL)
            except Exception:result.configure(text='Enter valid non-negative numbers; absence must be 0-100%.',text_color=RED)
        self.button(sc,'Calculate scenario',update,width=180).pack(anchor='w',padx=22,pady=(0,18));update()
        self._report_table(self.content,self._rows,limit=100)

    # ======================================================================
    # OPERATOR EXCEPTIONS  /  STAFFING SHORTAGE CONTROL
    # ======================================================================
    def _show_recovery_code(self, username, code):
        """Display a newly generated single-use offline recovery code exactly once."""
        dialog=ctk.CTkToplevel(self)
        dialog.title('Save account recovery code')
        dialog.geometry('620x300');dialog.resizable(False,False)
        dialog.transient(self);dialog.grab_set()
        self.label(dialog,'Save this recovery code securely',17,WHITE,True).pack(anchor='w',padx=20,pady=(20,6))
        self.label(dialog,f'Account: {username}  •  This code is shown once.',12,AMBER).pack(anchor='w',padx=20)
        code_view=ctk.CTkEntry(dialog,width=565,font=ctk.CTkFont(family='Consolas',size=16))
        code_view.pack(padx=20,pady=(16,7));code_view.insert(0,code)
        code_view.configure(state='readonly')
        self.label(dialog,'Store the code outside the application. Anyone who has it can reset this password. '
                   'There is no email recovery because this app runs locally.',11,MUTED,
                   wraplength=565,justify='left').pack(anchor='w',padx=20,pady=(4,12))
        buttons=ctk.CTkFrame(dialog,fg_color='transparent');buttons.pack(fill='x',padx=20)
        def copy_code():
            dialog.clipboard_clear();dialog.clipboard_append(code)
            messagebox.showinfo('Copied','Recovery code copied. Clear your clipboard after storing it securely.',parent=dialog)
        self.button(buttons,'Copy code',copy_code,width=130).pack(side='left')
        self.button(buttons,'I have saved it',dialog.destroy,color='#166653',width=160).pack(side='right')
        dialog.focus_force()

    def _settings_switch_account(self):
        """Sign out of a non-admin account before administrator login."""
        if self.exception_session:
            core.revoke_app_session(self.exception_session['token'])
        self.exception_session=None
        self._render()

    def _exception_login(self):
        heading = ('Settings - Administrator Sign-in' if self.page=='Settings'
                   else 'Staffing Changes - Account Access')
        self._title(self.content,heading,
                    'Individual usernames, passwords and a private audit history for each user.')
        card=self.panel(self.content);card.pack(fill='x',pady=10)
        if not core.app_users_ready():
            self.label(card,'INITIAL ADMINISTRATOR SETUP',13,TEAL,True).pack(anchor='w',padx=18,pady=(16,5))
            self.label(card,'Create the first administrator. Additional staff accounts can then be added by this administrator.',
                       11,MUTED,wraplength=890,justify='left').pack(anchor='w',padx=18)
            cols=ctk.CTkFrame(card,fg_color='transparent');cols.pack(anchor='w',padx=15,pady=10)
            username=ctk.CTkEntry(cols,width=165,placeholder_text='Admin username')
            fullname=ctk.CTkEntry(cols,width=210,placeholder_text='Full name')
            password=ctk.CTkEntry(cols,width=185,placeholder_text='Password (12+)',show='•')
            confirm=ctk.CTkEntry(cols,width=185,placeholder_text='Confirm password',show='•')
            for w in (username,fullname,password,confirm):w.pack(side='left',padx=4)
            legacy=None
            if core.legacy_operator_migration_required():
                self.label(card,'Existing legacy database detected: supply the old Operator 1 passcode to authorise this upgrade.',
                           11,AMBER,wraplength=890,justify='left').pack(anchor='w',padx=19)
                legacy=ctk.CTkEntry(card,width=275,placeholder_text='Previous Operator 1 passcode',show='•')
                legacy.pack(anchor='w',padx=19,pady=9)
            def setup():
                try:
                    if password.get()!=confirm.get():raise ValueError('Passwords do not match.')
                    code=core.bootstrap_admin_account(username.get(),fullname.get(),password.get(),
                                                       legacy.get() if legacy else None)
                    self._show_recovery_code(username.get(),code)
                    self._render()
                except Exception as exc:messagebox.showerror('Unable to create administrator',str(exc))
            self.button(card,'Create administrator',setup,color='#166653',width=190).pack(anchor='w',padx=19,pady=(3,18))
            return
        self.label(card,'SIGN IN TO RECORD STAFFING CHANGES',14,WHITE,True).pack(anchor='w',padx=18,pady=(17,8))
        controls=ctk.CTkFrame(card,fg_color='transparent');controls.pack(anchor='w',padx=13,pady=(0,14))
        username=ctk.CTkEntry(controls,width=235,placeholder_text='Username')
        password=ctk.CTkEntry(controls,width=235,placeholder_text='Password',show='•')
        username.pack(side='left',padx=5);password.pack(side='left',padx=5)
        def login():
            try:
                self.exception_session=core.authenticate_app_user(username.get(),password.get())
                password.delete(0,'end')
                self._render()
            except Exception as exc:messagebox.showerror('Sign-in failed',str(exc))
        self.button(controls,'Sign in',login,color='#166653',width=125).pack(side='left',padx=6)
        password.bind('<Return>',lambda _:login())
        self.button(card,'Forgot password? Use recovery code',self._exception_recovery,
                    color='transparent',width=290).pack(anchor='w',padx=16,pady=(0,15))
        self.label(card,'Forgot both password and recovery code? Ask an administrator to generate a new recovery code. '
                   'Accounts are local to this database.',11,MUTED,wraplength=900,justify='left').pack(anchor='w',padx=18,pady=(0,16))

    def _exception_recovery(self):
        win=ctk.CTkToplevel(self);win.title('Offline password recovery');win.geometry('550x385')
        win.transient(self);win.grab_set()
        self.label(win,'Reset your password',18,WHITE,True).pack(anchor='w',padx=22,pady=(20,10))
        fields=[]
        for hint,masked in [('Username',False),('Recovery code',False),('New password (12+)',True),('Confirm password',True)]:
            entry=ctk.CTkEntry(win,width=490,placeholder_text=hint,show='•' if masked else '')
            entry.pack(padx=22,pady=6);fields.append(entry)
        def reset():
            try:
                if fields[2].get()!=fields[3].get():raise ValueError('New passwords do not match.')
                next_code=core.recover_app_password(fields[0].get(),fields[1].get(),fields[2].get())
                user=fields[0].get().strip()
                win.destroy()
                self._show_recovery_code(user,next_code)
                messagebox.showinfo('Password updated','Use your new password to sign in.')
            except Exception as exc:messagebox.showerror('Recovery unsuccessful',str(exc),parent=win)
        self.button(win,'Reset password',reset,width=185,color='#166653').pack(pady=(16,9))

    def _exception_manage_users(self, parent=None):
        """Admin-only account management shown within the Settings section."""
        session=self.exception_session
        if not session or session.get('role')!='admin':return
        p=self.panel(parent if parent is not None else self.content)
        p.pack(fill='x',pady=(7,12))
        self.label(p,'ACCOUNT MANAGEMENT / ADMINISTRATOR',13,WHITE,True).pack(anchor='w',padx=15,pady=(13,7))
        self.label(p,'Create staff logins, disable departed users or issue a fresh recovery code. '
                   'Old codes stop working after rotation.',11,MUTED,wraplength=970,justify='left').pack(anchor='w',padx=15)
        form=ctk.CTkFrame(p,fg_color='transparent');form.pack(fill='x',padx=12,pady=10)
        name=ctk.CTkEntry(form,width=140,placeholder_text='New username')
        display=ctk.CTkEntry(form,width=180,placeholder_text='Display name')
        role=ctk.StringVar(value='operator')
        role_pick=ctk.CTkOptionMenu(form,values=['operator','admin'],variable=role,width=115)
        temp_pwd=ctk.CTkEntry(form,width=200,placeholder_text='Initial password (12+)',show='•')
        for w in (name,display,role_pick,temp_pwd):w.pack(side='left',padx=3)
        def add_user():
            try:
                code=core.create_app_user(session['token'],name.get(),display.get(),role.get(),temp_pwd.get())
                target=name.get().strip()
                temp_pwd.delete(0,'end')
                self._show_recovery_code(target,code)
                self._render()
            except Exception as exc:messagebox.showerror('Cannot create user',str(exc))
        self.button(form,'Add user',add_user,width=100,color='#166653').pack(side='left',padx=5)
        users=core.list_app_users()
        self._style_table('Users',rowheight=35,fontsize=10)
        tree=ttk.Treeview(p,columns=('user','name','role','state','login'),show='headings',
                          height=min(6,len(users)+1),style='Users.Treeview')
        for column,heading,w in [('user','Username',150),('name','Name',200),('role','Role',125),
                                  ('state','Access',100),('login','Last sign-in',170)]:
            tree.heading(column,text=heading);tree.column(column,width=w,minwidth=85,anchor='w')
        for user in users:
            tree.insert('','end',iid=user['username'],values=(user['username'],user['display_name'],
                user['account_role'],'Enabled' if user['enabled'] else 'Disabled',
                (user['last_login'] or 'Never')[:19].replace('T',' ')))
        self._stripe_rows(tree)
        tree.pack(fill='x',padx=15,pady=(2,10))
        actions=ctk.CTkFrame(p,fg_color='transparent');actions.pack(fill='x',padx=14,pady=(0,15))
        def selected_user():
            ids=tree.selection()
            if not ids:raise ValueError('Select a user from the table.')
            return ids[0]
        def rotate():
            try:
                user=selected_user()
                if not messagebox.askyesno('Replace recovery code','Issue a new recovery code for '+user+'? The old code will become invalid.'):
                    return
                code=core.rotate_user_recovery_code(session['token'],user)
                self._show_recovery_code(user,code)
            except Exception as exc:messagebox.showerror('Recovery reset failed',str(exc))
        def toggle():
            try:
                user=selected_user()
                state=next(u['enabled'] for u in users if u['username']==user)
                core.set_app_user_enabled(session['token'],user,not state)
                self._render()
            except Exception as exc:messagebox.showerror('Account update failed',str(exc))
        self.button(actions,'Issue new recovery code',rotate,width=215).pack(side='left',padx=4)
        self.button(actions,'Enable / Disable selected',toggle,width=195,color='#69424D').pack(side='left',padx=4)

    def _exceptions(self):
        self.label(self.content,'STAFFING CHANGES / DAILY CONTROL',12,TEAL,True).pack(anchor='w',pady=(0,6))
        self.label(self.content,'One service date, every active group. Double-click a count to edit, press Enter to move to the '
                   'next group, then Save All. Actual WhatsApp attendance drives automatic shortages; '
                   'before/after change counts do not reduce that headcount again.',
                   11,MUTED,wraplength=1030,justify='left').pack(anchor='w',pady=(0,11))
        if self.exception_session is None:
            self._exception_login();return
        session=self.exception_session
        actor=session['username']
        top=ctk.CTkFrame(self.content,fg_color='transparent');top.pack(fill='x',pady=(0,11))
        self.label(top,'●  '+session['display_name']+' ('+actor+')  •  '+session['role'].title(),13,TEAL,True).pack(side='left')
        def logout():
            if self._exception_draft_dirty and not messagebox.askyesno('Unsaved entries',
                     'Unsaved staffing changes will be lost. Sign out anyway?'):
                return
            self._exception_draft_dirty=False
            core.revoke_app_session(session['token'])
            self.exception_session=None;self._render()
        self.button(top,'Sign out',logout,width=105,color='#69424D').pack(side='right')
        date_panel=self.panel(self.content);date_panel.pack(fill='x',pady=(0,12))
        datebar=ctk.CTkFrame(date_panel,fg_color='transparent');datebar.pack(fill='x',padx=14,pady=13)
        self.label(datebar,'Night-shift service date',12,WHITE,True).pack(side='left',padx=(0,9))
        ctk.CTkEntry(datebar,textvariable=self.exception_day,width=155).pack(side='left',padx=(0,10))
        def set_previous():
            if self._exception_draft_dirty and not messagebox.askyesno('Unsaved entries',
                 'Discard unsaved changes and load the previous night?'):
                return
            self._exception_draft_dirty=False
            self._exception_started_date=None
            self.exception_day.set((date.today()-timedelta(days=1)).isoformat());self._render()
        self.button(datebar,'Previous night',set_previous,width=125).pack(side='left',padx=(0,8))
        self.button(datebar,'Load / Refresh',self._exception_load,width=135).pack(side='left')
        self.label(datebar,'One record per group and service date.',11,MUTED).pack(side='right',padx=9)
        try: day=core.parse_date(self.exception_day.get()).isoformat()
        except Exception as exc:
            self.label(self.content,'Invalid service date: '+str(exc),12,RED).pack(anchor='w');return
        if not core.service_date_has_records(day) and getattr(self,'_exception_started_date',None)!=day:
            self._exception_draft_dirty=False
            def start_new_date():
                self._exception_started_date=day
                self._render()
            self._empty_state(self.content,'No records for '+day,
                'There are no saved staffing changes or final WhatsApp attendance reports for this '
                'night. This is an unstarted date, not a zero-shortage result. '
                'Use the action below if you need to enter change counts before attendance arrives.',
                'Start entries for this date',start_new_date,kind='pending')
            self._exception_recent_table()
            return
        active=sorted(core.active_group_names(core.group_registry()),key=str.casefold)
        if not active:
            self._empty_state(self.content,'No active WhatsApp groups',
                'The rapid-entry grid will appear after an administrator adds and enables groups in Settings.',
                kind='pending')
            return
        # Read the entire day's 40+ groups once; no selection dropdown.
        loaded={}
        for g in active:
            record=core.exception_entry(g,day,'Shift 1')
            computed=core.calculate_exception(g,day,'Shift 1')
            loaded[g]={'revision':record['revision'] if record else 0,
                       'before':record['before_count'] if record else None,
                       'after':record['after_count'] if record else None,
                       'computed':computed}
        done_before=sum(v['before'] is not None for v in loaded.values())
        done_after=sum(v['after'] is not None for v in loaded.values())
        attendance=sum(v['computed']['shortage'] is not None for v in loaded.values())
        short_count=sum(v['computed']['shortage'] or 0 for v in loaded.values())
        cards=ctk.CTkFrame(self.content,fg_color='transparent');cards.pack(fill='x',pady=(0,11))
        self._kpi(cards,'Before 15:00',f'{done_before}/{len(active)}','Groups submitted',TEAL)
        self._kpi(cards,'After 15:00',f'{done_after}/{len(active)}','Groups submitted',BLUE)
        self._kpi(cards,'WhatsApp actual',f'{attendance}/{len(active)}','Groups with final report',AMBER)
        self._kpi(cards,'Staff short',str(short_count),'Role-specific shortages',RED)
        workstation=self.panel(self.content);workstation.pack(fill='x',pady=(0,11))
        self.label(workstation,'ALL ACTIVE GROUPS  /  RAPID ENTRY',14,WHITE,True).pack(anchor='w',padx=15,pady=(15,6))
        filters=ctk.CTkFrame(workstation,fg_color='transparent');filters.pack(fill='x',padx=12,pady=(3,11))
        query=ctk.StringVar(value='');state=ctk.StringVar(value='All groups')
        search=ctk.CTkEntry(filters,textvariable=query,width=320,placeholder_text='Search any of the 40+ groups')
        search.pack(side='left',padx=4)
        choice=ctk.CTkOptionMenu(filters,variable=state,width=210,
               values=['All groups','Needs change entries','Awaiting WhatsApp','Staff shortages','Complete'])
        choice.pack(side='left',padx=8)
        self.label(filters,'Double-click Before or After values to edit',11,MUTED).pack(side='right',padx=7)
        frame=ctk.CTkFrame(workstation,fg_color=self._table_colors()['border'],
                           corner_radius=10,border_width=1,
                           border_color=self._table_colors()['border'])
        frame.pack(fill='x',padx=14,pady=(0,10))
        self._style_table('ExceptionEntry',rowheight=40,fontsize=11)
        columns=('name','before','after','attendance','shortage','state')
        table=ttk.Treeview(frame,columns=columns,show='headings',height=12,selectmode='browse',
                          style='ExceptionEntry.Treeview')
        for col,heading,width in [('name','WhatsApp group',235),('before','Before 15:00',137),
               ('after','After 15:00',137),('attendance','Actual attendance',150),
               ('shortage','Shortage',110),('state','Workflow status',225)]:
            table.heading(col,text=heading);table.column(col,width=width,minwidth=95,anchor='center' if col!='name' else 'w')
        scroll=ttk.Scrollbar(frame,orient='vertical',command=table.yview,
                             style='Ops.Vertical.TScrollbar')
        frame.grid_columnconfigure(0,weight=1)
        table.configure(yscrollcommand=scroll.set)
        table.grid(row=0,column=0,sticky='nsew',padx=(6,0),pady=(6,0))
        scroll.grid(row=0,column=1,sticky='ns',padx=(0,5),pady=(6,0))
        horizontal=ttk.Scrollbar(frame,orient='horizontal',command=table.xview,
                                 style='Ops.Horizontal.TScrollbar')
        horizontal.grid(row=1,column=0,sticky='ew',padx=(6,0),pady=(0,6))
        table.configure(xscrollcommand=horizontal.set)
        table.tag_configure('short',foreground=RED)
        table.tag_configure('waiting',foreground=AMBER)
        table.tag_configure('ready',foreground=TEAL)
        # Drafts live in memory until Save All. No background redraw discards keystrokes.
        drafts={g:{} for g in active}
        self._exception_draft_dirty=False
        selected={'group':active[0]}
        def cell_value(g,stage):
            value=drafts[g].get(stage,loaded[g][stage])
            return '-' if value is None else str(value)
        def status(g):
            v=loaded[g];cal=v['computed']
            before=drafts[g].get('before',v['before'])
            after=drafts[g].get('after',v['after'])
            if cal['shortage'] is None:
                return 'Awaiting WhatsApp'
            if before is None or after is None:
                return 'Attendance / changes pending'
            return 'Complete'
        unsaved_message=ctk.StringVar(value='No unsaved entries')
        def draw_grid():
            pending_count=sum(bool(drafts[g]) for g in drafts)
            self._exception_draft_dirty=bool(pending_count)
            unsaved_message.set(f'{pending_count} group(s) with unsaved edits' if pending_count else 'All entered values saved')
            active_selection=selected['group']
            for item in table.get_children():table.delete(item)
            term=query.get().strip().casefold();flt=state.get()
            for g in active:
                v=loaded[g];comp=v['computed'];s=status(g)
                if term not in g.casefold():continue
                if flt=='Needs change entries' and v['before'] is not None and v['after'] is not None:continue
                if flt=='Awaiting WhatsApp' and comp['shortage'] is not None:continue
                if flt=='Staff shortages' and not (comp['shortage'] is not None and comp['shortage']>0):continue
                if flt=='Complete' and s!='Complete':continue
                short='Pending' if comp['shortage'] is None else str(comp['shortage'])
                attendance='Pending' if comp['available'] is None else str(comp['available'])
                tag='short' if comp['shortage'] else ('waiting' if comp['shortage'] is None else 'ready')
                table.insert('','end',iid=g,values=(g,cell_value(g,'before'),cell_value(g,'after'),
                             attendance,short,s+('  •  Unsaved' if drafts[g] else '')),tags=(tag,))
            self._stripe_rows(table)
            if table.exists(active_selection):table.selection_set(active_selection)
            elif table.get_children():
                first=table.get_children()[0];selected['group']=first;table.selection_set(first)
            update_detail()
        detail=self.panel(self.content);detail.pack(fill='x',pady=(0,11))
        def update_detail(_=None):
            for w in detail.winfo_children():w.destroy()
            if not table.get_children():
                self.label(detail,'○  No groups match the current search or status filter',
                           13,WHITE,True).pack(anchor='w',padx=16,pady=(15,5))
                self.label(detail,'Clear the search or switch to All groups. '
                           'No saved entries were deleted by filtering.',
                           11,MUTED).pack(anchor='w',padx=16,pady=(0,15))
                return
            g=selected['group']
            c=loaded[g]['computed']
            self.label(detail,'AUTOMATIC STAFFING SHORTAGE  /  '+g.upper(),13,WHITE,True).pack(anchor='w',padx=16,pady=(13,7))
            if c['shortage'] is None:
                self.label(detail,'PENDING - '+c['reason'],12,AMBER,wraplength=970,justify='left').pack(anchor='w',padx=16,pady=(0,11))
            else:
                self.label(detail,f"{c['shortage']} staff short   •   Planned {c['plan_total']}   •   Actual {c['available']}",
                           19,RED if c['shortage'] else TEAL,True).pack(anchor='w',padx=16,pady=(0,8))
                self.label(detail,'    |    '.join(f"{x['role']}: {x['present']}/{x['required']} present, {x['shortage']} short"
                           for x in c['role_breakdown']),11,WHITE,wraplength=980,
                           justify='left').pack(anchor='w',padx=16,pady=(0,10))
            self.label(detail,'No-shows via WhatsApp: '+(str(c['no_shows']) if c['no_shows'] is not None
                       else 'Not explicitly reported'),11,MUTED).pack(anchor='w',padx=16,pady=(0,10))
            snaps=core.exception_reports(g,day)
            if c['shortage'] is None and len(snaps)>0:
                options={}
                for r in snaps:
                    label=f"{r['event_at'][:16].replace('T',' ')} · report {r['id']} · T:{r['t002_t003']} P:{r['pwt']} S:{r['spl']} PL:{r['pl']}"
                    options[label]=r['id']
                bottom=ctk.CTkFrame(detail,fg_color='transparent');bottom.pack(fill='x',padx=12,pady=(0,12))
                opt=ctk.StringVar(value=next(iter(options)))
                ctk.CTkOptionMenu(bottom,variable=opt,values=list(options),width=470).pack(side='left',padx=4)
                def link():
                    try:
                        if any(drafts[g]):raise ValueError('Save pending group edits before linking attendance.')
                        core.link_final_attendance_as_user(session['token'],g,day,options[opt.get()],loaded[g]['revision'])
                        self._render()
                    except Exception as e:messagebox.showerror('Cannot link report',str(e))
                self.button(bottom,'Link final report',link,width=155,color='#166653').pack(side='left',padx=5)
        def select_row(_=None):
            ids=table.selection()
            if ids:selected['group']=ids[0];update_detail()
        table.bind('<<TreeviewSelect>>',select_row)
        # In-place editor: double-click numeric columns. Enter saves the draft and advances to next group.
        edit=[None]
        edit_commit=[None]
        def edit_cell(group,column,advance=False):
            if edit[0] and edit[0].winfo_exists():edit[0].destroy()
            if not table.exists(group):return
            box=table.bbox(group,column)
            if not box:return
            x,y,w,h=box
            # Native Tk Entry overlays ttk.Treeview reliably on Windows and Tk 8.6.
            entry=tk.Entry(table,bd=1,relief='solid',font=('Segoe UI',11),
                           bg=self._table_colors()['editor'],
                           fg=self._table_colors()['foreground'],
                           insertbackground=self._table_colors()['foreground'],
                           highlightthickness=2,highlightcolor=BLUE,
                           highlightbackground=self._table_colors()['border'],
                           selectbackground=self._table_colors()['selected'],
                           selectforeground=self._table_colors()['selected_ink'])
            old=cell_value(group,'before' if column=='#2' else 'after')
            if old!='-':entry.insert(0,old)
            entry.place(x=x,y=y,width=w,height=h)
            edit[0]=entry;entry.focus_set();entry.select_range(0,'end')
            submitted=[False]
            def commit(next_row=False):
                if submitted[0]:return
                submitted[0]=True
                val=entry.get().strip()
                stage='before' if column=='#2' else 'after'
                previous=loaded[group][stage]
                if val=='':
                    if previous is not None:
                        messagebox.showwarning('Cannot clear saved count','Enter a non-negative count. To correct a saved value, enter the replacement number.')
                    drafts[group].pop(stage,None)
                else:
                    try:
                        value=core._ex_validate_nonneg(val,'Changes '+stage+' 15:00')
                        if value==previous:drafts[group].pop(stage,None)
                        else:drafts[group][stage]=value
                    except ValueError as exc:messagebox.showerror('Invalid number',str(exc))
                entry.destroy();edit[0]=None;edit_commit[0]=None;draw_grid()
                if next_row:
                    rows=list(table.get_children())
                    if group in rows and rows.index(group)+1<len(rows):
                        next_group=rows[rows.index(group)+1]
                        table.see(next_group);table.selection_set(next_group)
                        self.after(30,lambda:edit_cell(next_group,column))
            edit_commit[0]=commit
            entry.bind('<Return>',lambda _:(commit(True),'break'))
            entry.bind('<Escape>',lambda _:(entry.destroy(),edit.__setitem__(0,None),edit_commit.__setitem__(0,None),'break'))
            entry.bind('<FocusOut>',lambda _:commit(False))
        def double_click(event):
            row=table.identify_row(event.y)
            col=table.identify_column(event.x)
            if row and col in ('#2','#3'):edit_cell(row,col)
        table.bind('<Double-1>',double_click)
        def save_all():
            if edit[0] and edit[0].winfo_exists() and edit_commit[0]:
                edit_commit[0](False)
            payload=[]
            for group in active:
                if drafts[group]:
                    payload.append({'group':group,'revision':loaded[group]['revision'],**drafts[group]})
            if not payload:
                messagebox.showinfo('No changes','No unsaved values to submit. Blank means not entered; zero is an explicit entry.')
                return
            try:
                updated=core.bulk_save_change_counts(session['token'],day,payload)
                self.status_text.set(f'{updated} groups saved for {day} by {actor}')
                messagebox.showinfo('Changes saved',f'{updated} group(s) updated. Your entries are timestamped and audited.')
                self._render()
            except Exception as exc:
                messagebox.showerror('No changes saved',str(exc)+'\nThe entire batch was rolled back. Reload to resolve conflicts.')
        toolbar=ctk.CTkFrame(workstation,fg_color='transparent');toolbar.pack(fill='x',padx=13,pady=(1,13))
        self.label(toolbar,'',11,AMBER,textvariable=unsaved_message).pack(side='left')
        self.label(toolbar,'Blank = pending  •  0 = confirmed none',10,MUTED).pack(side='left',padx=14)
        self.button(toolbar,'Save All Changes',save_all,width=185,color='#166653').pack(side='right',padx=5)
        def save_hotkey(event):
            if self.page=='Exceptions':save_all();return 'break'
        table.bind('<Control-s>',save_hotkey)
        search.bind('<Control-s>',save_hotkey)
        query.trace_add('write',lambda *_:draw_grid())
        state.trace_add('write',lambda *_:draw_grid())
        draw_grid()
        # Historical legacy shift labels remain in the DB; new submissions always use one service-date record.
        self._exception_recent_table()

    def _exception_load(self):
        if getattr(self,'_exception_draft_dirty',False):
            if not messagebox.askyesno('Unsaved entries','Discard unsaved edits and reload the selected date?'):
                return
            self._exception_draft_dirty=False
        try:core.parse_date(self.exception_day.get())
        except Exception as exc:
            messagebox.showerror('Invalid date',f'{exc}\\nUse YYYY-MM-DD or DD/MM/YYYY.');return
        self._exception_started_date=None
        self._render()

    def _exception_recent_table(self):
        self._title(self.content,'Recent service-date change and attendance records',
              'This is historical information; select a date above to edit all active groups in one grid.')
        today=date.today();metrics=core.exception_dashboard(today-timedelta(days=14),today)
        p=self.panel(self.content);p.pack(fill='x',pady=(4,8))
        self.label(p,f"{metrics['verified']} attendance calculations   •   {metrics['pending']} pending   •   "
                    f"{metrics['shortage']} combined role-specific shortages",11,TEAL).pack(anchor='w',padx=15,pady=(11,7))
        if not metrics['rows']:
            self.label(p,'○  No recent change or attendance records',13,WHITE,True).pack(
                anchor='w',padx=15,pady=(5,3))
            self.label(p,'The register will populate after operators submit changes or WhatsApp '
                'attendance is captured.',11,MUTED).pack(anchor='w',padx=15,pady=(0,18))
            return
        self._style_table('ExceptionHistory',rowheight=34,fontsize=10)
        tree=ttk.Treeview(p,columns=('date','group','before','after','present','short','status'),
                          show='headings',height=7,style='ExceptionHistory.Treeview')
        for key,name,width in [('date','Service date',110),('group','Group',190),
              ('before','Before 15:00',130),('after','After 15:00',130),
              ('present','Actual staff',110),('short','Shortage',85),('status','Status',160)]:
            tree.heading(key,text=name);tree.column(key,width=width,minwidth=75,anchor='center')
        for row in metrics['rows'][:100]:
            tree.insert('','end',values=(row['service_date'],row['group'],
               '-' if row['before_count'] is None else row['before_count'],
               '-' if row['after_count'] is None else row['after_count'],
               '-' if row['available'] is None else row['available'],
               'Pending' if row['shortage'] is None else row['shortage'],row['status']))
        self._stripe_rows(tree)
        tree.pack(fill='x',padx=15,pady=(2,13))

    def _exception_executive_panel(self):
        self._title(self.content,'Workforce changes, absences and shortages',
                    'Final WhatsApp attendance drives the shortage KPI; the single operator records both change counts.')
        k=ctk.CTkFrame(self.content,fg_color='transparent');k.pack(fill='x',pady=(4,10))
        self._live_widgets['ex_before']=self._kpi(k,'Changes before 15:00','-','Changes coordinator',BLUE)
        self._live_widgets['ex_after']=self._kpi(k,'Changes after 15:00','-','Same coordinator',AMBER)
        self._live_widgets['ex_no_shows']=self._kpi(k,'No shows','-','Only if explicitly reported in WhatsApp',RED)
        self._live_widgets['ex_shortage']=self._kpi(k,'Staff short','-','Confirmed role-specific vacancies',RED)
        outer=self.panel(self.content);outer.pack(fill='x',pady=8)
        self._live_widgets['ex_review']=self.label(outer,'',11,AMBER)
        self._live_widgets['ex_review'].pack(anchor='w',padx=15,pady=(13,5))
        self.label(outer,'CHART: Four colours = changes before 15:00, changes after 15:00, '
                   'explicit WhatsApp no-shows, and staffing shortages. Values are event counts, '
                   'not unique employees; unreported no-shows are unknown.',
                   10,MUTED,wraplength=1000,justify='left').pack(anchor='w',padx=15,pady=(0,5))
        from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg as _ExCanvas
        fig=Figure(figsize=(10.6,4.1),dpi=90,facecolor=PANEL)
        ax=fig.add_subplot(111);ax.set_facecolor(PANEL)
        canvas=_ExCanvas(fig,master=outer)
        canvas.get_tk_widget().pack(fill='x',padx=12,pady=(0,8))
        self._live_widgets['ex_chart']=(fig,ax,canvas)
        self._style_table('ExceptionReview',rowheight=34,fontsize=10)
        tbl=ttk.Treeview(outer,columns=('day','gang','change1','change2','no','short','state'),
                         show='headings',height=7,style='ExceptionReview.Treeview')
        for col,heading,width in [('day','Date',105),('gang','Gang',160),
              ('change1','Before 15:00',120),('change2','After 15:00',120),
              ('no','No shows',98),('short','Shortage',90),('state','State',104)]:
            tbl.heading(col,text=heading);tbl.column(col,width=width,minwidth=65,anchor='center')
        tbl.pack(fill='x',padx=15,pady=(4,16))
        self._live_widgets['ex_tbl']=tbl
        self._live_widgets['ex_signature']=None
        def refresh():
            a,b=self._date_bounds()
            data=core.exception_dashboard(a,b,self.group.get())
            self._update_kpi_in_place('ex_before',data['changes_before'] if data['changes_before_known'] else '-')
            self._update_kpi_in_place('ex_after',data['changes_after'] if data['changes_after_known'] else '-')
            self._update_kpi_in_place('ex_no_shows',data['no_shows'] if data['no_shows_known'] else '-',
                 f"Explicit WhatsApp counts for {data['no_shows_known']} shift(s)")
            self._update_kpi_in_place('ex_shortage',data['shortage'],
                         f"{data['short_shifts']} shift(s) below plan")
            self._configure_text(self._live_widgets['ex_review'],
                f"Attendance-based calculations: {data['verified']}   •   Pending: {data['pending']}   •   "
                f"Waiting for WhatsApp: {data['awaiting_whatsapp']}   •   Awaiting change entries: {data['awaiting_manual_changes']}   •   Change logs completed: {data['change_logs_complete']}")
            signature=tuple((r['group'],r['service_date'],r['shift'],r['revision'],
                 r['before_count'],r['after_count'],r['no_shows'],
                 r['report_id'],r['shortage'],r['status'],r['reason']) for r in data['rows'])
            if signature==self._live_widgets.get('ex_signature'):
                return
            self._live_widgets['ex_signature']=signature
            for item in tbl.get_children():tbl.delete(item)
            for r in data['rows'][:80]:
                tbl.insert('','end',values=(r['service_date'],r['group'],
                  '-' if r['before_count'] is None else r['before_count'],
                  '-' if r['after_count'] is None else r['after_count'],
                  'Not reported' if r['no_shows'] is None else r['no_shows'],
                  '-' if r['shortage'] is None else r['shortage'],r['status']))
            self._stripe_rows(tbl)
            # The category chart has four distinct colours like the stacked
            # workforce chart. Each category is an independent KPI event count;
            # stacked width is NOT a number of unique employees.
            chart_rows=core.workforce_exception_chart_rows(data['rows'],limit=10)
            ax.clear();ax.set_facecolor(PANEL)
            category_colors=['#2980B9','#16A085','#9B59B6','#E74C3C']
            specs=[('before','Changes before 15:00'),('after','Changes after 15:00'),
                   ('no_shows','No Shows'),('shortage','Shortages')]
            if chart_rows:
                names=[r['group'] for r in chart_rows]
                y=list(range(len(names)))
                left=[0]*len(names)
                for (field,label),color in zip(specs,category_colors):
                    values=[r[field] for r in chart_rows]
                    ax.barh(y,values,left=left,height=.66,label=label,color=color,
                            edgecolor=PANEL,linewidth=.5)
                    left=[a+b for a,b in zip(left,values)]
                ax.set_yticks(y,labels=names)
                ax.invert_yaxis()
                ax.set_xlim(left=0)
                ax.set_xlabel('Reported category counts - not unique employees',color=WHITE,fontsize=9)
                ax.legend(loc='lower center',bbox_to_anchor=(0.5,1.015),
                          ncol=4,fontsize=8,frameon=False,labelcolor=WHITE)
                ax.tick_params(colors=WHITE,labelsize=9)
                ax.xaxis.grid(True,color=BORDER,alpha=.45)
                ax.set_axisbelow(True)
                for side in ('top','right'):ax.spines[side].set_visible(False)
                ax.spines['bottom'].set_color(MUTED)
                ax.spines['left'].set_color(MUTED)
            else:
                ax.text(.5,.5,'No staffing events for this reporting period',
                        color=MUTED,ha='center',va='center',transform=ax.transAxes)
                ax.set_xticks([]);ax.set_yticks([])
            fig.tight_layout(pad=1.5)
            fig.subplots_adjust(top=.80,left=.18,bottom=.17)
            canvas.draw_idle()
        self._live_widgets['ex_refresh']=refresh
        refresh()

    def _staffing_plan_settings(self):
        """Dedicated Settings sub-page for group-specific role requirements."""
        self.label(self.content,'STAFFING PLANS / GROUP REQUIREMENTS',12,TEAL,True).pack(anchor='w',pady=(0,8))
        self.label(self.content,'Set the staff required for each registered WhatsApp group. '                   'The total is calculated automatically and used for shortage monitoring. '                   'A manually saved plan overrides Excel from its effective date onward.',
                   12,MUTED,wraplength=960,justify='left').pack(anchor='w',pady=(0,15))
        self.button(self.content,'← Back to Settings',lambda:self._navigate('Settings'),width=170).pack(anchor='w',pady=(0,12))
        registry=core.group_registry()
        names=[record['name'] for record in registry]
        if not names:
            self.label(self.content,'No saved WhatsApp groups. Add and save one in Settings first.',12,AMBER).pack(anchor='w',pady=10)
            return
        if not hasattr(self,'plan_group'):
            self.plan_group=ctk.StringVar(value=names[0])
        if not hasattr(self,'plan_effective_day'):
            self.plan_effective_day=ctk.StringVar(value=date.today().isoformat())
        if self.plan_group.get() not in names:
            self.plan_group.set(names[0])
        box=self.panel(self.content);box.pack(fill='x',pady=(0,14))
        self.label(box,'SELECT WHATSAPP GROUP AND EFFECTIVE DATE',13,WHITE,True).pack(anchor='w',padx=16,pady=(15,12))
        row=ctk.CTkFrame(box,fg_color='transparent');row.pack(fill='x',padx=16,pady=(0,14))
        self.label(row,'WhatsApp group',12,MUTED).pack(side='left',padx=(0,9))
        ctk.CTkOptionMenu(row,variable=self.plan_group,values=names,width=270,
              command=lambda _:self._render(),fg_color=PANEL,text_color=WHITE,
              button_color=('#B8D4ED' if self.theme_mode=='Light' else '#254C72')).pack(side='left',padx=(0,20))
        self.label(row,'Effective from',12,MUTED).pack(side='left',padx=(0,8))
        ctk.CTkEntry(row,textvariable=self.plan_effective_day,width=134).pack(side='left',padx=(0,12))
        self.button(row,'Load plan',self._render,width=118).pack(side='left')
        self.label(box,'Use YYYY-MM-DD. Save new dates to retain previous plans for historical reports. '                   'A group added in Settings appears here after Save settings.',
                   11,MUTED,wraplength=950,justify='left').pack(anchor='w',padx=16,pady=(0,14))
        group=self.plan_group.get()
        try:
            day=core.parse_date(self.plan_effective_day.get()).isoformat()
            current,error=core.exception_plan(group,day)
        except Exception as exc:
            self.label(self.content,'Invalid effective date: '+str(exc),12,RED).pack(anchor='w',pady=9)
            return
        plan_panel=self.panel(self.content);plan_panel.pack(fill='x',pady=(0,15))
        if current:
            self.label(plan_panel, f"CURRENT PLAN  ·  {current['planned_total']} people required",17,BLUE,True).pack(anchor='w',padx=16,pady=(14,4))
            self.label(plan_panel, f"Source: {current.get('source','Excel')}  |  "
                       f"From: {current['effective_from']}  |  Through: {current['effective_to']}",
                       11,MUTED,wraplength=930).pack(anchor='w',padx=16,pady=(0,12))
        else:
            self.label(plan_panel,'NO STAFFING PLAN AVAILABLE',16,AMBER,True).pack(anchor='w',padx=16,pady=(14,5))
            self.label(plan_panel,error,11,MUTED,wraplength=930).pack(anchor='w',padx=16,pady=(0,12))
        edit=self.panel(self.content);edit.pack(fill='x',pady=(0,12))
        self.label(edit,'REQUIRED STAFF PER ROLE',14,WHITE,True).pack(anchor='w',padx=16,pady=(15,5))
        self.label(edit,'Enter 0 where a role is not required. Values represent required headcount, not actual attendance.',
                   11,MUTED,wraplength=930).pack(anchor='w',padx=16,pady=(0,12))
        inputs={}
        total_var=ctk.StringVar(value='')
        roles=list(core.EX_METRIC_FIELDS)
        fields_row=ctk.CTkFrame(edit,fg_color='transparent');fields_row.pack(fill='x',padx=16,pady=8)
        for role in roles:
            block=ctk.CTkFrame(fields_row,fg_color=PANEL,corner_radius=8)
            block.pack(side='left',padx=(0,15))
            self.label(block,role,12,WHITE,True).pack(anchor='w',padx=10,pady=(8,3))
            field=ctk.CTkEntry(block,width=130,placeholder_text='0')
            value=(current['components'].get(role,0) if current else 0)
            field.insert(0,str(value))
            field.pack(padx=10,pady=(0,10))
            inputs[role]=field
        def update_preview(*_):
            try:
                values=[int(inputs[role].get()) for role in roles]
                if any(x<0 or x>10000 for x in values):raise ValueError()
                total_var.set(f'TOTAL REQUIRED PER SHIFT: {sum(values)} staff')
            except ValueError:
                total_var.set('Enter valid whole numbers (0-10000).')
        for field in inputs.values():
            field.bind('<KeyRelease>',update_preview)
        update_preview()
        self.label(edit,'',16,TEAL,True,textvariable=total_var).pack(anchor='w',padx=16,pady=(9,13))
        action_row=ctk.CTkFrame(edit,fg_color='transparent');action_row.pack(fill='x',padx=16,pady=(0,16))
        def save_plan():
            try:
                data={role:field.get() for role,field in inputs.items()}
                saved=core.save_manual_staffing_plan(group,day,data)
                self.status_text.set(f'Staffing plan saved: {group}, {saved["planned_total"]} required from {day}')
                self._last_sig=None
                messagebox.showinfo('Plan saved',f'{group}: {saved["planned_total"]} required staff from {day}.\n'
                    'All shortage calculations for dates covered by this plan will now use it.')
                self._render()
            except Exception as exc:
                messagebox.showerror('Cannot save staffing plan',str(exc))
        self.button(action_row,'Save staffing plan',save_plan,width=200,color='#166653').pack(side='left',padx=(0,12))
        versions=core.manual_plan_versions(group)
        matching=next((v for v in versions if v['effective_from']==day),None)
        if matching:
            def remove_version():
                if not messagebox.askyesno('Remove staffing plan version',
                    f'Remove the manual plan for {group} starting {day}?\n\n'
                    'Previous manual plans or the Excel tracker may become the active baseline. This can change historical shortages.'):
                    return
                try:
                    core.delete_manual_staffing_plan(group,day)
                    self._last_sig=None;self._render()
                except Exception as exc:
                    messagebox.showerror('Cannot remove version',str(exc))
            self.button(action_row,'Delete this date’s version',remove_version,width=210,color='#69424D').pack(side='left')
        history=self.panel(self.content);history.pack(fill='x',pady=(0,15))
        self.label(history,'SAVED PLAN HISTORY  /  '+group,13,WHITE,True).pack(anchor='w',padx=16,pady=(13,8))
        if not versions:
            self.label(history,'No manual versions saved yet. Excel is used only where a valid imported plan exists.',
                       11,MUTED).pack(anchor='w',padx=16,pady=(0,14))
        else:
            for item in versions[:12]:
                parts='  ·  '.join(f'{key}: {item["components"].get(key,0)}' for key in roles)
                self.label(history,f"{item['effective_from']}  |  {item['planned_total']} required  |  {parts}",
                           11,WHITE,wraplength=940,justify='left').pack(anchor='w',padx=16,pady=5)
            if len(versions)>12:
                self.label(history,f'{len(versions)-12} older versions not displayed.',11,MUTED).pack(anchor='w',padx=16)
            self.label(history,'Plan edits and deletions are saved in the database audit history.',
                       10,MUTED).pack(anchor='w',padx=16,pady=(4,14))

    def _import_planned_tracker(self):
        source=filedialog.askopenfilename(title='Import visible planned staffing from the KPI tracker',
                filetypes=[('Excel workbook','*.xlsm *.xlsx')])
        if not source:return
        try:
            plans,warnings=core.import_visible_staffing_plans(source)
            cfg=core.configuration();cfg['staffing_tracker_path']=str(Path(source).resolve())
            core.save_configuration(cfg)
            self._plan_source_mtime=Path(source).stat().st_mtime_ns
            messagebox.showinfo('Planned staffing imported',
                f'{len(plans)} non-empty staffing baselines imported from visible sheets.\n'
                f'{len(warnings)} skipped/invalid blocks (see log).\n\n'
                'Only dates covered by the source workbook will be used.')
            self._render()
        except Exception as exc:
            messagebox.showerror('Import failed',str(exc))

    def _safety(self):
        try:core.evaluate_alerts()
        except Exception as e:logging.exception('Alert check: %s',e)
        s=core.summary(self._rows)
        cards=ctk.CTkFrame(self.content,fg_color='transparent');cards.pack(fill='x')
        self._kpi(cards,'Reported near-miss',s['near_miss'],'Not verified distinct incidents',RED)
        # Display Late reports alongside Near Miss in Safety & Alerts.
        # Late is a reported attendance exception, not a safety incident.
        self._kpi(cards,'Late reported',s['late'],'Reported attendance exception values',AMBER)
        # Missing is distinct from a reported zero. Show a completeness rate
        # and the underlying numerator so executives can judge data quality.
        reports = s['reports']
        complete = reports - s['missing_safety']
        completeness = f'{complete / reports:.0%}' if reports else '-'
        completeness_detail = (f'{complete:,} of {reports:,} reports include both safety fields'
                               if reports else 'No reports in selected period')
        self._kpi(cards, 'Safety data completeness', completeness,
                  completeness_detail, TEAL if reports and complete == reports else AMBER)
        self._title(self.content,'Alert queue','Acknowledge or close locally; alert rules use rolling 2-day monitoring windows.')
        # Include Late alerts from the existing configurable threshold rule,
        # alongside Near Miss, quality and missing-report alerts.
        # Alert rules use the existing rolling two-day evaluation period.
        items=core.alerts();p=self.panel(self.content);p.pack(fill='x',pady=8)
        for r in items[:25]:
            row=ctk.CTkFrame(p,fg_color=BG,corner_radius=9);row.pack(fill='x',padx=9,pady=5)
            color=RED if r['level']=='High' else AMBER if r['level']=='Medium' else BLUE
            self.label(row,f"●  {r['title']}  •  {r['status']}",12,color,True).pack(anchor='w',padx=12,pady=(7,0))
            self.label(row,r['detail'],11,MUTED,wraplength=850,justify='left').pack(anchor='w',padx=12,pady=(2,8))
            ctrl=ctk.CTkFrame(row,fg_color='transparent');ctrl.pack(anchor='e',padx=12,pady=(0,7))
            owner=ctk.CTkEntry(ctrl,placeholder_text='Owner',width=100);owner.insert(0,r['owner'] or '');owner.pack(side='left',padx=4)
            self.button(ctrl,'Acknowledge',lambda id=r['id'],v=owner:self._change_alert(id,'Acknowledged',v.get()),width=105).pack(side='left',padx=3)
            self.button(ctrl,'Close',lambda id=r['id'],v=owner:self._change_alert(id,'Closed',v.get()),width=70,color='#286D5B').pack(side='left',padx=3)
        if not items:
            self.label(p,'✓  No alerts recorded',14,TEAL,True).pack(anchor='w',padx=16,pady=(15,6))
            self.label(p,'Alerts will appear here after WhatsApp collection or file import. '
                       'No alerts recorded does not prove all operational risks are absent.',
                       11,MUTED,wraplength=900).pack(anchor='w',padx=16,pady=(0,18))
        # The evidence section must always render something visible. An empty
        # section previously looked broken, particularly with date filters.
        # Show only reports with a positive Near Miss or Late numeric value.
        evidence_rows = [r for r in self._rows
                         if (r.get('near_miss') or 0) > 0
                         or (r.get('late') or 0) > 0]
        self._title(self.content, 'Near-miss and Late report evidence',
                    'Filtered to reports containing positive Near Miss or Late values.')
        self.label(self.content,
                   f'{len(evidence_rows):,} matching evidence report(s) from '
                   f'{len(self._rows):,} records in the selected period and group.',
                   12, TEAL if evidence_rows else AMBER).pack(
                       anchor='w', pady=(2, 10))
        if evidence_rows:
            # A scrollable table also works for >80 evidence records.
            self._report_table(self.content, evidence_rows, limit=200,
                               title='Report evidence')
        else:
            self._empty_state(self.content,'No positive Near Miss or Late evidence in this view',
                'No matching report contains a positive numeric Near Miss or Late value. '
                'This does not confirm zero incidents: check missing fields and widen the period if needed.',
                'Show all dates & groups',self._reset_view_filters,kind='neutral',compact=True)

    def _change_alert(self,id,status,owner):
        core.update_alert(id,status,owner);self._render()

    def _health(self):
        state=core.monitor_state()
        self._title(self.content,'Collection and persistence status')
        self.label(self.content,'The connection indicator turns green only after authenticated WhatsApp chat access is confirmed.',
                   11,MUTED).pack(anchor='w',pady=(0,9))
        k=self.panel(self.content);k.pack(fill='x')
        for name,value in [('Monitor',state['status']),('Last scan',state['last_checked'] or '-'),
                           ('Last success',state['last_success'] or '-'),('Last error',state['last_error'] or 'None'),
                           ('SQLite file',str(core.DB)),('Database size',f"{core.DB.stat().st_size/1024:.1f} KB")]:
            self.label(k,f'{name}:  {value}',12,TEAL if name=='Monitor' and value=='Connected' else MUTED,wraplength=920,justify='left').pack(anchor='w',padx=16,pady=7)
        diag=collector_diagnostics()
        self._title(self.content,'Live capture diagnostics (most recent group scan)')
        for name,key in [('Group','group_name'),('Last capture scan','updated_at'),
                         ('Message DOM containers','visible_messages'),('Metric-bearing messages','metric_messages'),
                         ('Newly saved','saved'),('Already stored','duplicate'),
                         ('Unrecognised timestamps','missing_timestamp'),('Collector note','last_note')]:
            self.label(self.content,f'{name}: {diag.get(key) or 0}',12,MUTED,
                       wraplength=920,justify='left').pack(anchor='w',padx=12,pady=3)
        self.label(self.content,'If the collector says No groups matched, set the exact group names in Settings. '
                   'If metrics stay at 0, review your WhatsApp message text against the parsing patterns.',
                   12,AMBER,wraplength=900).pack(anchor='w',pady=8)
        self._title(self.content,'Data tools')
        buttons=ctk.CTkFrame(self.content,fg_color='transparent');buttons.pack(fill='x',pady=6)
        # Keep every data action on one toolbar. The backup button sits
        # immediately beside Load demo as requested; it does not move data.
        for name,method in [('Import CSV',self._import_csv),('Import Excel',self._import_excel),
                            ('Export CSV',self._csv),('Export Excel',self._excel),
                            ('Load demo',self._demo)]:
            self.button(buttons,name,method,width=118).pack(side='left',padx=4)
        self.button(buttons,'Backup SQLite',self._backup,width=135,color='#286D5B').pack(side='left',padx=4)
        self._title(self.content,'Local audit trail (latest 80 entries)')
        p=self.panel(self.content);p.pack(fill='x')
        with core.connect() as c:logs=c.execute('SELECT at,action,detail FROM audit ORDER BY id DESC LIMIT 80').fetchall()
        if not logs:self.label(p,'No audit events yet',12,MUTED).pack(padx=15,pady=18)
        for row in logs:
            self.label(p,f"{row['at']}  |  {row['action']}  |  {row['detail']}",11,MUTED,wraplength=970,justify='left').pack(anchor='w',padx=15,pady=5)
        self._title(self.content,'Collection log')
        self.logs=ctk.CTkTextbox(self.content,height=150,fg_color=PANEL);self.logs.pack(fill='x',pady=(0,14))

    def _settings(self):
        """Persist user preferences and the structured group registry.

        All group edits are staged in the UI. Save settings commits registry
        metadata AND updates the backwards-compatible flat list consumed by
        WhatsAppMonitor.start(). Existing chat parsing remains untouched.
        """
        cfg=core.configuration()
        self._title(self.content,'Account Management',
                    'Create administrator/operator logins and manage access from Settings, not Exceptions.')
        account_controls=self.panel(self.content);account_controls.pack(fill='x',pady=(0,12))
        account_row=ctk.CTkFrame(account_controls,fg_color='transparent')
        account_row.pack(fill='x',padx=15,pady=13)
        self.label(account_row,'Signed in: '+self.exception_session['username']+'  •  Administrator',
                   12,TEAL,True).pack(side='left',padx=(0,18))
        # Expanding/collapsing accounts does not rebuild the Settings form,
        # protecting unsaved registry changes from accidental loss.
        account_details=ctk.CTkFrame(self.content,fg_color='transparent')
        is_open=getattr(self,'_show_settings_accounts',False)
        def toggle_account_management():
            self._show_settings_accounts=not self._show_settings_accounts
            if self._show_settings_accounts:
                account_details.pack(fill='x',after=account_controls)
                if not account_details.winfo_children():
                    self._exception_manage_users(parent=account_details)
            else:
                account_details.pack_forget()
            accounts_button.configure(text='Hide accounts' if self._show_settings_accounts
                                      else 'Manage accounts')
        accounts_button=self.button(account_row,
                  'Hide accounts' if is_open else 'Manage accounts',
                  toggle_account_management,width=150)
        accounts_button.pack(side='right')
        if is_open:
            account_details.pack(fill='x',after=account_controls)
            self._exception_manage_users(parent=account_details)
        self.label(self.content,'Monitoring and group settings are saved locally in data/settings.json.',
                   12,MUTED).pack(anchor='w',pady=8)
        p=self.panel(self.content);p.pack(fill='x',pady=12)
        # Structured group management replaces the old free-text textarea.
        # A group can be PAUSED without deleting its history or changing the
        # exact-match logic used by the original WhatsApp monitoring engine.
        self.label(p,'WHATSAPP MONITORING  /  GROUP REGISTRY',14,WHITE,True).pack(anchor='w',padx=16,pady=(16,4))
        self.label(p,'Search and manage exact WhatsApp group titles. Paused groups are not monitored; historical reports remain available.',
                   11,MUTED,wraplength=970,justify='left').pack(anchor='w',padx=16,pady=(0,12))
        registry = core.group_registry(cfg)  # Local staged edits until Save settings.
        search = ctk.StringVar(value='')
        entry_name = ctk.StringVar(value='')
        status_text = ctk.StringVar(value='')
        search_row=ctk.CTkFrame(p,fg_color='transparent')
        search_row.pack(fill='x',padx=16,pady=(0,8))
        self.label(search_row,'Search groups',11,MUTED).pack(side='left',padx=(0,8))
        ctk.CTkEntry(search_row,textvariable=search,width=275,placeholder_text='Filter by group name').pack(side='left')
        self.label(search_row,'Monitoring state: Active / Paused',11,MUTED).pack(side='right',padx=6)
        # The last report is read from the database, not inferred from online
        # status; Active means selected for monitoring, not connected.
        with core.connect() as connection:
            last_reports = {r['group_name'].casefold(): r['last_at']
                            for r in connection.execute('SELECT group_name, MAX(event_at) AS last_at FROM reports GROUP BY group_name')}
        self._style_table('GroupRegistry',rowheight=34,fontsize=10)
        registry_frame=ctk.CTkFrame(p,fg_color='transparent')
        registry_frame.pack(fill='x',padx=16,pady=(0,8))
        tree=ttk.Treeview(registry_frame,columns=('name','state','last'),show='headings',
                          selectmode='browse',height=7,style='GroupRegistry.Treeview')
        for col,title,width in [('name','Group name (exact WhatsApp title)',400),
                                ('state','Monitoring',110),('last','Last stored report',185)]:
            tree.heading(col,text=title)
            tree.column(col,width=width,minwidth=90,anchor='w',stretch=(col=='name'))
        scrollbar=ttk.Scrollbar(registry_frame,orient='vertical',command=tree.yview,
                                style='Ops.Vertical.TScrollbar')
        tree.configure(yscrollcommand=scrollbar.set)
        tree.pack(side='left',fill='x',expand=True)
        scrollbar.pack(side='right',fill='y')
        tree.tag_configure('paused', foreground=MUTED)
        tree.tag_configure('active', foreground=WHITE)

        def selected_group_index():
            selected = tree.selection()
            return int(selected[0]) if selected else None

        def redraw_registry(select_index=None):
            current_index = select_index if select_index is not None else selected_group_index()
            for item in tree.get_children():
                tree.delete(item)
            query = search.get().strip().casefold()
            for index,item in enumerate(registry):
                if query and query not in item['name'].casefold():
                    continue
                state = 'ACTIVE' if item['enabled'] else 'PAUSED'
                last = last_reports.get(item['name'].casefold())
                displayed = last.replace('T',' ')[:16] if last else 'No reports stored'
                tree.insert('',tk.END,iid=str(index),
                            values=(item['name'],state,displayed),
                            tags=('active' if item['enabled'] else 'paused',))
            self._stripe_rows(tree)
            active = len(core.active_group_names(registry))
            status_text.set(f'{len(registry)} registered  •  {active} active  •  {len(registry)-active} paused'
                            '  •  Changes take effect after saving and restarting monitoring')
            if current_index is not None and tree.exists(str(current_index)):
                tree.selection_set(str(current_index));tree.focus(str(current_index))

        def pick_group(_=None):
            index=selected_group_index()
            if index is not None:
                entry_name.set(registry[index]['name'])

        tree.bind('<<TreeviewSelect>>',pick_group)
        search.trace_add('write',lambda *_:redraw_registry())
        editor=ctk.CTkFrame(p,fg_color='transparent')
        editor.pack(fill='x',padx=16,pady=(5,4))
        self.label(editor,'Group title',11,MUTED).pack(side='left',padx=(0,8))
        ctk.CTkEntry(editor,textvariable=entry_name,placeholder_text='Exact name displayed in WhatsApp',
                     width=345).pack(side='left',padx=(0,8))

        def add_group():
            try:
                name=core.validated_group_title(entry_name.get())
                if any(x['name'].casefold()==name.casefold() for x in registry):
                    raise ValueError('This group is already registered. Select it to edit or reactivate.')
                registry.append({'name':name,'enabled':True})
                search.set('');redraw_registry(len(registry)-1)
                entry_name.set('')
            except ValueError as exc:messagebox.showwarning('Cannot add group',str(exc))

        def rename_group():
            index=selected_group_index()
            if index is None:
                messagebox.showinfo('Select a group','Select a row to rename.');return
            try:
                name=core.validated_group_title(entry_name.get())
                if any(i!=index and x['name'].casefold()==name.casefold() for i,x in enumerate(registry)):
                    raise ValueError('A group with this name already exists.')
                previous=registry[index]['name']
                if name != previous and not messagebox.askyesno('Change monitored group name',
                    f'Change the monitored title from:\n{previous}\n\nto:\n{name}\n\n'
                    'Existing reports retain their original group name. Continue?'):
                    return
                registry[index]['name']=name
                redraw_registry(index)
            except ValueError as exc:messagebox.showwarning('Cannot rename group',str(exc))

        def toggle_group():
            index=selected_group_index()
            if index is None:
                messagebox.showinfo('Select a group','Select the row to enable or pause.');return
            registry[index]['enabled']=not registry[index]['enabled']
            redraw_registry(index)

        def remove_group():
            index=selected_group_index()
            if index is None:
                messagebox.showinfo('Select a group','Select a row to remove.');return
            name=registry[index]['name']
            if messagebox.askyesno('Remove group from monitoring',
                    f'Remove "{name}" from the monitoring register?\n'
                    'Previously collected reports will NOT be deleted.'):
                registry.pop(index)
                entry_name.set('');redraw_registry()

        self.button(editor,'Add group',add_group,width=106,color='#166653').pack(side='left',padx=3)
        action_row=ctk.CTkFrame(p,fg_color='transparent');action_row.pack(fill='x',padx=16,pady=(5,7))
        self.button(action_row,'Rename selected',rename_group,width=145).pack(side='left',padx=(0,8))
        self.button(action_row,'Enable / Pause',toggle_group,width=135).pack(side='left',padx=(0,8))
        self.button(action_row,'Remove selected',remove_group,width=145,color='#69424D').pack(side='left')
        self.label(p,'Changes are staged until you select Save settings below. Stop and restart WhatsApp monitoring to apply a changed group list.',
                   11,AMBER,wraplength=965,justify='left').pack(anchor='w',padx=16,pady=(4,6))
        self.label(p,'',11,MUTED,textvariable=status_text).pack(anchor='w',padx=16,pady=(0,12))
        redraw_registry()
        fields=[]
        for key,title in [('sweep_seconds','Rescan interval (seconds)'),('alert_late_threshold','Late count alert threshold'),
                           ('alert_near_miss_threshold','Near-miss alert threshold'),('alert_missing_hours','Team silence alert after hours')]:
            row=ctk.CTkFrame(p,fg_color='transparent');row.pack(fill='x',padx=16,pady=8)
            self.label(row,title,12,MUTED).pack(side='left')
            inp=ctk.CTkEntry(row,width=110);inp.insert(0,str(cfg[key]));inp.pack(side='right');fields.append((key,inp))
        automatic=ctk.CTkCheckBox(p,text='Create a local daily executive PDF when this desktop app is running',text_color=WHITE)
        automatic.pack(anchor='w',padx=16,pady=10)
        if cfg.get('auto_pdf_daily',False):automatic.select()
        def save():
            try:
                cfg['auto_pdf_daily']=bool(automatic.get())
                # New structured registry remains backwards compatible with the
                # original collector's list of exact enabled group names.
                enabled=core.active_group_names(registry)
                if not enabled:raise ValueError('Enable at least one group before saving.')
                cfg['group_registry']=[dict(item) for item in registry]
                cfg['groups']=enabled
                for key,inp in fields:
                    val=int(inp.get());
                    if val<1:raise ValueError(f'{key} must be at least 1')
                    cfg[key]=val
                core.save_configuration(cfg);core.audit('SETTINGS_CHANGED','Updated local monitoring and alert thresholds')
                messagebox.showinfo('Settings saved',
                    'Monitoring settings and group register saved.\n\n'
                    'Stop and restart WhatsApp monitoring to apply group additions, pauses or renames.'
                    if self.monitor.active else
                    'Settings saved. The next WhatsApp monitoring session will use the enabled group list.')
            except Exception as e:messagebox.showerror('Invalid setting',str(e))
        self.button(p,'Save settings',save,width=170,color='#166653').pack(anchor='w',padx=16,pady=18)
        self._title(self.content,'Staffing plans per WhatsApp group',
                    'Manage each group’s required T002/T003, PWT, SPL and PL headcount in a separate planning screen.')
        manual_box=self.panel(self.content);manual_box.pack(fill='x',pady=(0,16))
        self.label(manual_box,'Set staffing requirements without editing Excel. New saved WhatsApp groups appear automatically.',
                   12,MUTED,wraplength=930).pack(anchor='w',padx=16,pady=(13,8))
        self.button(manual_box,'Manage Staffing Plans',lambda:self._navigate('Staffing Plans'),
                    color='#166653',width=230).pack(anchor='w',padx=16,pady=(0,14))
        self._title(self.content,'Optional Excel planning import - visible worksheets only',
                    'Workbook column C remains a fallback for dates without an active manually configured staffing plan.')
        plan_box=self.panel(self.content);plan_box.pack(fill='x',pady=(0,15))
        with core.connect() as c:
            plan_count=c.execute('SELECT COUNT(*) FROM staffing_plans').fetchone()[0]
        self.label(plan_box,f"{plan_count} dated gang plans available. Workbook: {cfg.get('staffing_tracker_path') or 'not connected'}",
                   12,MUTED,wraplength=920).pack(anchor='w',padx=16,pady=(13,9))
        self.button(plan_box,'Import visible planned staffing (.xlsm)',
                    self._import_planned_tracker,color='#166653',width=300).pack(anchor='w',padx=16,pady=(0,16))
        self._title(self.content, 'Appearance', 'The setting applies to every desktop tab and chart.')
        theme_panel = self.panel(self.content)
        theme_panel.pack(fill='x',pady=(0,15))
        theme_choice = ctk.StringVar(value=self.theme_mode)
        self.label(theme_panel,'Theme',12,MUTED).pack(side='left',padx=16,pady=14)
        ctk.CTkOptionMenu(theme_panel,values=['Light','Dark'],variable=theme_choice,
                          command=self._set_theme,width=120,
                          fg_color=PANEL,text_color=WHITE,
                          button_color=('#B8D4ED' if self.theme_mode == 'Light' else '#254C72')).pack(side='left',padx=8,pady=12)
        # Keep configuration and session in the project; only operations.db moves.
        self._title(self.content, 'Main database directory',
                    'Move the active SQLite file safely to a different local folder. The old copy is retained as a backup.')
        storage = self.panel(self.content)
        storage.pack(fill='x',pady=(0,15))
        self.db_location_label = self.label(storage, str(core.DB.resolve()),12,MUTED,wraplength=920,justify='left')
        self.db_location_label.pack(anchor='w',padx=16,pady=(14,7))
        self.button(storage,'Change database folder',self._choose_database_directory,width=230).pack(anchor='w',padx=16,pady=(0,15))
        self.label(self.content,'WhatsApp browser login data is stored locally in the project data folder. Restrict access to this folder.',12,AMBER,wraplength=930).pack(anchor='w',pady=12)

    def _choose_database_directory(self):
        """Move operations.db with a verified SQLite backup, never a raw file copy.

        Changing database paths while the browser monitor is running could
        split an incoming batch across two files, so require a stopped monitor.
        """
        if (self.monitor.active or self._reconcile_busy or
                (worker_loop is not None and worker_loop.is_running())):
            messagebox.showwarning('Stop monitoring first',
                'Stop WhatsApp monitoring and let CSV reconciliation and browser shutdown finish before moving the database.')
            return
        directory = filedialog.askdirectory(title='Choose a new folder for operations.db',
                                             initialdir=str(core.DB.parent))
        if not directory:
            return
        new_file = (Path(directory).expanduser().resolve() / 'operations.db')
        if new_file == core.DB.resolve():
            return
        if not messagebox.askyesno('Move main database',
                f'Create a verified copy of the active database at:\n{new_file}\n\n'
                f'The original will remain at:\n{core.DB}\n\nProceed?'):
            return
        try:
            result = core.relocate_database(directory)
            self._last_sig = None
            self.db_location_label.configure(text=str(result))
            messagebox.showinfo('Database relocated',
                                f'Now using:\n{result}\n\nThe previous database remains as a backup.')
        except Exception as exc:
            messagebox.showerror('Unable to change database', str(exc))

    def _start(self):
        if self.monitor.start():messagebox.showinfo('Monitoring','The local WhatsApp Web browser will open. Scan QR if prompted.')
        else:messagebox.showinfo('Monitoring','Collector is already running.')
    def _stop(self):self.monitor.stop()
    @staticmethod
    def _configure_text(label, text):
        if label is not None and label.winfo_exists() and str(label.cget('text'))!=str(text):
            label.configure(text=str(text))

    def _update_kpi_in_place(self,key,value,sub=None):
        pair=self._live_widgets.get(key)
        if pair:
            self._configure_text(pair[0],value)
            if sub is not None:self._configure_text(pair[1],sub)

    def _update_executive_in_place(self):
        """Apply newly committed SQLite data without destroying any Tk widgets.

        Called on Tk's event loop only. Preserve scroll position, focus, chart
        canvas, table selection, and the existing window geometry.
        """
        if self.page!='Executive' or 'counter' not in self._live_widgets:return
        rows=self._selected_rows()
        self._rows=rows
        if rows and self._live_widgets.get('report_empty_titles'):
            for label in self._live_widgets.pop('report_empty_titles'):
                if label.winfo_exists():label.pack_forget()
        if rows and self._live_widgets.get('executive_empty'):
            placeholder=self._live_widgets.pop('executive_empty')
            if placeholder.winfo_exists():placeholder.pack_forget()
        s=core.summary(rows,core.configuration()['groups'])
        prior=core.compare(rows,*self._date_bounds())
        with core.connect() as con:
            all_count=con.execute('SELECT COUNT(*) FROM reports').fetchone()[0]
        with core.connect() as con:
            sig_row=con.execute('SELECT COUNT(*),MAX(id) FROM reports').fetchone()
        self._last_sig=(tuple(sig_row),core.exception_data_signature(),self.page)
        self._configure_text(self._live_widgets['counter'],
            f'LIVE DATA  •  {s["reports"]:,} displayed / {all_count:,} in database  •  Period: {self.period.get()}  •  Team: {self.group.get()}')
        note=(f"{prior['reports']:+.1f}% vs previous period" if prior and prior['reports'] is not None else 'Comparison unavailable')
        self._update_kpi_in_place('report_kpi',f"{s['reports']:,}",note)
        self._update_kpi_in_place('team_kpi',s['teams'],f"{len(s['missing_groups'])} configured team(s) not represented")
        self._update_kpi_in_place('near_kpi',s['near_miss'])
        complete,total,pct=core.safety_completeness(rows)
        self._update_kpi_in_place('complete_kpi',f'{pct:.1f}%' if pct is not None else '-',
                                  f'{complete} of {total} reports have both fields')
        issues=[f"{len(core.alerts('Open'))} open alerts",
                f"{s['missing_safety']} reports missing safety values" if s['missing_safety'] else 'Safety values complete in received reports',
                f"{len(s['missing_groups'])} teams without a captured report",
                f"Last captured: {s['last_event'][:16] if s['last_event'] else 'No records'}"]
        for label,text in zip(self._live_widgets.get('issues',[]),issues):
            self._configure_text(label,'●  '+text)
        signature=tuple((r['id'],r['near_miss'],r['group_name']) for r in rows)
        if signature!=self._live_widgets.get('safety_signature'):
            self._live_widgets['safety_signature']=signature
            bar=self._live_widgets.get('safety_chart')
            if bar:
                fig,ax,canvas=bar
                self._draw_safety_bars(ax,rows)
                fig.tight_layout(pad=1.5);canvas.draw_idle()
            pie=self._live_widgets.get('safety_pie')
            if pie:
                fig,ax,canvas=pie
                per_gang=core.near_miss_by_group(rows)
                core.original_style_pie(ax,list(per_gang),list(per_gang.values()),
                    (['#d35400','#2980b9','#27ae60','#8e44ad','#f39c12']*max(1,len(per_gang))),
                    dark=self.theme_mode == 'Dark')
                fig.tight_layout(pad=1.2);canvas.draw_idle()
        exception_refresh=self._live_widgets.get('ex_refresh')
        if exception_refresh: exception_refresh()
        tree=self._live_widgets.get('reports_tree')
        if tree and tree.winfo_exists():
            wanted=rows[:80]
            signature=tuple(r['id'] for r in wanted)
            if signature!=self._live_widgets.get('table_signature'):
                self._live_widgets['table_signature']=signature
                mapping=self._live_widgets['reports_mapping']
                # Reuse row widgets whenever possible, and keep the selected row.
                existing={r['id']:item for item,r in mapping.items()}
                selected_ids={mapping[item]['id'] for item in tree.selection() if item in mapping}
                for item,r in list(mapping.items()):
                    if r['id'] not in signature:
                        tree.delete(item);mapping.pop(item,None)
                for i,r in enumerate(wanted):
                    vals=[r['event_at'].replace('T',' '),r['group_name'],r.get('sender') or '-']+[r[k] if r[k] is not None else '-' for k in ('t002_t003','pwt','spl','pl','near_miss')]
                    item=existing.get(r['id'])
                    if item and tree.exists(item):
                        if tuple(str(v) for v in tree.item(item,'values'))!=tuple(str(v) for v in vals):tree.item(item,values=vals)
                        tree.move(item,'',i)
                    else:
                        item=tree.insert('',i,values=vals)
                        mapping[item]=r
                self._stripe_rows(tree)
                tree.selection_set([item for item,r in mapping.items() if r['id'] in selected_ids])

    def _schedule_dashboard_refresh(self):
        """Debounce bursts, and always update Tk from the main GUI thread."""
        if self._refresh_pending:
            return
        self._refresh_pending=True
        self.after(75, self._apply_dashboard_refresh)

    def _apply_dashboard_refresh(self):
        self._refresh_pending=False
        if self.page=='Overview':
            self._update_overview_in_place()
        elif self.page=='Executive':
            self._update_executive_in_place()
        elif self.page=='Workforce':
            # Transition once from the empty view to a live chart when the
            # first matching report arrives; otherwise keep no-flicker updates.
            if not self._rows and self._selected_rows():
                self._render()
            else:
                refresh = self._live_widgets.get('analysis_refresh')
                if refresh: refresh()
        # Other views refresh when revisited. Avoid rebuilding the screen on
        # every insert (the previous behaviour caused severe visual flicker).

    def _poll_planned_tracker(self):
        source = core.configuration().get('staffing_tracker_path')
        if not source or self._plan_import_busy:
            return
        path = Path(source)
        if not path.is_file():
            return
        try:
            modified = path.stat().st_mtime_ns
        except OSError:
            return
        if modified == self._plan_source_mtime:
            return
        self._plan_import_busy=True
        def ingest():
            try:
                plans,warnings=core.import_visible_staffing_plans(path)
                self._plan_source_mtime=modified
                self.queue.put(('planned_imported',len(plans),len(warnings)))
            except Exception as exc:
                logging.warning('Unable to refresh workbook staffing plans: %s',exc)
                self.queue.put('Planned staffing update failed: '+str(exc))
            finally:
                self._plan_import_busy=False
        threading.Thread(target=ingest,daemon=True).start()

    def _poll(self):
        """Run message notifications, status LED and backup checks on the Tk thread."""
        import time
        new_reports=False
        while True:
            try: msg=self.queue.get_nowait()
            except queue.Empty: break
            if isinstance(msg, tuple) and msg and msg[0]=='planned_imported':
                new_reports=True
                self.status_text.set(f'{msg[1]} planned staffing baselines refreshed ({msg[2]} notes)')
                continue
            if isinstance(msg, tuple) and msg and msg[0]=='reports_saved':
                new_reports=True
                self.status_text.set(f'{msg[1]} new records saved from {msg[2]}')
                continue
            self.status_text.set(str(msg))
            if self.page=='System Health' and hasattr(self,'logs') and self.logs.winfo_exists():
                self.logs.insert('end',f'{datetime.now():%H:%M:%S}  {msg}\n')
                self.logs.see('end')
        if new_reports:
            if self.page=='Exceptions':
                if not getattr(self,'_exception_draft_dirty',False):
                    self._render()  # New actual attendance shown immediately without losing edits.
                else:
                    self.status_text.set('WhatsApp updates received. Save group changes before refreshing Exceptions.')
            else:
                self._schedule_dashboard_refresh()
        # Fallback for imports/external database writers. This also detects
        # changes when the collector cannot send an event to this GUI process.
        now=time.monotonic()
        if now >= self._next_fallback_check:
            self._next_fallback_check=now+0.8
            if self.page in ('Overview','Executive','Workforce','Safety & Alerts'):
                with core.connect() as c:
                    row=c.execute('SELECT COUNT(*), MAX(id) FROM reports').fetchone()
                sig=(tuple(row),core.exception_data_signature(),self.page)
                if self._last_sig != sig:
                    self._last_sig=sig
                    self._schedule_dashboard_refresh()
        if now >= self._next_plan_check:
            self._next_plan_check=now+30.0
            self._poll_planned_tracker()
        if now >= self._next_connection_led_check:
            self._next_connection_led_check = now + 0.5
            self._update_connection_led()
        if now >= self._next_daily_check:
            self._next_daily_check=now+30.0
            self._maybe_daily_report()
        self.after(100,self._poll)

    def _backup(self):
        import sqlite3
        path=filedialog.asksaveasfilename(defaultextension='.db',filetypes=[('SQLite database','*.db')])
        if not path:return
        try:
            with core.connect() as source, sqlite3.connect(path) as target:
                source.backup(target)
            core.audit('DB_BACKUP',Path(path).name)
            messagebox.showinfo('Backup saved',f'Consistent SQLite snapshot created at:\n{path}')
        except Exception as e:messagebox.showerror('Backup failed',str(e))

    def _maybe_daily_report(self):
        cfg=core.configuration()
        if not cfg.get('auto_pdf_daily'):return
        now=datetime.now()
        if now.hour < 18:return # generate at first poll at/after 18:00 local time
        day=now.date().isoformat()
        if self._last_report_day==day:return
        target=core.DATA / 'reports';target.mkdir(exist_ok=True)
        destination=target / f'executive_{day}.pdf'
        if destination.exists():
            self._last_report_day=day;return
        try:
            rows=core.filtered(now.date(),now.date())
            core.export_pdf(destination,rows,'All groups - daily',
                exception_rows=core.exception_dashboard(now.date(),now.date(),'All Groups')['rows'])
            self._last_report_day=day
        except Exception as e:logging.exception('Automatic local PDF report failed: %s',e)

    def _import(self,extension):
        path=filedialog.askopenfilename(filetypes=[('Reporting file',f'*.{extension}')]);
        if not path:return
        try:
            result=core.import_csv(path) if extension=='csv' else core.import_excel(path)
            core.evaluate_alerts();added,dupes,errors=result
            self.status_text.set(f'Import complete · {added} new · {dupes} duplicates · {len(errors)} rejected')
            messagebox.showinfo('Import complete',f'Added: {added}\nDuplicates skipped: {dupes}\nRejected: {len(errors)}'+(f'\n\nFirst errors:\n'+ '\n'.join(errors[:5]) if errors else ''))
            self._last_sig=None;self._render()
        except Exception as e:messagebox.showerror('Import failed',str(e))
    def _import_csv(self):self._import('csv')
    def _import_excel(self):self._import('xlsx')
    def _save(self,ext,method):
        path=filedialog.asksaveasfilename(defaultextension='.'+ext,filetypes=[(ext.upper(),f'*.{ext}')]);
        if not path:return
        try:
            self.status_text.set('Preparing '+ext.upper()+' export…')
            self.update_idletasks()
            rows=self._selected_rows()
            method(path,rows)
            try:self._register_local_report(path)
            except Exception as catalog_exc:
                logging.warning('Export completed but report was not indexed: %s',catalog_exc)
            self.status_text.set('Export completed · '+Path(path).name)
            messagebox.showinfo('Export successful',f'Executive report saved to:\n{path}\n\nIncludes WhatsApp and staffing exception records for the selected scope.')
        except Exception as e:messagebox.showerror('Export failed',str(e))
    def _csv(self):self._save('csv',core.export_csv)
    def _excel(self):self._save('xlsx',core.export_excel)
    def _pdf(self):
        # WhatsApp reports are filtered by message date; exceptions by the
        # corresponding night-shift service date. Include early entries even
        # where no WhatsApp attendance report has arrived yet.
        start,end=self._date_bounds()
        group=self.group.get()
        def run(path,rows):
            exceptions=core.exception_dashboard(start,end,group)['rows']
            core.export_pdf(path,rows,scope=f'{group} / {self.period.get()}',exception_rows=exceptions)
        self._save('pdf',run)
    def _html(self):
        start,end=self._date_bounds()
        group=self.group.get()
        def run(path,rows):
            exceptions=core.exception_dashboard(start,end,group)['rows']
            core.export_interactive_html(path,rows,scope=f'{group} / {self.period.get()}',exception_rows=exceptions)
        self._save('html',run)
    def _demo(self):
        if not messagebox.askyesno('Synthetic demonstration','Add labelled SYNTHETIC DEMO records to this local database?\n\nUse a new empty database for production.'):
            return
        count=core.demo_data();core.evaluate_alerts()
        messagebox.showinfo('Demo data',f'Added {count} labelled synthetic reports.');self._last_sig=None;self._navigate('Executive')
    def _close(self):
        if self.monitor.active:
            if not messagebox.askyesno('Close monitoring','Stopping the application will also stop monitoring. Continue?'):return
            self.monitor.stop()
        self.destroy()


## GUI refresh — visual-only behaviour

- Use the **« / »** control to collapse or expand the sidebar.
- Switching pages briefly displays an animated **Preparing your dashboard** state.
- Reporting pages distinguish **no attendance data** from **zero staffing shortages**.
- On an unstarted Exceptions date, select **Start entries for this date** to open the rapid-entry grid.
- Existing tables, editor shortcuts, PDF/HTML exports and calculations are retained.
- The GUI requires CustomTkinter and a graphical Windows or Linux environment; execute the notebook cells in order.

### Unified navigation additions

- **Overview** is the default landing page; original Executive, Workforce, Exceptions, Safety & Alerts, System Health and Settings remain available as before.
- **Reports** provides format-specific generation to `data/reports`, a searchable report library, and open-in-default-app actions.
- Existing PDF/HTML export controls in other pages are not removed. Their successfully saved files are now indexed in the library without copying them.
- Automated PDF files saved in `data/reports` appear in the library after refresh. Use **Add existing files** to index older reports without moving them, or put them in the report folder.
- A report catalogue stores only local file paths and scope metadata in `data/reports/report_catalog.json`.


## 7. Launch desktop application

Run last; it starts the GUI event loop. Close GUI before restarting the kernel.

In [ ]:
if __name__=='__main__':
    logging.basicConfig(level=logging.INFO,format='%(asctime)s %(levelname)s %(message)s',
                         handlers=[logging.StreamHandler(), logging.FileHandler(DATA / 'collector.log', encoding='utf-8')])
    Application().mainloop()
